# Loading 

In [ ]:
import numpy as np
import os
from tqdm import tqdm
import pandas as pd
from utils.test_functions import config
from utils.hplc import linear_reg
from utils.plot import imshow_area, mycmp
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd 
from statsmodels.regression.linear_model import OLS
import statsmodels.api as sm
from matplotlib.colors import Normalize, LogNorm, PowerNorm
from matplotlib.cm import ScalarMappable
from matplotlib.colors import ListedColormap
from cmocean import cm
import matplotlib
import matplotlib.pyplot as plt
from utils.temporal_decomposition import temporal_decomposition
from utils.functions import compute_trend_iav, onehot_sparse
from matplotlib.patches import Patch
from utils.functions import onehot_sparse, extend_nan_both_dimensions
import scipy.io 
from matplotlib.colors import LogNorm
import xarray as xr
matplotlib.rcParams.update({'font.size': 8})

#load data
physics=xr.load_dataset("/home/luther/Documents/npy_data/physics/physics_completion/processed_1998_2023_reanalysis_100km_monthly_lat50_seaice.nc")[['mld','sst','sss','ssh','solar','norm_currents','vort_winds']].to_array(dim='variables').data
physics=np.swapaxes(physics, 0, 1)
mask_coast = np.where(np.isnan(extend_nan_both_dimensions(physics[0,0], steps=2)), np.nan,1)

physics=physics*mask_coast

######

#avw 
chl=xr.open_dataset("/home/luther/Documents/npy_data/chl/monthly_avw/chl_avw_m_glob_lat50_1998_2023.nc")['CHL1_mean'].values*mask_coast

#psc_roy
psc=xr.open_dataset("/home/luther/Documents/npy_data/PSC/chl_psc_avw_roy_monthly_glob_100km_1998_2023_1.nc")[['Micro','Nano','Pico']].to_array(dim='variables').data[...,40:-40,:] * mask_coast
psc = np.swapaxes(psc, 0, 1) if len(psc) < 10 else psc  # Swap axes only for .nc files, in nc files the variables are on the first axis instead of the second one
psc_conc=psc*chl[:,None]
base_path = "/home/luther/Documents/result/avw_psc_chl_monthly/"
sub_dirs = ["variables"]#, "variables_anomalies"]#,"fun"]#, "chl_anomalies"]#, "test", "period"]

#sub_dirs=['test']

#####

#occci
"""
chl=np.load("/home/luther/Documents/npy_data/chl/chl_occci_1998_2023_100km_monthly_lat50.npy")*mask_coast
base_path = "/home/luther/Documents/result/chl_occci_only_monthly/"
sub_dirs = ["all_period"]#["variables", "variables_anomalies"]#, "chl_anomalies"]
"""


configs_path = {}
chl_pred_glob = {}
psc_pred_glob={}
psc_conc_pred_glob={}
configs={}

for sub_dir in sub_dirs:
    path_dir = os.path.join(base_path, sub_dir)
    configs_path[sub_dir] = os.listdir(path_dir)
    if 'runs.txt' in configs_path[sub_dir]:
        configs_path[sub_dir].remove('runs.txt')

    for name in configs_path[sub_dir]:
        print(name)

        key = name#[20:] if sub_dir in ["variables", "variables_anomalies", "chl_anomalies"] else name
        #if name=="SmaAt_chlm_avw_only_anom_allA":
        #    key="SmaAt_chlm_avw_only_anom_all_ssta"
        path_dir_name = os.path.join(path_dir, name)
        configs[key] = config(os.path.join(path_dir_name, name + '_config.json'))
        chl_pred_glob[key] = np.squeeze(np.load(os.path.join(path_dir_name, 'chl_pred_glob.npy'))) * mask_coast
        psc_pred_glob[key] = np.squeeze(np.load(os.path.join(path_dir_name, 'psc_pred_glob.npy'))) * mask_coast
        psc_conc_pred_glob[key] = psc_pred_glob[key] * chl_pred_glob[key][:,None] * mask_coast

#occci
chl_occci=np.load("/home/luther/Documents/npy_data/chl/chl_occci_1998_2023_100km_monthly_lat50.npy")*mask_coast
base_path_occci = "/home/luther/Documents/result/chl_occci_only_monthly/"
sub_dirs_occci = ["variables", "variables_anomalies"]#, "chl_anomalies"]

configs_path_occci = {}
chl_pred_glob_occci = {}
configs_occci={}


for sub_dir in sub_dirs_occci:
    path_dir = os.path.join(base_path_occci, sub_dir)
    configs_path_occci[sub_dir] = os.listdir(path_dir)
    if 'runs.txt' in configs_path_occci[sub_dir]:
        configs_path_occci[sub_dir].remove('runs.txt')

    for name in configs_path_occci[sub_dir]:
        print(name)

        key = name#[20:] if sub_dir in ["variables", "variables_anomalies", "chl_anomalies"] else name

        path_dir_name = os.path.join(path_dir, name)
        configs_occci[key] = config(os.path.join(path_dir_name, name + '_config.json'))
        chl_pred_glob_occci[key] = np.squeeze(np.load(os.path.join(path_dir_name, 'chl_pred_glob.npy'))) * mask_coast



print()

# anomalies + mean and seasonal

In [ ]:
#anomalies computing
def compute_anomalies(array):
    timestep=len(array)#we suppose that t is the first dim
    shape=array.shape
    weekly_mean=np.concatenate([np.nanmean(array.reshape(timestep//12,12,*shape[1:]),axis=0)]*(timestep//12))
    return array-weekly_mean

chl_pred_glob_anom={}
psc_conc_pred_glob_anom={}
psc_pred_glob_anom={}
#anomalies
for key in chl_pred_glob.keys():
    if chl_pred_glob[key].shape[1]==2:
        chl_pred_glob_anom[key]=chl_pred_glob[key][:,1]
        chl_pred_glob[key]=chl_pred_glob[key][:,0]
        print('anom')
    else : 
        chl_pred_glob_anom[key]=compute_anomalies(chl_pred_glob[key])
        psc_conc_pred_glob_anom[key]=compute_anomalies(psc_conc_pred_glob[key])
        psc_pred_glob_anom[key]=compute_anomalies(psc_pred_glob[key])

chl_anom=compute_anomalies(chl)
psc_conc_anom=compute_anomalies(psc_conc)
psc_anom=compute_anomalies(psc)
#chl_pred_glob_anom_bonus=compute_anomalies(chl_pred_glob)


# Test set

In [ ]:
total_timesteps = chl.shape[0] #1998-2023 attention pour les anciennes versions des PSC
timesteps_per_year = 12

# Calculate the total number of years in the dataset
total_years = total_timesteps // timesteps_per_year
# Initialize indices
test_indices = []
# Iterate over years and append indices to sets
for i in range(total_years):
    start_idx = i * timesteps_per_year
    end_idx = (i + 1) * timesteps_per_year
    if i>=13 :
        test_indices.append(slice(start_idx, end_idx))

psc_test=np.concatenate([psc[index] for index in test_indices])

psc_conc_test=np.concatenate([psc_conc[index] for index in test_indices])
psc_conc_anom_test=np.concatenate([psc_conc_anom[index] for index in test_indices])

chl_test=np.concatenate([chl[index] for index in test_indices]) 
chl_anom_test=np.concatenate([chl_anom[index] for index in test_indices]) 

chl_pred_test={}
chl_pred_anom_test={}

psc_pred_test={}

psc_conc_pred_test={}
psc_conc_anom_pred_test={}


for key in chl_pred_glob.keys():
    chl_pred_test[key]=np.concatenate([chl_pred_glob[key][60:][index] for index in test_indices]) 
    psc_pred_test[key]=np.concatenate([psc_pred_glob[key][60:][index] for index in test_indices],axis=0) 
    chl_pred_anom_test[key]=np.concatenate([chl_pred_glob_anom[key][60:][index] for index in test_indices])
    psc_conc_pred_test[key]=np.concatenate([psc_conc_pred_glob[key][60:][index] for index in test_indices],axis=0) 
    psc_conc_anom_pred_test[key]=np.concatenate([psc_conc_pred_glob_anom[key][60:][index] for index in test_indices],axis=0) 


mean_chl=np.stack([np.nanmean(chl,axis=0)]*156,axis=0)
seasonal_mean_chl=np.concatenate([np.nanmean(chl.reshape(312//12,12,100,360),axis=0)]*13)

mean_psc=np.stack([np.nanmean(psc,axis=0)]*156,axis=0)
seasonal_mean_psc=np.concatenate([np.nanmean(psc.reshape(312//12,12,3,100,360),axis=0)]*13,axis=0)

mean_psc_conc=np.stack([np.nanmean(psc_conc,axis=0)]*156,axis=0)
seasonal_mean_psc_conc=np.concatenate([np.nanmean(psc_conc.reshape(312//12,12,3,100,360),axis=0)]*13,axis=0)

chl_train=chl[48:-156]
psc_train=psc[48:-156]
psc_conc_train=psc_conc[48:-156]

seasonal_train_mean_chl=np.concatenate([np.nanmean(chl_train.reshape(108//12,12,100,360),axis=0)]*13)
seasonal_train_mean_psc=np.concatenate([np.nanmean(psc_train.reshape(108//12,12,3,100,360),axis=0)]*13,axis=0)
seasonal_train_mean_psc_conc=np.concatenate([np.nanmean(psc_conc_train.reshape(108//12,12,3,100,360),axis=0)]*13,axis=0)



chl_pred_test['mean']=mean_chl
chl_pred_test['seasonal']=seasonal_mean_chl

psc_pred_test['mean']=mean_psc
psc_pred_test['seasonal']=seasonal_mean_psc

psc_conc_pred_test['mean']=mean_psc_conc
psc_conc_pred_test['seasonal']=seasonal_mean_psc_conc

chl_pred_anom_test['mean']=np.zeros_like(mean_chl)
chl_pred_anom_test['seasonal']=np.zeros_like(mean_chl)

psc_conc_anom_pred_test['mean']=np.zeros_like(mean_psc_conc)
psc_conc_anom_pred_test['seasonal']=np.zeros_like(seasonal_mean_psc_conc)

chl_pred_test['train_seasonal']=seasonal_train_mean_chl
psc_pred_test['train_seasonal']=seasonal_train_mean_psc
psc_conc_pred_test['train_seasonal']=seasonal_train_mean_psc_conc

chl_pred_anom_test['train_seasonal']=np.zeros_like(seasonal_train_mean_chl)
psc_conc_anom_pred_test['train_seasonal']=np.zeros_like(seasonal_train_mean_psc_conc)

In [ ]:
def RMSE(a,b, axis=None):
    return np.sqrt(np.nanmean((a-b)**2,axis=(0)))

rmse_climato=RMSE(chl[-156:]-chl_anom[-156:],chl[-156:])
rmse_chl_all=RMSE(chl_pred_glob['SmaAt_chlm_avw_variables_all'][-156:],chl[-156:])

imshow_area((rmse_chl_all-rmse_climato)/rmse_climato,cmap='coolwarm',vmin=-1,vmax=1)

# Test Metric

### Bio region pacifique loading

In [ ]:
bio_regions=scipy.io.loadmat('/home/luther/Documents/npy_data/biomes/biomes_cabre_2016.mat')
imshow_area(bio_regions['regions_1Degree'][40:-40],cmap='tab20',title='bioregions', )
            #save="/home/luther/Documents/plot/article_1/cabre_bioreg.png") 

In [ ]:

#one hot encoding bio reg to ease the plotting process with chl/psc
bio_regions=onehot_sparse(bio_regions['regions_1Degree'][40:-40])
bio_regions=np.where(bio_regions==0,np.nan,1)

#essential variables
bioreg_name=['Undetermined','North-Polar','North SuPolar','Transition','Seosanal strat Subtropics North','Subtropics North',
              'Subtropics South','Tropical Pacific North','Tropical Pacific South','Tropical Atlantic North',
              'Tropical Atlantic South','Indian ocean North', 'Indian ocean South', 'Seasonal strat Subtropics South', 
             'SO-SA-high biomass','SO-SA-low biomass','SO-Acc']

bio_reg=np.where(~np.isnan(bio_regions[...,7])+~np.isnan(bio_regions[...,8])==1,1,np.nan)
bio_reg_2=np.where(~np.isnan(bio_regions[...,9])+~np.isnan(bio_regions[...,10])==1,1,np.nan)

In [ ]:
def compute_and_sort_metrics(metric_func, data_dict, verbose=True):
    """
    Computes a metric for each key-value pair in a dictionary and prints the keys sorted by the metric.

    Parameters:
        metric_func (function): A function that takes the data (value of the dict) as input and returns a computed metric.
        data_dict (dict): A dictionary where the metric function is applied to the values.

    Returns:
        None
    """
    # Compute metrics for each key
    metrics = {key: metric_func(value) for key, value in data_dict.items()}
    
    # Sort by metric values (descending order)
    sorted_metrics = sorted(metrics.items(), key=lambda x: x[1], reverse=True)

    # Print the keys and corresponding metrics
    for key, metric in sorted_metrics:
        if verbose :
            print(f"{key}: {metric:.4f}")
    return metrics 
    
def RMSE_chl(chl_pred, axis=None):
    return np.sqrt(np.nanmean((chl_pred-chl_test)**2,axis=axis))

def RMSE_chl_anomalies(chl_pred, axis=None):
    return np.sqrt(np.nanmean((chl_pred-chl_anom_test)**2,axis=axis))

def RMSE_chl_pacifique(chl_pred, axis=None):
    return np.sqrt(np.nanmean(((chl_pred*bio_reg)-(chl_test*bio_reg))**2,axis=axis))

def RMSE_chl_anomalies_pacifique(chl_pred, axis=None):
    return np.sqrt(np.nanmean(((chl_pred*bio_reg)-(chl_anom_test*bio_reg))**2,axis=axis))

def RMSE_chl_atlantique(chl_pred, axis=None):
    return np.sqrt(np.nanmean(((chl_pred*bio_reg_2)-(chl_test*bio_reg_2))**2,axis=axis))

def RMSE_chl_anomalies_atlantique(chl_pred, axis=None):
    return np.sqrt(np.nanmean(((chl_pred*bio_reg_2)-(chl_anom_test*bio_reg_2))**2,axis=axis))

def IOU_psc(psc_pred, axis=None):
    return np.nanmean(np.minimum(psc_pred,psc_test)/np.maximum(psc_pred,psc_test))

def IOU_psc_pacifique(psc_pred, axis=None):
    return np.nanmean(np.minimum(psc_pred*bio_reg,psc_test*bio_reg)/np.maximum(psc_pred*bio_reg,psc_test*bio_reg))

def NRMSE_chl(chl_pred, axis=None):

    rmse = np.sqrt(np.nanmean((chl_pred - chl_test)**2, axis=axis))
    q75 = np.nanpercentile(chl_test.flatten(), 75, axis=axis)
    q25 = np.nanpercentile(chl_test.flatten(), 25, axis=axis)
    iqr = q75 - q25
    with np.errstate(divide='ignore', invalid='ignore'):
        nrmse = rmse / iqr
        nrmse = np.where(iqr == 0, np.nan, nrmse)

    return nrmse

print('\nRoot Mean Squared Error on Chl-a reconstructed on Test set : ') 
rmse_chl=compute_and_sort_metrics(RMSE_chl,chl_pred_test)
print('\nRoot Mean Squared Error on Chl-a anomalies reconstructed (and computed) on Test set : ') 
rmse_chl_anom=compute_and_sort_metrics(RMSE_chl_anomalies,chl_pred_anom_test)

print('\nPacifique Root Mean Squared Error on Chl-a reconstructed on Test set : ') 
rmse_chl_pacifique=compute_and_sort_metrics(RMSE_chl_pacifique,chl_pred_test)
print('\nPacifique Root Mean Squared Error on Chl-a anomalies reconstructed (and computed) on Test set : ') 
rmse_chl_anom_pacifique=compute_and_sort_metrics(RMSE_chl_anomalies_pacifique,chl_pred_anom_test)

print('\nAtlantique Root Mean Squared Error on Chl-a reconstructed on Test set : ') 
rmse_chl_atlantique=compute_and_sort_metrics(RMSE_chl_atlantique,chl_pred_test)
print('\nAtlantique Root Mean Squared Error on Chl-a anomalies reconstructed (and computed) on Test set : ') 
rmse_chl_anom_atlantique=compute_and_sort_metrics(RMSE_chl_anomalies_atlantique,chl_pred_anom_test)

print('\nIOU on PSC reconstructed (and computed) on Test set : ') 
iou_psc=compute_and_sort_metrics(IOU_psc,psc_pred_test)
print('\nPacifique IOU on PSC reconstructed (and computed) on Test set : ') 
iou_psc_pacifique=compute_and_sort_metrics(IOU_psc_pacifique,psc_pred_test)

print('\nNormalized Root Mean Squared Error on Chl-a reconstructed on Test set : ') 
nrmse_chl=compute_and_sort_metrics(NRMSE_chl,chl_pred_test)

### L1 Metric

In [ ]:
    
def MAE_chl(chl_pred, axis=None):
    return np.nanmean(np.abs(chl_pred-chl_test),axis=axis)

def MAE_chl_anomalies(chl_pred, axis=None):
    return np.nanmean(np.abs(chl_pred-chl_anom_test),axis=axis)

def MAE_chl_pacifique(chl_pred, axis=None):
    return np.nanmean(np.abs((chl_pred*bio_reg)-(chl_test*bio_reg)),axis=axis)


def MAE_chl_anomalies_pacifique(chl_pred, axis=None):
    return np.nanmean(np.abs((chl_pred*bio_reg)-(chl_anom_test*bio_reg)),axis=axis)


print('\nMean abs Error on Chl-a reconstructed on Test set : ') 
mae_chl=compute_and_sort_metrics(MAE_chl,chl_pred_test)
print('\nMean abs Error on Chl-a anomalies reconstructed (and computed) on Test set : ') 
mae_chl_anom=compute_and_sort_metrics(MAE_chl_anomalies,chl_pred_anom_test)

print('\nPacifique Mean abs Error on Chl-a reconstructed on Test set : ') 
mae_chl_pacifique=compute_and_sort_metrics(MAE_chl_pacifique,chl_pred_test)
print('\nPacifique Mean abs Error Error on Chl-a anomalies reconstructed (and computed) on Test set : ') 
mae_chl_anom_pacifique=compute_and_sort_metrics(MAE_chl_anomalies_pacifique,chl_pred_anom_test)

In [ ]:
rmse=np.sqrt(np.nanmean((chl[-156:]-chl_pred_glob['SmaAt_chlm_avw_variables_all'][-156:])**2,axis=0))
imshow_area(rmse,cmap='cividis',vmin=0.001,vmax=1)
rmse=np.sqrt(np.nanmean((psc[-156:,0]-psc_pred_glob['SmaAt_chlm_avw_variables_all'][-156:,0])**2,axis=0))
imshow_area(rmse,cmap='cividis',vmin=0.001,vmax=1)
rmse=np.sqrt(np.nanmean((psc[-156:,1]-psc_pred_glob['SmaAt_chlm_avw_variables_all'][-156:,1])**2,axis=0))
imshow_area(rmse,cmap='cividis',vmin=0.001,vmax=1)
rmse=np.sqrt(np.nanmean((psc[-156:,2]-psc_pred_glob['SmaAt_chlm_avw_variables_all'][-156:,2])**2,axis=0))
imshow_area(rmse,cmap='cividis',vmin=0.001,vmax=1)

# Figure metrique

In [ ]:
matplotlib.rcParams.update({'font.size': 12})
import matplotlib.colors as mcolors

keys=list(rmse_chl.keys())
keys=['SmaAt_chlm_avw_variables_winds',
 'SmaAt_chlm_avw_variables_sss',
 'SmaAt_chlm_avw_variables_all',
 'SmaAt_chlm_avw_variables_ssh',
 'SmaAt_chlm_avw_variables_sst',
 'SmaAt_chlm_avw_variables_solar',
 'SmaAt_chlm_avw_variables_sst_ssh',
 'SmaAt_chlm_avw_variables_mld',
 'SmaAt_chlm_avw_variables_currents',
 'SmaAt_chlm_avw_variables_sst_ssh_solar',
 'SmaAt_chlm_avw_variables_sst_solar',
 #'SmaAt_chlm_avw_variables_sst_ssh_solarA',
 #'SmaAt_chlm_avw_variables_alla',
 #'SmaAt_chlm_avw_variables_sstA',
 #'SmaAt_chlm_avw_variables_allA',
 #'SmaAt_chlm_avw_variables_ssta',
 #'SmaAt_chlm_avw_variables_cosin',
 #'SmaAt_chlm_avw_variables_sst_ssh_solar_spatial',
 'seasonal']

rename=['Winds','SSS','All','SSH','SST','Solar','SST SSH','MLD','Currents','Core Variables', 'SST Solar','Monthly Climatology']# 'Core Variables & Anomalies',
        #'All Anomalies', 'SST & Anomalies', 'All & Anomalies', 'SST Anomalies'] #'cosin','lat',


#keys.remove('ies_mld_sst_ssh_A_pacifique')

data = np.stack([[rmse_chl[key],rmse_chl_anom[key],rmse_chl_pacifique[key],rmse_chl_anom_pacifique[key], iou_psc[key], iou_psc_pacifique[key]] for key in keys])


#models = [f'Model_{i}' for i in keys]  # 10 model names
metrics = ['RMSE Chl', 'RMSE Chl Anom', 'Pacifique RMSE Chl', 'Pacifique RMSE Chl Anom', 'IOU PSC', 'Pacifique IOU PSC']  # metrics



# Create DataFrame
#df = pd.DataFrame(data, index=models, columns=metrics)
df = pd.DataFrame(data, index=keys, columns=metrics)

df=df.rename(index={keys[i]:rename[i] for i in range(len(keys))})

df_sorted = df.sort_values(by=metrics[0], ascending=True)
seasonal_values = df_sorted.loc['Monthly Climatology']

# Create subplots for each metric with different color scales
fig, axes = plt.subplots(1, 6, figsize=(20, 12), gridspec_kw={'width_ratios': [1, 1, 1, 1, 1, 1]}, sharey=True)



# Define color maps for each metric
color_maps = ["coolwarm", "coolwarm", "coolwarm", "coolwarm","coolwarm_r","coolwarm_r"]


for i, metric in enumerate(metrics):
    vmin = df_sorted[metric].min()
    vmax = df_sorted[metric].max()
    midpoint = seasonal_values[metric]  # Set 'seasonal' as the white reference point
    if vmin==midpoint:
        vmin-=0.1
    if vmax==midpoint:
        vmax+=0.1
    # Define custom normalization to center the colormap around 'seasonal'
    norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=midpoint, vmax=vmax)

    # Create heatmap
    sns.heatmap(df_sorted[[metric]], annot=True, cmap=color_maps[i], fmt=".4f", linewidths=0.5, ax=axes[i],  
                cbar=False, norm=norm)

    axes[i].set_title(metric)
    axes[i].set_xlabel("")
    
    # Only show y-axis labels on the first subplot
    if i == 0:
        axes[i].set_ylabel("Experiment")
        axes[i].set_yticklabels(df_sorted.index, rotation=0)

plt.tight_layout()


#plt.savefig("/home/luther/Documents/plot/article_1/heatmap_avw_0.png")

plt.show()



# Figure Metric 1
<br/>
- bioreg digression
<br/>
- megaplotdelamortquitue

In [ ]:
for i,name in enumerate(bioreg_name):
    region=np.where(~np.isnan(bio_regions[...,i])==1,1,np.nan)
    imshow_area(region,title=name+f'{i}')



In [ ]:
new_bio_regions={}
new_bio_regions['Seasonal strat Subtropics North']=np.where(~np.isnan(bio_regions[...,4])==1,1,np.nan)
new_bio_regions['Subtropics North']=np.where(~np.isnan(bio_regions[...,5])==1,1,np.nan)
new_bio_regions['Subtropics South']=np.where(~np.isnan(bio_regions[...,6])==1,1,np.nan)
new_bio_regions['Tropical Pacific']=np.where(~np.isnan(bio_regions[...,7])+~np.isnan(bio_regions[...,8])==1,1,np.nan)
new_bio_regions['Tropical Atlantique']=np.where(~np.isnan(bio_regions[...,9])+~np.isnan(bio_regions[...,10])==1,1,np.nan)
new_bio_regions['Indian ocean']=np.where(~np.isnan(bio_regions[...,11])+~np.isnan(bio_regions[...,12])==1,1,np.nan)
new_bio_regions['Seasonal strat Subtropics South']=np.where(~np.isnan(bio_regions[...,13])==1,1,np.nan)#Seasonal strat Subtropics South
new_bio_regions['SO-SA-high biomass']=np.where(~np.isnan(bio_regions[...,14])==1,1,np.nan)
new_bio_regions['Global']=np.ones(bio_regions[...,0].shape)
#new_bio_regions['ENSO']=EL_NINO_area
#new_bio_regions['test_area']=test_area

In [ ]:


Atlantique_Nord_area=np.zeros((100,360))
Atlantique_Nord_area=np.where(new_bio_regions['Seasonal strat Subtropics North']==1,1,np.nan)
Atlantique_Nord_area[0:60,0:80]=np.nan
Atlantique_Nord_area[0:60,250:]=np.nan

Atlantique_Nord_area[~Atlantique_Nord_area.astype(np.bool_)]=np.nan
imshow_area(Atlantique_Nord_area, colorbar=False, title="North Atlantique Ocean",)#save="/home/luther/Documents/plot/prez_egu2/ENSO_area.png")


Pacifique_Nord_area=np.zeros((100,360))
Pacifique_Nord_area=np.where(new_bio_regions['Seasonal strat Subtropics North']==1,1,np.nan)
Pacifique_Nord_area=np.where(Atlantique_Nord_area==1,np.nan,Pacifique_Nord_area)

Pacifique_Nord_area[~Pacifique_Nord_area.astype(np.bool_)]=np.nan
imshow_area(Pacifique_Nord_area, colorbar=False, title="North Pacific Ocean",)#save="/home/luther/Documents/plot/prez_egu2/ENSO_area.png")


In [ ]:
plot_bio_reg=np.zeros_like(new_bio_regions['Global'])

for i,name in enumerate(new_bio_regions.keys()):
    if name!='Global':
        plot_bio_reg=np.where(new_bio_regions[name]==1,i+1,plot_bio_reg)
imshow_area(plot_bio_reg,cmap='tab10',title='bioregions', colorbar=True)#, save='/home/luther/Documents/plot/article_1/bio_regions_1.png')

In [ ]:
imshow_area(np.nanmean(psc[:,0]*chl,axis=0),cmap='jet',title='Micro mg/m3',log=True, vmin=-3, vmax=1,)
           #save='/home/luther/Documents/plot/article_1/micro.png')
imshow_area(np.nanmean(psc[:,1]*chl,axis=0),cmap='jet',title='Nano mg/m3',log=True, vmin=-3, vmax=1,)
           #save='/home/luther/Documents/plot/article_1/nano.png')
imshow_area(np.nanmean(psc[:,2]*chl,axis=0),cmap='jet',title='Pico mg/m3',log=True, vmin=-3, vmax=1,)
           #save='/home/luther/Documents/plot/article_1/pico.png')

### Bioregions plot

In [ ]:
new_bio_regions.keys()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
import cartopy.crs as ccrs

bio_regions_for_plot=new_bio_regions
bio_regions_for_plot['Seasonal strat Subtropics Atlantic North']=Atlantique_Nord_area
bio_regions_for_plot['Seasonal strat Subtropics Pacific North']=Pacifique_Nord_area

#PLOT

matplotlib.rcParams.update({'font.size': 7})
proj=ccrs.Robinson(central_longitude=200)
fig = plt.figure(figsize=(7, 3), constrained_layout=False, dpi=300)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=0.7,  top=1., bottom=0.),  # bioregion
    fig.add_gridspec(nrows=1, ncols=1, left=0.7, right=1.,  top=1., bottom=0.),  #legend
]

axes = [fig.add_subplot(grid_specs[0][0,0], projection=proj),
       fig.add_subplot(grid_specs[1][0,0])]


plot_bio_reg=np.zeros_like(new_bio_regions['Global'])

for i,name in enumerate(bio_regions_for_plot.keys()):
    if name!='Global' and name!='Seasonal strat Subtropics North':
        plot_bio_reg=np.where(bio_regions_for_plot[name]==1,i+1,plot_bio_reg)
        
imshow_area(plot_bio_reg,cmap='tab20',title='bioregions', colorbar=False, fig=fig, ax=axes[0])

labels = ['Other'] + list(bio_regions_for_plot.keys())
labels.remove('Global')
labels.remove('Seasonal strat Subtropics North')
cmap = plt.cm.get_cmap('tab10')
colors = [cmap(i) for i in range(len(labels))]
colors = [cmap(0),cmap(1),cmap(2),cmap(3),cmap(6),cmap(5),cmap(7),cmap(8),cmap(9),cmap(4)]
# Create color patches for legend
patches = [mpatches.Patch(color=color, label=label) for color, label in zip(colors, labels)]

# Plot an empty figure with just the legend
axes[1].axis('off')  # Hide axes
legend = axes[1].legend(handles=patches, loc='center left', frameon=False)
plt.tight_layout()
#plt.savefig('/home/luther/Documents/plot/article_1/color_scale_bioreg.png',bbox_inches='tight' )
plt.show()

### Attention stop

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(np.nanmean((y_pred - y_true) ** 2))


def pearson_corr(y_true, y_pred):
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    if np.sum(mask) < 2:
        return np.nan
    return np.corrcoef(y_true[mask].flatten(), y_pred[mask].flatten())[0, 1]


def cross_entropy(y_true, y_pred, eps=1e-12):
    """
    y_true, y_pred shape: (time, classes, lat, lon) or similar
    axis=1 is classes (PSC)
    """
    y_pred = np.clip(y_pred, eps, 1.0)  # avoid log(0)
    ce = -np.sum(y_true * np.log(y_pred), axis=1)  # sum over classes
    return np.nanmean(ce)
    
def spatial_mean(data, area=None):
    if area is not None:
        data = data * area
    return data #np.nanmean(data, axis=(-2, -1))

keys=['SmaAt_chlm_avw_variables_winds',
 'SmaAt_chlm_avw_variables_sss',
 'SmaAt_chlm_avw_variables_all',
 'SmaAt_chlm_avw_variables_ssh',
 'SmaAt_chlm_avw_variables_sst',
 'SmaAt_chlm_avw_variables_solar',
 'SmaAt_chlm_avw_variables_mld',
 'SmaAt_chlm_avw_variables_currents']

def evaluate_experiments(
    chl_true,
    chl_pred_dict,
    psc_true,
    psc_pred_dict,
    area=None
):
    results = {}

    class_names = ["Micro", "Nano", "Pico"]

    for exp in keys:

        chl_pred = chl_pred_dict[exp]
        psc_pred = psc_pred_dict[exp]

        # --- Spatial averaging ---
        chl_t = spatial_mean(chl_true, area)
        chl_p = spatial_mean(chl_pred, area)

        psc_t = spatial_mean(psc_true, area)   # shape (time, class)
        psc_p = spatial_mean(psc_pred, area)

        # --- Base metrics ---
        res = {
            "RMSE_chl": rmse(chl_t, chl_p),
            "Corr_chl": pearson_corr(chl_t, chl_p),

            "CrossEntropy_psc": cross_entropy(psc_t, psc_p)
        }

        # --- Per-class metrics ---
        for i, cname in enumerate(class_names):

            y_true_c = psc_t[:, i]
            y_pred_c = psc_p[:, i]

            res[f"RMSE_psc%_{cname}"] = rmse(y_true_c, y_pred_c)
            res[f"RMSE_psc_{cname}"] = rmse(y_true_c*chl_t,y_pred_c*chl_p)
            res[f"Corr_psc_{cname}"] = pearson_corr(y_true_c, y_pred_c)

        results[exp] = res

    return results

In [ ]:
import pandas as pd
imshow_area(new_bio_regions['Seasonal strat Subtropics South'])
results = evaluate_experiments(
    chl_test,
    chl_pred_test,
    psc_test,
    psc_pred_test,
    area=new_bio_regions['Seasonal strat Subtropics South']  
)

In [ ]:
results

In [ ]:
"""
Metrics
"""
from scipy.stats import spearmanr
from tqdm import tqdm 

rmse_chl_bio_regions={}
rmse_chl_anomalies_bio_regions={}
ce_psc_bio_regions={}
corr_chl_bio_regions={}
acc_chl_anomalies_bio_regions={}


#iou_psc_bio_regions={}
#rmse_chl_climato_bio_regions={}

for name,regions in tqdm(new_bio_regions.items()):
    
    def RMSE_bio_region(chl_pred):
        return np.sqrt(np.nanmean(((chl_pred*regions)-(chl_test*regions))**2,axis=None))
        #return np.nanmean(np.abs((chl_pred*regions)-(chl_test*regions)),axis=None)

    def RMSE_chl_anomalies_bio_region(chl_pred):
        return np.sqrt(np.nanmean(((chl_pred*regions)-(chl_anom_test*regions))**2,axis=None))

    def cross_entropy_psc_region(psc_pred, psc_test=psc_test, regions=regions):
    
        eps = 1e-12
        psc_pred_clipped = np.clip(psc_pred, eps, 1. - eps)*regions # Clip to avoid log(0)
    
        # Cross-entropy: sum over classes
        ce = -np.nansum(psc_test*regions * np.log(psc_pred_clipped), axis=1)  # shape: (time, lat, lon)
        
        return np.nanmean(ce)


    #def IOU_psc_bio_region(psc_pred):
    #    return np.nanmean(np.minimum(psc_pred*regions,psc_test*regions)/np.maximum(psc_pred*regions,psc_test*regions))

    
    def spearman_chl_region(chl_pred, chl_test=chl_test, regions=regions):
    
        time, lat, lon = chl_pred.shape
        spearman_vals = np.full((lat, lon), np.nan)
    
        for i in range(lat):
            for j in range(lon):
                if not(np.isnan(regions[i, j])) :
                    x = chl_pred[:, i, j]
                    y = chl_test[:, i, j]
                    if np.all(np.isnan(x)) or np.all(np.isnan(y)):
                        continue
                    valid = ~np.isnan(x) & ~np.isnan(y)
                    if np.sum(valid) >= 2:  # Need at least 2 points
                        r, _ = spearmanr(x[valid], y[valid])
                        spearman_vals[i, j] = r
    
        return np.nanmean(spearman_vals)

    
    def sign_accuracy(chl_anomalies_pred, chl_anomalies_test=chl_anom_test, regions=regions):

        # Get sign: np.sign returns -1, 0, or 1
        pred_sign = np.sign(chl_anomalies_pred)
        test_sign = np.sign(chl_anomalies_test)
    
        # Equal sign => correct prediction
        correct = (pred_sign == test_sign)
    
        # Handle optional region masking
        if regions is not None:
            region_mask = np.broadcast_to(regions, correct.shape)
            correct = np.where(region_mask==1, correct, np.nan)
    
        # Ignore positions with NaNs in either input
        valid = ~np.isnan(chl_anomalies_pred) & ~np.isnan(chl_anomalies_test)
        correct = np.where(valid, correct, np.nan)
    
        return np.nanmean(correct)



    rmse_chl_bio_regions[name]={key: RMSE_bio_region(value) for key, value in chl_pred_test.items()}
    rmse_chl_anomalies_bio_regions[name]={key: RMSE_chl_anomalies_bio_region(value) for key, value in chl_pred_anom_test.items()}
    ce_psc_bio_regions[name]={key: cross_entropy_psc_region(value) for key, value in psc_pred_test.items()}
    corr_chl_bio_regions[name]={key: spearman_chl_region(value) for key, value in chl_pred_test.items()}
    acc_chl_anomalies_bio_regions[name]={key: sign_accuracy(value) for key, value in chl_pred_anom_test.items()}
    #iou_psc_bio_regions[name]={key: IOU_psc_bio_region(value) for key, value in psc_pred_test.items()}
    



"""
Dataframe
"""

keys=['SmaAt_chlm_avw_variables_winds',
 'SmaAt_chlm_avw_variables_sss',
 'SmaAt_chlm_avw_variables_all',
 'SmaAt_chlm_avw_variables_ssh',
 'SmaAt_chlm_avw_variables_sst',
 'SmaAt_chlm_avw_variables_solar',
 'SmaAt_chlm_avw_variables_sst_ssh',
 'SmaAt_chlm_avw_variables_mld',
 'SmaAt_chlm_avw_variables_currents',
 'SmaAt_chlm_avw_variables_sst_ssh_solar',
 'SmaAt_chlm_avw_variables_sst_solar',
 #'SmaAt_chlm_avw_variables_sst_ssh_solarA',
 #'SmaAt_chlm_avw_variables_alla',
 #'SmaAt_chlm_avw_variables_sstA',
 #'SmaAt_chlm_avw_variables_allA',
 #'SmaAt_chlm_avw_variables_ssta',
 #'SmaAt_chlm_avw_variables_cosin',
 #'SmaAt_chlm_avw_variables_sst_ssh_solar_spatial',
 'seasonal']

rename=['Winds','SSS','All','SSH','SST','Solar','SST SSH','MLD','Currents','SST Solar SSH', 'SST Solar','Monthly Climatology'] #'cosin','lat'


data_chl_bioregions = np.stack([[rmse_chl_bio_regions[key1][key2] for key1 in rmse_chl_bio_regions.keys()] for key2 in keys])
data_chl_anom_bioregions = np.stack([[rmse_chl_anomalies_bio_regions[key1][key2] for key1 in rmse_chl_anomalies_bio_regions.keys()] for key2 in keys])
data_ce_psc_bioregions = np.stack([[ce_psc_bio_regions[key1][key2] for key1 in ce_psc_bio_regions.keys()] for key2 in keys])
data_corr_chl_bioregions = np.stack([[corr_chl_bio_regions[key1][key2] for key1 in corr_chl_bio_regions.keys()] for key2 in keys])
data_acc_chl_anom_bioregions = np.stack([[acc_chl_anomalies_bio_regions[key1][key2] for key1 in acc_chl_anomalies_bio_regions.keys()] for key2 in keys])


#data_psc_bioregions = np.stack([[iou_psc_bio_regions[key1][key2] for key1 in iou_psc_bio_regions.keys()] for key2 in keys])


#models = [f'Model_{i}' for i in keys]  # 10 model names
metrics = list(new_bio_regions.keys())


# Create DataFrame
df_chl_bio_regions = pd.DataFrame(data_chl_bioregions, index=keys, columns=metrics)
df_chl_anom_bio_regions = pd.DataFrame(data_chl_anom_bioregions, index=keys, columns=metrics)
df_ce_psc_bioregions = pd.DataFrame(data_ce_psc_bioregions, index=keys, columns=metrics)
df_corr_chl_bioregions = pd.DataFrame(data_corr_chl_bioregions, index=keys, columns=metrics)
df_acc_chl_anom_bioregions = pd.DataFrame(data_acc_chl_anom_bioregions, index=keys, columns=metrics)
#df_psc_bio_regions = pd.DataFrame(data_psc_bioregions, index=keys, columns=metrics)

# rename columns 
for df in [df_chl_bio_regions,df_chl_anom_bio_regions,df_ce_psc_bioregions,df_corr_chl_bioregions,df_acc_chl_anom_bioregions]:
    df.rename(index={keys[i]:rename[i] for i in range(len(keys))},inplace=True)

#df_psc_bio_regions=df_psc_bio_regions.rename(index={keys[i]:rename[i] for i in range(len(keys))})

#sort
df_sorted_chl_bio_regions=df_chl_bio_regions.sort_values(by=metrics[-1], ascending=True)
df_sorted_chl_anom_bio_regions = df_chl_anom_bio_regions.loc[df_sorted_chl_bio_regions.index]
df_sorted_ce_psc_bioregions = df_ce_psc_bioregions.loc[df_sorted_chl_bio_regions.index]
df_sorted_corr_chl_bioregions = df_corr_chl_bioregions.loc[df_sorted_chl_bio_regions.index]
df_sorted_acc_chl_anom_bioregions = df_acc_chl_anom_bioregions.loc[df_sorted_chl_bio_regions.index]



In [ ]:
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'font.size': 26})

fig = plt.figure(figsize=(45, 25), constrained_layout=False)#,dpi=800)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=5, left=0.0, right=0.18, top=0.95, bottom=0.5, wspace=0.05),  # First column
    fig.add_gridspec(nrows=1, ncols=5, left=0.2, right=0.38, top=0.95, bottom=0.5, wspace=0.05),  # Second column
    fig.add_gridspec(nrows=1, ncols=5, left=0.4, right=0.58, top=0.95, bottom=0.5,wspace=0.05),  # Third column
    fig.add_gridspec(nrows=1, ncols=5, left=0.6, right=0.78, top=0.95, bottom=0.5,wspace=0.05),   # Fourth column
    fig.add_gridspec(nrows=1, ncols=5, left=0.8, right=0.98, top=0.95, bottom=0.5,wspace=0.05),   # Fifth column
    fig.add_gridspec(nrows=1, ncols=5, left=0.09, right=0.27, top=0.45, bottom=0., wspace=0.05),   # Sixth column
    fig.add_gridspec(nrows=1, ncols=5, left=0.29, right=0.47, top=0.45, bottom=0.,wspace=0.05),   # Seventh column
    fig.add_gridspec(nrows=1, ncols=5, left=0.49, right=0.67, top=0.45, bottom=0., wspace=0.05),   # Height column
    fig.add_gridspec(nrows=1, ncols=5, left=0.69, right=0.89, top=0.45, bottom=0., wspace=0.05),   # Nineth column
]

axes = [  
    [fig.add_subplot(gs[:, 0]), fig.add_subplot(gs[:, 1]), fig.add_subplot(gs[:, 2]), 
    fig.add_subplot(gs[:, 3]), fig.add_subplot(gs[:, 4])] for gs in grid_specs  
]

# Add ghost axes and titles for gs
ghost_axes = [fig.add_subplot(gs[:]) for gs in grid_specs]

# Define color maps
color_maps = ["coolwarm", "coolwarm", "coolwarm", "coolwarm_r", "Blues"]
metric_names=['rmse chl', 'chl anom', 'ce psc', 'corr chl', 'acc']
for j,dataframe in enumerate([df_sorted_chl_bio_regions,df_sorted_chl_anom_bio_regions,
                              df_sorted_ce_psc_bioregions, df_sorted_corr_chl_bioregions,
                              df_sorted_acc_chl_anom_bioregions]):
    color_map=color_maps[j]
    for i, metric in enumerate(metrics):

        unique_sorted = np.sort(np.unique(dataframe[metric]))
        vmin = unique_sorted[0]
        vmax =  unique_sorted[-1]
        midpoint = dataframe.loc['Monthly Climatology'][metric]

        data=dataframe[[metric]]


        # Define custom normalization to center the colormap around 'seasonal'
        if j<4:
            norm = mcolors.TwoSlopeNorm(vmin=midpoint-(np.abs(midpoint)/2), vcenter=midpoint, vmax=midpoint+(np.abs(midpoint)/2))
        #norm = mcolors.TwoSlopeNorm(vcenter=midpoint)
        else:
            vmin = unique_sorted[1]
            norm=mcolors.TwoSlopeNorm(vmin=vmin, vcenter=(vmax+vmin)/2, vmax=vmax)
            data.loc['Monthly Climatology']=np.nan
        # Create heatmap75
        sns.heatmap(data, annot=True, cmap=color_map, fmt=".4f", linewidths=0.5, ax=axes[i][j],  
                    cbar=False, norm=norm, xticklabels=False)

        ghost_axes[i].axis('off')
        ghost_axes[i].set_title(metric)

        axes[i][j].set_xlabel(metric_names[j])

        if (i == 0 or i==5) and j==0 :
            axes[i][j].set_ylabel("Experiment")
            axes[i][j].set_yticklabels(dataframe.index, rotation=0)
        else:
            axes[i][j].get_yaxis().set_ticks([])

#plt.savefig("/home/luther/Documents/plot/article_1/heatmap_monthly_avw_psc_2.png", bbox_inches='tight' )

plt.show()

In [ ]:
perf_rmse_chl={}
perf_rmse_chl_anomalies={}
perf_ce_psc={}

for name,regions in new_bio_regions.items():

    def perf_rmse_bio_region(rmse_pred):
        return ((rmse_chl_bio_regions[name]['seasonal']-rmse_pred)/rmse_chl_bio_regions[name]['seasonal'])*100

    def perf_rmse_anomalies_bio_region(rmse_anom_pred):
        return ((rmse_chl_anomalies_bio_regions[name]['seasonal']-rmse_anom_pred)/rmse_chl_anomalies_bio_regions[name]['seasonal'])*100

    def perf_ce_psc_bio_region(ce_psc_pred):
        return ((ce_psc_pred-ce_psc_bio_regions[name]['seasonal'])/ce_psc_bio_regions[name]['seasonal'])*100

    perf_rmse_chl[name]={key: perf_rmse_bio_region(value) for key, value in rmse_chl_bio_regions[name].items()}
    perf_rmse_chl_anomalies[name]={key: perf_rmse_anomalies_bio_region(value) for key, value in rmse_chl_anomalies_bio_regions[name].items()}
    perf_ce_psc[name]={key: perf_ce_psc_bio_region(value) for key, value in ce_psc_bio_regions[name].items()}


metrics = list(new_bio_regions.keys())


data_perf_rmse_chl = np.stack([[perf_rmse_chl[key1][key2] for key1 in metrics] for key2 in keys])
data_perf_rmse_chl_anomalies = np.stack([[perf_rmse_chl_anomalies[key1][key2] for key1 in metrics] for key2 in keys])
data_perf_ce_psc = np.stack([[perf_ce_psc[key1][key2] for key1 in metrics] for key2 in keys])


#models = [f'Model_{i}' for i in keys]  # 10 model names


# Create DataFrame
#df = pd.DataFrame(data, index=models, columns=metrics)
df_perf_rmse_chl = pd.DataFrame(data_perf_rmse_chl, index=keys, columns=metrics)
df_perf_rmse_chl_anomalies = pd.DataFrame(data_perf_rmse_chl_anomalies, index=keys, columns=metrics)
df_perf_ce_psc = pd.DataFrame(data_perf_ce_psc, index=keys, columns=metrics)
# rename columns 
for df in [df_perf_rmse_chl,df_perf_rmse_chl_anomalies,df_perf_ce_psc]:
    df.rename(index={keys[i]:rename[i] for i in range(len(keys))},inplace=True)
    
df_sorted_perf_rmse_chl = df_perf_rmse_chl.loc[df_sorted_chl_bio_regions.index]
df_sorted_perf_rmse_chl_anomalies = df_perf_rmse_chl_anomalies.loc[df_sorted_chl_bio_regions.index]
df_sorted_perf_ce_psc = df_perf_ce_psc.loc[df_sorted_chl_bio_regions.index]


import matplotlib.pyplot as plt
matplotlib.rcParams.update({'font.size': 24})

fig = plt.figure(figsize=(45, 25), constrained_layout=False)#,dpi=800)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=5, left=0.0, right=0.18, top=0.95, bottom=0.5, wspace=0.05),  # First column
    fig.add_gridspec(nrows=1, ncols=5, left=0.2, right=0.38, top=0.95, bottom=0.5, wspace=0.05),  # Second column
    fig.add_gridspec(nrows=1, ncols=5, left=0.4, right=0.58, top=0.95, bottom=0.5,wspace=0.05),  # Third column
    fig.add_gridspec(nrows=1, ncols=5, left=0.6, right=0.78, top=0.95, bottom=0.5,wspace=0.05),   # Fourth column
    fig.add_gridspec(nrows=1, ncols=5, left=0.8, right=0.98, top=0.95, bottom=0.5,wspace=0.05),   # Fifth column
    fig.add_gridspec(nrows=1, ncols=5, left=0.09, right=0.27, top=0.45, bottom=0., wspace=0.05),   # Sixth column
    fig.add_gridspec(nrows=1, ncols=5, left=0.29, right=0.47, top=0.45, bottom=0.,wspace=0.05),   # Seventh column
    fig.add_gridspec(nrows=1, ncols=5, left=0.49, right=0.67, top=0.45, bottom=0., wspace=0.05),   # Height column
    fig.add_gridspec(nrows=1, ncols=5, left=0.69, right=0.89, top=0.45, bottom=0., wspace=0.05),   # Nineth column
]

axes = [  
    [fig.add_subplot(gs[:, 0]), fig.add_subplot(gs[:, 1]), fig.add_subplot(gs[:, 2]), 
    fig.add_subplot(gs[:, 3]), fig.add_subplot(gs[:, 4])] for gs in grid_specs  
]

# Add ghost axes and titles for gs
ghost_axes = [fig.add_subplot(gs[:]) for gs in grid_specs]

# Define color maps
color_maps = ["coolwarm_r", "coolwarm_r", "coolwarm_r", "coolwarm_r", "Blues"]
metric_names=['rmse chl', 'chl anom', 'ce psc', 'corr chl', 'acc']
for j,dataframe in enumerate([df_sorted_perf_rmse_chl,df_sorted_perf_rmse_chl_anomalies,
                              df_sorted_perf_ce_psc, df_sorted_corr_chl_bioregions,
                              df_sorted_acc_chl_anom_bioregions]):
    color_map=color_maps[j]
    for i, metric in enumerate(metrics):

        unique_sorted = np.sort(np.unique(dataframe[metric]))
        vmin = unique_sorted[0]
        vmax =  unique_sorted[-1]
        midpoint = dataframe.loc['Monthly Climatology'][metric]
        data=dataframe[[metric]]

        
        #data=dataframe[[metric]].values

        if j<3:
            data.loc['Monthly Climatology']=np.nan
            norm = mcolors.TwoSlopeNorm(vcenter=0)
            labels = (np.asarray(["{0:.2f}%".format(value)
                    for value in data.values.flatten()])).reshape(*data.values.shape)
            fmt=""


        elif j==3:
            midpoint = dataframe.loc['Monthly Climatology'][metric]
            norm = mcolors.TwoSlopeNorm(vmin=midpoint-(np.abs(midpoint)/2), vcenter=midpoint, vmax=midpoint+(np.abs(midpoint)/2))
            labels=True
            fmt=".4f"

        else:
            labels=True
            fmt=".4f"            
            vmin = unique_sorted[1]
            norm=mcolors.TwoSlopeNorm(vmin=vmin, vcenter=(vmax+vmin)/2, vmax=vmax)
            data.loc['Monthly Climatology']=np.nan

        # Create heatmap
        sns.heatmap(data, annot=labels, cmap=color_map, fmt=fmt, linewidths=0.5, ax=axes[i][j],  
                    cbar=False, norm=norm, xticklabels=False)


        ghost_axes[i].axis('off')
        ghost_axes[i].set_title(metric)

        axes[i][j].set_xlabel(metric_names[j])

        if (i == 0 or i==5) and j==0 :
            axes[i][j].set_ylabel("Experiment")
            axes[i][j].set_yticklabels(dataframe.index, rotation=0)
        else:
            axes[i][j].get_yaxis().set_ticks([])

#plt.savefig("/home/luther/Documents/plot/article_1/heatmap_monthly_perc_avw_psc_2.png", bbox_inches='tight' )

plt.show()

In [ ]:
"""
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'font.size': 20})

fig = plt.figure(figsize=(45, 16), constrained_layout=False)#,dpi=800)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=3, left=0.02, right=0.11, wspace=0.05),  # First column
    fig.add_gridspec(nrows=1, ncols=3, left=0.13, right=0.22, wspace=0.05),  # Second column
    fig.add_gridspec(nrows=1, ncols=3, left=0.24, right=0.33, wspace=0.05),  # Third column
    fig.add_gridspec(nrows=1, ncols=3, left=0.35, right=0.44, wspace=0.05),   # Fourth column
    fig.add_gridspec(nrows=1, ncols=3, left=0.46, right=0.55, wspace=0.05),   # Fifth column
    fig.add_gridspec(nrows=1, ncols=3, left=0.57, right=0.66, wspace=0.05),   # Sixth column
    fig.add_gridspec(nrows=1, ncols=3, left=0.68, right=0.77, wspace=0.05),   # Seventh column
    fig.add_gridspec(nrows=1, ncols=3, left=0.79, right=0.88, wspace=0.05),   # Height column
    fig.add_gridspec(nrows=1, ncols=3, left=0.90, right=0.99, wspace=0.05),   # Nineth column
]

axes = [  
    [fig.add_subplot(gs[:, 0]), fig.add_subplot(gs[:, 1]), fig.add_subplot(gs[:, 2])] for gs in grid_specs  
]

# Add ghost axes and titles for gs
ghost_axes = [fig.add_subplot(gs[:]) for gs in grid_specs]

# Define color maps
color_maps = ["coolwarm_r", "coolwarm_r", "coolwarm"]
for j,dataframe in enumerate([df_sorted_chl_bio_regions,df_sorted_chl_anom_bio_regions,df_sorted_chl_climato_bio_regions]):
    color_map=color_maps[j]
    for i, metric in enumerate(metrics):


        vmin = dataframe[metric].min()
        vmax = dataframe[metric].max()
        #midpoint = seasonal_values_chl[metric] if j==0 or j==1 else seasonal_values_psc[metric]  # Set 'seasonal' as the white reference point
        midpoint = dataframe.loc['Monthly Climatology'][metric]
        if vmin==midpoint:
            vmin-=0.1
        if vmax==midpoint:
            vmax+=0.1

        # Define custom normalization to center the colormap around 'seasonal'
        #norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=midpoint, vmax=vmax)
        norm = mcolors.TwoSlopeNorm(vcenter=midpoint)
        norm=None
        # Create heatmap
        sns.heatmap(dataframe[[metric]], annot=True, cmap=color_map, fmt=".4f", linewidths=0.5, ax=axes[i][j],  
                    cbar=False, norm=norm, xticklabels=False)

        ghost_axes[i].axis('off')
        ghost_axes[i].set_title(metric)

        
        # Only show y-axis labels on the first subplot
        if j==0:
            axes[i][j].set_xlabel("Chl")
            if i == 0:
                axes[i][j].set_ylabel("Experiment")
                axes[i][j].set_yticklabels(dataframe.index, rotation=0)
        elif j==1 :
            axes[i][j].set_xlabel("Chl anom")
        else :
            axes[i][j].set_xlabel("Chl climato")
            
        if not(j==0 and i==0):
            axes[i][j].get_yaxis().set_ticks([])

plt.savefig("/home/luther/Documents/plot/article_1/heatmap_monthly_avw_psc_2.png", bbox_inches='tight' )

plt.show()
"""
print()

# EOF

In [ ]:
import numpy as np
import xarray as xr
from scipy.stats import pearsonr
from sklearn.decomposition import PCA
enso34=np.loadtxt("/home/luther/Documents/npy_data/oni.txt")
enso34=np.loadtxt("/home/luther/Documents/npy_data/pdo_1993_2024.txt")[:-1]


# Normalize anomalies by their standard deviation
#chl_anom_norm = chl_anom[736:] / np.nanstd(chl_anom[644:828] ,axis=0)
#chl_pred_glob_anom_bonus_norm = chl_pred_glob_anom_bonus[644+230:828+230] / np.nanstd(chl_pred_glob_anom_bonus[644+230:230+828],axis=0)

# Perform EOF (PCA) analysis
# Flatten spatial dimensions and apply PCA
# EOF on interannual anomalies
flat_data = np.nan_to_num(chl_anom[-156:]/np.nanstd(chl_anom[-156:] ,axis=0)).reshape(156,360*100)
pca = PCA()
interannual_pcs = pca.fit_transform(flat_data)
n_components=pca.components_.shape[0]
interannual_patterns = pca.components_.reshape((n_components, 360*100))
interannual_variance = pca.explained_variance_ratio_


def project_onto_eof(reconstructed_data, patterns):
    flat_data = np.nan_to_num(reconstructed_data.reshape(156,360*100))
    projections = np.dot(flat_data, patterns.T)
    return projections

reconstructed_interannual_pcs={}
interannual_correlations={}

for key in chl_pred_glob_anom.keys():
    reconstructed_interannual_pcs[key] = project_onto_eof(chl_pred_anom_test[key]/np.nanstd(chl_pred_anom_test[key],axis=0), interannual_patterns)

    # Compare temporal PCs using Pearson correlation
    interannual_correlations[key] = pearsonr(interannual_pcs[:, 0], reconstructed_interannual_pcs[key][:, 0])[0]


filtered_data = {k: v for k, v in interannual_correlations.items() if not np.isnan(v)}
sorted_keys = sorted(filtered_data, key=filtered_data.get)
interannual_correlations = {key: (filtered_data[key], sorted_keys.index(key)) for key in filtered_data}





In [ ]:
from cmocean import cm
imshow_area(pca.components_[0].reshape(100,360),cmap=cm.curl,vmin=-0.01,vmax=0.01,)
           #save="/home/luther/Documents/plot/prez_egu2/pca_1.png")

In [ ]:
imshow_area(pca.components_[1].reshape(100,360),cmap='coolwarm',vmin=-0.015,vmax=0.015,)
            #save="/home/luther/Documents/plot/prez_egu2/pca_2.png")

### EOF seasonal 

In [ ]:
for key in new_bio_regions.keys():
    chl_bio_reg=chl*new_bio_regions[key]
    data=(chl_bio_reg[-156:]-np.nanmean(chl_bio_reg[-156:]))/np.nanstd(chl_bio_reg[-156:] ,axis=0)
    flat_data = data.reshape(156,360*100)
    mask = ~np.all(np.isnan(flat_data), axis=0)
    flat_data = np.nan_to_num(flat_data[:, mask]) 
    pca_season = PCA()
    season_pcs = pca_season.fit_transform(flat_data)
    n_components=pca_season.components_.shape[0]
    #season_patterns = pca_season.components_.reshape((n_components, 360*100))
    season_variance = pca_season.explained_variance_ratio_

    print(f"{key} 3 first modes : {np.round(season_variance[0],2)*100}%",
    f",{np.round(season_variance[1],2)*100}%,{np.round(season_variance[2],2)*100}%")

    chl_bio_reg=chl_pred_glob['SmaAt_chlm_avw_variables_sst']*new_bio_regions[key]
    data=(chl_bio_reg[-156:]-np.nanmean(chl_bio_reg[-156:]))/np.nanstd(chl_bio_reg[-156:] ,axis=0)
    flat_data = data.reshape(156,360*100)
    mask = ~np.all(np.isnan(flat_data), axis=0)
    flat_data = np.nan_to_num(flat_data[:, mask]) 
    pca_season = PCA()
    season_pcs = pca_season.fit_transform(flat_data)
    n_components=pca_season.components_.shape[0]
    #season_patterns = pca_season.components_.reshape((n_components, 360*100))
    season_variance = pca_season.explained_variance_ratio_

    print(f"{key} 3 first modes pred sst: {np.round(season_variance[0],2)*100}%",
    f",{np.round(season_variance[1],2)*100}%,{np.round(season_variance[2],2)*100}%")

    chl_bio_reg=chl_pred_glob['SmaAt_chlm_avw_variables_all']*new_bio_regions[key]
    data=(chl_bio_reg[-156:]-np.nanmean(chl_bio_reg[-156:]))/np.nanstd(chl_bio_reg[-156:] ,axis=0)
    flat_data = data.reshape(156,360*100)
    mask = ~np.all(np.isnan(flat_data), axis=0)
    flat_data = np.nan_to_num(flat_data[:, mask]) 
    pca_season = PCA()
    season_pcs = pca_season.fit_transform(flat_data)
    n_components=pca_season.components_.shape[0]
    #season_patterns = pca_season.components_.reshape((n_components, 360*100))
    season_variance = pca_season.explained_variance_ratio_

    print(f"{key} 3 first modes pred all: {np.round(season_variance[0],2)*100}%",
    f",{np.round(season_variance[1],2)*100}%,{np.round(season_variance[2],2)*100}%")

In [ ]:

keys=list(reconstructed_interannual_pcs.keys())
#keys.remove('all_period_test')
#keys.remove('cosin')
#keys.remove('all_0')
#keys.remove('all_ocean')
#keys.remove('bathy')


# Plot reconstructed EOFs for each dataset in the dictionary
colors = ["cyan", "orange", "purple", "green", "red"]  # Define different colors for each dataset
# Generate colors from the "coolwarm" colormap
cmap = plt.cm.get_cmap("coolwarm", len(reconstructed_interannual_pcs.keys()))
colors = [cmap(i) for i in range(len(reconstructed_interannual_pcs.keys()))]  # Sample colors

for i, key in enumerate(keys):
    # Plot configuration
    pcs=reconstructed_interannual_pcs[key]
    corr=interannual_correlations[key][0]
    idx_color=interannual_correlations[key][1]
    
    fig, ax1 = plt.subplots(figsize=(10, 4))
    year = np.linspace(2011, 2023, 156)
    
    
    # Plot primary curves
    ax1.plot(year, interannual_pcs[:, 0], color="black", label="avw", linewidth=1.5)

    ax1.plot(year, pcs[:, 0], color=colors[idx_color], label=f"Reconstructed ({key})", linewidth=1.5)
    
    ax1.text(0.25, 0.97, f"Pearson corr : {np.round(corr, 2)}\n", transform=ax1.transAxes,
            ha='left', va='top', color='black')

    # Configure primary y-axis
    ax1.set_ylabel("PC Amplitude")
    ax1.axhline(0, color="black", linewidth=0.8, linestyle="--")
    #ax1.set_xlim([2012, 2015])
    ax1.set_ylim([-75, 75])
    ax1.set_xlabel("Year")
    ax1.legend(loc="upper left", frameon=False)
    
    # Add secondary y-axis for MEI Index
    ax2 = ax1.twinx()
    yearm = np.linspace(2011, 2023, 156)
    
    ax2.plot(yearm, enso34[-13:].flatten(), color="gray", linestyle="--", label="MEI Index")
    ax2.set_ylabel("MEI Index")
    ax2.set_ylim([-2.5, 2.5])
    
    # Add secondary y-axis legend
    ax2.legend(loc="upper right", frameon=False)
    
    # Final layout adjustments
    plt.tight_layout()
    #plt.savefig(f"/home/luther/Documents/plot/CR_2A1/eof_anomalies_{key}.png")
    plt.show()


# Hovmoller

In [ ]:
### code stuff for stations
def coord_to_index(lon, lat):
    
    idx_lon=lon+180
    idx_lat=50-lat    
    return (idx_lat, idx_lon)

pacifique_eq={'lat':(-5,5),'lon':(-150,90)}

EL_NINO_area=np.zeros((100,360))
EL_NINO_area[45:55,0:90]=1
EL_NINO_area[45:55,330:]=1

EL_NINO_area[~EL_NINO_area.astype(np.bool_)]=np.nan
imshow_area(EL_NINO_area, colorbar=False, title="Equatorial Pacific Ocean",)#save="/home/luther/Documents/plot/prez_egu2/ENSO_area.png")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

def hovmoller_diagram(time, data, ax, cmap='viridis', log_scale=True, xlabel='longitude', ylabel='Years', title='ENSO', colorbar_label='chl',
                     vmax=None, colorbar=True):
    """
    Create a Hovmöller diagram using imshow.
    
    Parameters:
        data (2D array): 2D array of data values with shape (len(time), len(space)).
        cmap (str): Colormap to use for the plot.
        log_scale (bool): Whether to use a logarithmic scale for the color map.
        xlabel (str): Label for the x-axis.
        ylabel (str): Label for the y-axis.
        title (str): Title of the plot.
        colorbar_label (str): Label for the color bar.
    """
    # Ensure data is a numpy array
    data = np.array(data)

    # Create the Hovmöller diagram using imshow
    if log_scale:
        norm = LogNorm(vmin=data.min(), vmax=data.max())
    else:
        if not(vmax):
            boundary=max([np.abs(data.min()),data.max()])
        else : boundary=vmax
        norm=Normalize(vmin=-boundary, vmax=boundary)
        #fig.colorbar(ScalarMappable(norm,cmap=cmap), ax=ax,orientation='horizontal', shrink=0.7,label=colorbar_label)

              
    ax.imshow(data, aspect='auto', cmap=cmap, norm=norm, extent=[0, 120, time.max(), time.min()])
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    x_ticks = [0, 30, 80, 120]
    x_labels = ['180°W', '130°W','90°W','120W']
    ax.set_xticks(x_ticks, x_labels)
        

In [ ]:
#chl=np.load("/home/luther/Documents/npy_data/chl/chl_avw_lat50_100km_8d_1997_2023.npy")
#chl=np.load("/home/luther/Documents/npy_data/chl/chl_occi_lat50_100km_8d_1997_2023.npy")
matplotlib.rcParams.update({'font.size': 12})

#true process
chl_el_nino_true=np.nanmean(np.squeeze(EL_NINO_area*chl_anom),axis=1)
chl_el_nino_true=np.concatenate([chl_el_nino_true[:,330:],chl_el_nino_true[:,0:90],],axis=1)


#Nino index
enso34=enso34[5:]


#pred process
chl_el_nino_pred={}

for key in chl_pred_glob.keys():
        
    chl_el_nino_pred[key]=np.nanmean(np.squeeze(EL_NINO_area*chl_pred_glob_anom[key]),axis=1)
    chl_el_nino_pred[key]=np.concatenate([chl_el_nino_pred[key][:,330:],chl_el_nino_pred[key][:,0:90]],axis=1)



#plot
for key in chl_pred_glob.keys():

    fig, axs = plt.subplots(1, 3, figsize=(12, 5),sharey=True)
    time = np.linspace(1998, 2023, 312)
    
    hovmoller_diagram(time, chl_el_nino_pred[key][60:], ax=axs[0],
                      log_scale=False,cmap=mycmp,vmax=0.2, 
                      title='Reconstructed CHL ' +key, colorbar_label='monthly anomalies')
    hovmoller_diagram(time, chl_el_nino_true, 
                      ax=axs[1],log_scale=False, cmap=mycmp,vmax=0.2,
                      title='CHL AVW', colorbar_label='monthly anomalies')
    

    #axs[2].set_yticks([])
    axs[2].invert_yaxis()
    pos=np.where(enso34.flatten()>0, enso34.flatten(),0)
    neg=np.where(enso34.flatten()<0, enso34.flatten(),0)
    axs[2].plot(pos,time,c='red')
    axs[2].plot(neg,time,c='blue')

    axs[2].plot(np.zeros_like(enso34.flatten()), time)
    periods = [(1998, 2011, 'lightyellow'), (2011, 2023, 'lightblue')]
    proxies=[Patch(color='lightblue', alpha=0.3, label='test'),
             Patch(color='lightyellow', alpha=0.3, label='training and validation')]   

# Add shaded regions for different periods
    for start, end, color in periods:
        axs[2].axhspan(start, end, facecolor=color, alpha=0.5)

# Set title and labels
    axs[2].set_title(f'ENSO index (ONI)')

# Add legend
    #axs[2].legend(handles=axs[2].get_lines() + proxies, title='Legend')
# Adjust layout and add colorbars
plt.tight_layout()

plt.show()

# Hovmoller + EOF 
<br/> 


In [ ]:
#chl=np.load("/home/luther/Documents/npy_data/chl/chl_avw_lat50_100km_8d_1997_2023.npy")
#chl=np.load("/home/luther/Documents/npy_data/chl/chl_occi_lat50_100km_8d_1997_2023.npy")
matplotlib.rcParams.update({'font.size': 12})

key1='SmaAt_chlm_avw_variables_sst_ssh_solar'
key2='SmaAt_chlm_avw_variables_sst'


#plot
fig, axs = plt.subplots(1, 4, figsize=(12, 10),sharey=True)
time = np.linspace(1998, 2023, 312)

#hovmoller
hovmoller_diagram(time, chl_el_nino_pred[key1][60:], ax=axs[0],
                  log_scale=False,cmap=mycmp,vmax=0.2, 
                  title='Reconstructed CHL sst_ssh_solar', colorbar_label='monthly anomalies')
hovmoller_diagram(time, chl_el_nino_pred[key2][60:], ax=axs[1],
                  log_scale=False,cmap=mycmp,vmax=0.2, 
                  title='Reconstructed CHL sst', colorbar_label='monthly anomalies', ylabel=None)
hovmoller_diagram(time, chl_el_nino_true, 
                  ax=axs[2],log_scale=False, cmap=mycmp,vmax=0.2,
                  title='CHL AVW', colorbar_label='monthly anomalies',ylabel=None)


#EOF
pcs1=reconstructed_interannual_pcs[key1]
pcs2=reconstructed_interannual_pcs[key2]

#corr=interannual_correlations[key][0]
#idx_color=interannual_correlations[key][1]

axs[3].invert_yaxis()

#ax_eof1=axs[3]
year = np.linspace(1998, 2023, 312)
train_mask=np.zeros((156))*np.nan
    
# Plot primary curves
axs[3].plot(np.concatenate([train_mask,interannual_pcs[:, 0]]), time, color="black", label="avw", linewidth=1.5, alpha=0.5)

axs[3].plot(np.concatenate([train_mask,pcs1[:, 0]]),time, color='orange', label='CHL sst_ssh_solar', linewidth=1.5)
axs[3].plot(np.concatenate([train_mask,pcs2[:, 0]]), time, color='green', label='CHL sst', linewidth=1.5)

#ax_eof1.text(0.25, 0.97, f"Pearson corr : {np.round(corr, 2)}\n", transform=ax1.transAxes,
#       ha='left', va='top', color='black')

# Configure primary y-axis
axs[3].set_xlabel("PC Amplitude")
axs[3].axvline(0, color="black", linewidth=0.8, linestyle="--")
#ax1.set_xlim([2012, 2015])
axs[3].set_xlim([-100, 100])
#axs[3].set_ylabel("Year")
axs[3].legend(loc="lower left", frameon=False,bbox_transform=axs[3].transAxes)

# Add secondary y-axis for MEI Index
ax_eof2 = axs[3].twiny()
#yearm = np.linspace(2011, 2023, 156)

#ax_eof2.plot(yearm, enso34[-13:].flatten(), color="gray", linestyle="--", label="MEI Index")
#ax_eof2.set_ylabel("MEI Index")

# Add secondary y-axis legend
#ax_eof2.legend(loc="upper right", frameon=False)

#axs[2].set_yticks([])

pos=np.where(enso34.flatten()>0, enso34.flatten(),0)
neg=np.where(enso34.flatten()<0, enso34.flatten(),0)
ax_eof2.plot(pos,time,c='red', alpha=0.3)
ax_eof2.plot(neg,time,c='blue', alpha=0.3)
ax_eof2.set_xlim([-2.5, 2.5])

#ax_eof2.plot(enso34.flatten(),time,c='blue')

#ax_eof2.plot(year, time)
#periods = [(1998, 2011, 'lightyellow'), (2011, 2023, 'lightblue')]
#proxies=[Patch(color='lightblue', alpha=0.3, label='test'),
#       Patch(color='lightyellow', alpha=0.3, label='training and validation')]   

# Add shaded regions for different periods
#for start, end, color in periods:
#    axs[2].axhspan(start, end, facecolor=color, alpha=0.5)

# Set title and labels
#axs[2].set_title(f'ENSO index (ONI)')

# Add legend
    #axs[2].legend(handles=axs[2].get_lines() + proxies, title='Legend')
# Adjust layout and add colorbars
plt.tight_layout()

plt.show()

In [ ]:
#chl=np.load("/home/luther/Documents/npy_data/chl/chl_avw_lat50_100km_8d_1997_2023.npy")
#chl=np.load("/home/luther/Documents/npy_data/chl/chl_occi_lat50_100km_8d_1997_2023.npy")
matplotlib.rcParams.update({'font.size': 15})

#plot
fig = plt.figure(figsize=(20, 12), constrained_layout=False)#, dpi=1200)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.20, wspace=0.05),  # First column
    fig.add_gridspec(nrows=1, ncols=1, left=0.23, right=0.43, wspace=0.05),  # Second column
    fig.add_gridspec(nrows=1, ncols=1, left=0.46, right=0.69, wspace=0.05),  # Third column
    fig.add_gridspec(nrows=1, ncols=1, left=0.72, right=1.00, wspace=0.05)   # Fourth column
]

axs = [fig.add_subplot(gs[0,0]) for gs in grid_specs]


time = np.linspace(2011, 2023, 156)

#hovmoller
hovmoller_diagram(time, chl_el_nino_pred[key1][60+156:], ax=axs[0],
                  log_scale=False,cmap=mycmp,vmax=0.2, 
                  title='Reconstructed CHL sst_ssh_solar', colorbar_label='monthly anomalies')
hovmoller_diagram(time, chl_el_nino_pred[key2][60+156:], ax=axs[1],
                  log_scale=False,cmap=mycmp,vmax=0.2, 
                  title='Reconstructed CHL sst', colorbar_label='monthly anomalies',ylabel=None)
hovmoller_diagram(time, chl_el_nino_true[156:], 
                  ax=axs[2],log_scale=False, cmap=mycmp,vmax=0.2,
                  title='CHL AVW', colorbar_label='monthly anomalies',ylabel=None)


#EOF
pcs1=reconstructed_interannual_pcs[key1]
pcs2=reconstructed_interannual_pcs[key2]

eofcorr1=interannual_correlations[key1][0]
eofcorr2=interannual_correlations[key2][0]

corrONI=pearsonr(enso34.flatten()[156:],interannual_pcs[:, 0])[0]

onicorr1=pearsonr(pcs1[:,0],enso34.flatten()[156:])[0]
onicorr2=pearsonr(pcs2[:,0],enso34.flatten()[156:])[0]



axs[3].invert_yaxis()


# Plot primary curves
axs[3].plot(interannual_pcs[:, 0], time, color="black", label="AVW", linewidth=1.5)

axs[3].plot(pcs1[:, 0],time, color='orange', label='CHL sst_ssh_solar', linewidth=1.5)
axs[3].plot(pcs2[:, 0], time, color='green', label='CHL sst', linewidth=1.5)


# Configure primary y-axis
axs[3].set_xlabel("PC Amplitude")
axs[3].axvline(0, color="black", linewidth=0.8, linestyle="--")
axs[3].set_xlim([-100, 100])
axs[3].legend(loc="lower right", frameon=False,bbox_transform=axs[3].transAxes)

"""
axs[3].text(0.02, 0.65, 
            f"Pearson corr with EOF:\n SST SSH Solar {np.round(eofcorr1, 2)}\n SST: {np.round(eofcorr2, 2)}", 
            transform=axs[3].transAxes, ha='left', va='top', color='black')

axs[3].text(0.02, 0.55, 
            f"Pearson corr with ONI:\n SST SSH Solar {np.round(onicorr1, 2)}\n SST: {np.round(onicorr2, 2)}\n AVW {np.round(corrONI,2)}", 
            transform=axs[3].transAxes, ha='left', va='top', color='black')
"""

# Add secondary y-axis for MEI Index
ax_eof2 = axs[3].twiny()

pos=np.where(enso34.flatten()>0, enso34.flatten(),0)
neg=np.where(enso34.flatten()<0, enso34.flatten(),0)

ax_eof2.fill_betweenx(time,pos[156:],color='red', alpha=0.3,label='ONI Index', linestyle='--',)
ax_eof2.fill_betweenx(time,neg[156:],color='blue', alpha=0.3,label=' ', linestyle='--',)

ax_eof2.set_xlim([-2.5, 2.5])
ax_eof2.legend(loc="upper right", frameon=False,bbox_transform=ax_eof2.transAxes)

ax_eof2.set_xlabel("ONI Amplitude")


#plt.savefig('/home/luther/Documents/plot/article_1/hovmoler_eof/chl_psc_roy_hov_eof_1.png',bbox_inches = 'tight')
plt.show()

In [ ]:
#chl=np.load("/home/luther/Documents/npy_data/chl/chl_avw_lat50_100km_8d_1997_2023.npy")
#chl=np.load("/home/luther/Documents/npy_data/chl/chl_occi_lat50_100km_8d_1997_2023.npy")
matplotlib.rcParams.update({'font.size': 15})
import cartopy
import cartopy.feature as cfeature
import cartopy.crs as ccrs
proj=ccrs.Robinson(central_longitude=200)

physics_clim=np.concatenate([np.nanmean(physics.reshape(26,12,7,100,360),axis=0)]*26,axis=0)
physics_anom=physics-physics_clim
physics_el_nino=np.nanmean(np.squeeze(EL_NINO_area*physics_anom),axis=2)
physics_el_nino=np.concatenate([physics_el_nino[...,330:],physics_el_nino[...,0:90],],axis=2)

#plot
fig = plt.figure(figsize=(24, 15), constrained_layout=False)#, dpi=1200)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.18,  top=1., bottom=0.5, wspace=0.05),  # First column
    fig.add_gridspec(nrows=1, ncols=1, left=0.19, right=0.36,  top=1., bottom=0.5, wspace=0.05),  # Second column
    fig.add_gridspec(nrows=1, ncols=1, left=0.37, right=0.54,  top=1., bottom=0.5, wspace=0.05),  # Third column
    fig.add_gridspec(nrows=1, ncols=1, left=0.60, right=1.,  top=0.95, bottom=0.45, wspace=0.05),  # Fourth column
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=1.,  top=0.40, bottom=0., wspace=0.05),  # Fifth column
    #fig.add_gridspec(nrows=1, ncols=1, left=0.61, right=1.,  top=0.45, bottom=0., wspace=0.05)   # Sixth column

]

axes=[
    [
        fig.add_subplot(grid_specs[0][0,0]),
        fig.add_subplot(grid_specs[1][0,0]),
        fig.add_subplot(grid_specs[2][0,0]),
        fig.add_subplot(grid_specs[3][0,0]),
    ],
        [fig.add_subplot(grid_specs[4][0,0]),
         ' '] #fig.add_subplot(grid_specs[5][0,0],projection=proj)

]


key1='SmaAt_chlm_avw_variables_all'
key2='SmaAt_chlm_avw_variables_sst'

time = np.linspace(2011, 2023, 156)

#hovmoller chl
hovmoller_diagram(time, chl_el_nino_pred[key1][60+156:], ax=axes[0][0],
                  log_scale=False,cmap=mycmp,vmax=0.2, 
                  title='Reconstructed CHL all', colorbar_label='monthly anomalies')

hovmoller_diagram(time, chl_el_nino_pred[key2][60+156:], ax=axes[0][1],
                  log_scale=False,cmap=mycmp,vmax=0.2, 
                  title='Reconstructed CHL sst', colorbar_label='monthly anomalies',ylabel=None)
axes[0][1].set_yticks([])
x_ticks = [30, 80]
x_labels = ['130°W','90°W']
axes[0][1].set_xticks(x_ticks, x_labels)
        
hovmoller_diagram(time, chl_el_nino_true[156:], 
                  ax=axes[0][2],log_scale=False, cmap=mycmp,vmax=0.2,
                  title='CHL AVW', colorbar_label='monthly anomalies',ylabel=None)
axes[0][2].yaxis.tick_right()
axes[0][2].set_xticks(x_ticks, x_labels)

#EOF
pcs1=reconstructed_interannual_pcs[key1]
pcs2=reconstructed_interannual_pcs[key2]


# Plot primary curves
axe=axes[1][0]
axe.plot(time, interannual_pcs[:, 0], color="black", label="AVW", linewidth=1.5)

axe.plot(time, pcs1[:, 0], color='orange', label='CHL all', linewidth=1.5)
axe.plot(time, pcs2[:, 0], color='green', label='CHL sst', linewidth=1.5)

# Configure primary y-axis
axe.set_ylabel("PC Amplitude")
axe.axhline(0, color="black", linewidth=0.8, linestyle="--")
axe.set_ylim([-100, 100])
axe.legend(loc="upper left", frameon=False)#,bbox_transform=axe.transAxes)
axe.set_xlim([2011, 2023])

# Add secondary y-axis for ONI Index

axe_1 = axe.twinx()

pos=np.where(enso34.flatten()>0, enso34.flatten(),0)
neg=np.where(enso34.flatten()<0, enso34.flatten(),0)

axe_1.fill_between(time,pos[156:],color='red', alpha=0.3,label='PDO Index', linestyle='--',)
axe_1.fill_between(time,neg[156:],color='blue', alpha=0.3,label=' ', linestyle='--',)

axe_1.set_ylim([-2.5, 2.5])
axe_1.legend(loc="upper right", frameon=False,bbox_transform=axe_1.transAxes)

axe_1.set_ylabel("PDO Amplitude")

#Spatial EOF map 

#imshow_area(pca.components_[0].reshape(100,360),cmap='coolwarm',vmin=-0.015,vmax=0.015,fig=fig,ax=axes[1][1])

#corr matrice

enso34_true=np.loadtxt("/home/luther/Documents/npy_data/oni.txt")
#enso34=np.loadtxt("/home/luther/Documents/npy_data/pdo_1993_2024.txt")[:-1]
r1=-pcs1[:,0]
r2=-pcs2[:,0]
rtrue=-interannual_pcs[:, 0]
PDO=enso34.flatten()[156:]
ENSO=enso34_true.flatten()[60+156:]
sst_tropic_anom=np.nanmean(physics_el_nino[156:,1],axis=-1)

# Create a DataFrame
df = pd.DataFrame({'PDO': PDO, 'AVW': rtrue, 'Model All': r1, 
                   'Model SST': r2, 'ENSO':ENSO,'SSTA ENSO':sst_tropic_anom})#, index=dates)

# Compute the correlation matrix using Pearson correlation
corr_matrix = df.corr(method='spearman')
# Generate a mask for the upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool),k=1)
# Plot the heatmap
sns.heatmap(np.abs(corr_matrix), annot=True, ax=axes[0][3],
            cmap='Greens', fmt='.2f', linewidths=0.5, mask=mask)
axes[0][3].set_title('Correlation Matrix of Time Series')
#plt.savefig('/home/luther/Documents/plot/article_1/Pacific_analysis_1.png',bbox_inches = 'tight')
plt.show()


In [ ]:
#chl=np.load("/home/luther/Documents/npy_data/chl/chl_avw_lat50_100km_8d_1997_2023.npy")
#chl=np.load("/home/luther/Documents/npy_data/chl/chl_occi_lat50_100km_8d_1997_2023.npy")
matplotlib.rcParams.update({'font.size': 15})
import cartopy
import cartopy.feature as cfeature
import cartopy.crs as ccrs
proj=ccrs.Robinson(central_longitude=200)


#plot
fig = plt.figure(figsize=(24, 15), constrained_layout=False)#, dpi=1200)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=1.,  top=1., bottom=0.5, wspace=0.05),  # First column
    fig.add_gridspec(nrows=1, ncols=1, left=0., right=0.5,  top=0.5, bottom=0., wspace=0.05),  # Second column
    fig.add_gridspec(nrows=1, ncols=1, left=0.5, right=1.,  top=0.45, bottom=0., wspace=0.05),  # Third column


]

axes=[
        fig.add_subplot(grid_specs[0][0,0]),
        fig.add_subplot(grid_specs[1][0,0], projection=proj),
        fig.add_subplot(grid_specs[2][0,0]),
]


key1='SmaAt_chlm_avw_variables_all'
key2='SmaAt_chlm_avw_variables_sst'

time = np.linspace(2011, 2023, 156)


#EOF
pcs1=reconstructed_interannual_pcs[key1]
pcs2=reconstructed_interannual_pcs[key2]


# Plot primary curves
axe=axes[0]
axe.plot(time, interannual_pcs[:, 0], color="black", label="AVW", linewidth=1.5)

axe.plot(time, pcs1[:, 0], color='orange', label='CHL all', linewidth=1.5)
axe.plot(time, pcs2[:, 0], color='green', label='CHL sst', linewidth=1.5)

# Configure primary y-axis
axe.set_ylabel("PC Amplitude")
axe.axhline(0, color="black", linewidth=0.8, linestyle="--")
axe.set_ylim([-100, 100])
axe.legend(loc="upper left", frameon=False)#,bbox_transform=axe.transAxes)
axe.set_xlim([2011, 2023])

# Add secondary y-axis for ONI Index

axe_1 = axe.twinx()

pos=np.where(enso34.flatten()>0, enso34.flatten(),0)
neg=np.where(enso34.flatten()<0, enso34.flatten(),0)

axe_1.fill_between(time,pos[156:],color='red', alpha=0.3,label='PDO Index', linestyle='--',)
axe_1.fill_between(time,neg[156:],color='blue', alpha=0.3,label=' ', linestyle='--',)

axe_1.set_ylim([-2.5, 2.5])
axe_1.legend(loc="upper right", frameon=False,bbox_transform=axe_1.transAxes)

axe_1.set_ylabel("PDO Amplitude")

#Spatial EOF map 

imshow_area(pca.components_[0].reshape(100,360),cmap=cm.curl,vmin=-0.015,vmax=0.015,fig=fig,ax=axes[1])

#corr matrice

enso34_true=np.loadtxt("/home/luther/Documents/npy_data/oni.txt")
#enso34=np.loadtxt("/home/luther/Documents/npy_data/pdo_1993_2024.txt")[:-1]
r1=-pcs1[:,0]
r2=-pcs2[:,0]
rtrue=-interannual_pcs[:, 0]
PDO=enso34.flatten()[156:]
ENSO=enso34_true.flatten()[60+156:]
#sst_tropic_anom=np.nanmean(physics_el_nino[156:,1],axis=-1)

# Create a DataFrame
df = pd.DataFrame({'PDO': PDO, 'AVW': rtrue, 'Model All': r1, 
                   'Model SST': r2, 'ENSO':ENSO,})#'SSTA ENSO':sst_tropic_anom})#, index=dates)

# Compute the correlation matrix using Pearson correlation
corr_matrix = df.corr(method='spearman')
# Generate a mask for the upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool),k=1)
# Plot the heatmap
sns.heatmap(np.abs(corr_matrix), annot=True, ax=axes[2],
            cmap='Greens', fmt='.2f', linewidths=0.5, mask=mask)
axes[2].set_title('Correlation Matrix of Time Series')
#plt.savefig('/home/luther/Documents/plot/article_1/Pacific_analysis_3.png',bbox_inches = 'tight')
plt.show()


# Correlation matrix

In [ ]:
ONI=enso34.flatten()[156:]
chl_serie=-interannual_pcs[:, 0]
r1=-pcs1[:,0]
r2=-pcs2[:,0]


print(ONI.shape,chl_serie.shape,r1.shape,r2.shape)

# Create a DataFrame
df = pd.DataFrame({'ONI': ONI, 'Chl_EOF': chl_serie, 'Core Variables': r1, 'SST': r2})#, index=dates)

# Compute the correlation matrix using Pearson correlation
corr_matrix = df.corr(method='pearson')
# Generate a mask for the upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool),k=1)
# Plot the heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(np.abs(corr_matrix), annot=True, cmap='Greens', fmt='.2f', linewidths=0.5, mask=mask)
plt.title('Correlation Matrix of Time Series AVW Roy')
#plt.savefig('/home/luther/Documents/plot/article_1/corr_matrix_occci_0.png',bbox_inches = 'tight')
plt.show()




In [ ]:
ONI=enso34.flatten()[156:]
chl_serie=-interannual_pcs[:, 0]
r1=-pcs1[:,0]
r2=-pcs2[:,0]

chl_serie_1=-interannual_pcs[:, 1]
r11=-pcs1[:,1]
r21=-pcs2[:,1]

print(ONI.shape,chl_serie.shape,r1.shape,r2.shape)

# Create a DataFrame
df = pd.DataFrame({'ONI': ONI, 'Chl_EOF': chl_serie, 'Core Variables': r1, 'SST': r2})#, index=dates)
df1 = pd.DataFrame({'ONI': ONI, 'Chl_EOF': chl_serie_1, 'Core Variables': r11, 'SST': r21})#, index=dates)

# Compute the correlation matrix using Pearson correlation
corr_matrix = df.corr(method='pearson')*(1-mask)

# Generate a mask for the upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))


corr_matrix1 = df1.corr(method='pearson')*(mask)

matrix=corr_matrix+corr_matrix1

# Plot the heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(np.abs(matrix), annot=True, cmap='Greens', fmt='.2f', linewidths=0.5, mask=np.diag(np.diag(np.ones_like(matrix,dtype=bool))))
plt.title('Correlation Matrix of Time Series AVW Roy')
#plt.savefig('/home/luther/Documents/plot/article_1/corr_matrix_avw_1.png',bbox_inches = 'tight')
plt.show()




# SNR 1
$\frac{V_{ar}(S)}{V_{ar}(T)}$

In [ ]:
def compute_snr(signal_decomposed):
    """
    inputs : signal_decomposed (needs to have S and T argumen
    """
    signal_power = np.nanvar(signal_decomposed.S,axis=0)
    noise_power = np.nanvar(signal_decomposed.T,axis=0)
    snr = signal_power / noise_power
    return snr
    
chl_decomposed=temporal_decomposition(np.squeeze(chl))

chl_decomposed.S.shape
chl_snr=compute_snr(chl_decomposed)
#chl_pred_snr=compute_snr(chl_pred_decomposed)
#np.save("/home/luther/Documents/npy_data/chl_snr.npy",chl_snr)
path_save="/home/luther/Documents/plot/article_1/SNR/"
imshow_area(chl_snr, cmap='seismic', log=True,title="SNR AVW", vmin=-1, vmax=1)#, save=path_save+'SNR_monthly_AVW')
#imshow_area(chl_pred_snr, cmap='seismic', log=True,title="amplitude saisonnier par rapport à l'interannuel", vmin=-1, vmax=1)

# SNR 2
$\frac{V_{ar}(Climato S)}{V_{ar}(anomalie)}$

In [ ]:
def compute_snr(signal):
    """
    inputs : signal_decomposed (needs to have S and T argumen
    """
    timestep=len(signal)#we suppose that t is the first dim
    shape=signal.shape
    weekly_mean=np.concatenate([np.nanmean(signal.reshape(timestep//12,12,*shape[1:]),axis=0)]*(timestep//12))
    anomalie_power = np.nanstd(signal-weekly_mean,axis=0)
    noise_power = np.nanstd(weekly_mean,axis=0)
    snr =  noise_power /anomalie_power
    return snr

chl_snr=compute_snr(chl)
#chl_pred_snr=compute_snr(chl_pred_decomposed)
#np.save("/home/luther/Documents/npy_data/chl_snr.npy",chl_snr)
path_save="/home/luther/Documents/plot/article_1/SNR/"
import cartopy
import cartopy.feature as cfeature
import cartopy.crs as ccrs
fig = plt.figure(figsize=(7, 7), constrained_layout=False, dpi=300)
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=1.,  top=1., bottom=0.2, wspace=0.05),  # Chl
    fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=1.,  top=0.4, bottom=0.3, wspace=0.05), #colorbar
]

colorbar=fig.add_subplot(grid_specs[-1][0,0])

matplotlib.rcParams.update({'font.size': 7})
proj=ccrs.Robinson(central_longitude=200)
axes=fig.add_subplot(grid_specs[0][0,0],projection=proj) 

#chl_snr=np.where(np.abs(1-chl_snr)<0.3,np.nan,chl_snr)

imshow_area(chl_snr, cmap='coolwarm', log=True,title="Ratio between Chl-a seasonal climatology standard ecart-type \nand \nChl-a anomalies standard ecart-type", vmin=-1, vmax=1,
            fig=fig,ax=axes,#save="/home/luther/Documents/plot/article_1/"+"SNR_monthly_avw_1_4.png",
           colorbar=False)

colorbar.axis('off')
fig.colorbar(
    ScalarMappable(norm=LogNorm(vmin=1e-1, vmax=10), cmap='coolwarm'),
    ax=colorbar,
    orientation='horizontal',
    shrink=0.8,
    fraction=1.,
    extend='both',
    aspect=50,
)

#plt.savefig("/home/luther/Documents/plot/article_1/SNR/snr_5.pdf",format="pdf", bbox_inches="tight")
plt.show()

# SNR 3
$\frac{V_{ar}(CS)}{V_{ar}(T+\Delta S)}$

In [ ]:
from utils.temporal_decomposition_monthly import temporal_decomposition_monthly

def compute_snr(signal_decomposed):
    """
    inputs : signal_decomposed (needs to have S and T argumen
    """
    timestep=len(signal_decomposed.S)#we suppose that t is the first dim
    shape=signal_decomposed.S.shape
    weekly_mean=np.concatenate([np.nanmean(signal_decomposed.S.reshape(timestep//12,12,*shape[1:]),axis=0)]*(timestep//12))
    delta_seasonal = signal_decomposed.S-weekly_mean
    weekly_var = np.nanstd(weekly_mean,axis=0)
    multiannual_power = np.nanstd(signal_decomposed.T+delta_seasonal,axis=0)
    snr = weekly_var / multiannual_power
    return snr

def compute_snr(signal):

    timestep=len(signal)#we suppose that t is the first dim
    shape=signal.shape
    weekly_mean=np.concatenate([np.nanmean(signal.reshape(timestep//12,12,*shape[1:]),axis=0)]*(timestep//12))
    anomalie_power = np.nanstd(signal-weekly_mean,axis=0)
    noise_power = np.nanstd(weekly_mean,axis=0)
    snr =  noise_power /anomalie_power
    return snr

chl_snr=compute_snr(chl1)

# mask equal ratio within a range around 1

seasonal_region=np.where(chl_snr>1.3,1.,np.nan)
anomalie_region=np.where(chl_snr<0.7,1.,np.nan)
neutral_region=np.where(np.abs(1-chl_snr)<0.3,1.,np.nan)

chl_snr=np.where(np.abs(1-chl_snr)<0.3,np.nan,chl_snr)

#plot
path_save="/home/luther/Documents/plot/article_1/SNR/"

fig = plt.figure(figsize=(7, 7), constrained_layout=False, dpi=300)
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=1.,  top=1., bottom=0.2, wspace=0.05),  # Chl
    fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=1.,  top=0.4, bottom=0., wspace=0.05), #colorbar
]

axes=fig.add_subplot(grid_specs[0][0,0],projection=proj) 
colorbar=fig.add_subplot(grid_specs[-1][0,0])

matplotlib.rcParams.update({'font.size': 6})
proj=ccrs.Robinson(central_longitude=200)

imshow_area(chl_snr, cmap='coolwarm', log=True,title="Ratio between Chl-a seasonal climatology amplitude \nand \nanomalies amplitude", vmin=-1, vmax=1,
            fig=fig,ax=axes,#save="/home/luther/Documents/plot/article_1/"+"SNR_monthly_avw_1_4.png",
           colorbar=False)

colorbar.axis('off')
fig.colorbar(
    ScalarMappable(norm=LogNorm(vmin=1e-1, vmax=10), cmap='coolwarm'),
    ax=colorbar,
    orientation='horizontal',
    shrink=0.8,
    fraction=1.,
    extend='both',
    aspect=30,
)
#plt.savefig("/home/luther/Documents/plot/article_1/"+"SNR_monthly_avw_1_5.png", bbox_inches='tight')
plt.show()
#imshow_area(chl_pred_snr, cmap='seismic', log=True,title="amplitude saisonnier par rapport à l'interannuel", vmin=-1, vmax=1)

# PSC Study 

In [ ]:
import numpy as np
from scipy.stats import pearsonr

def pearson_chl_region(pred, test=test_data, regions=regions):
    time, lat, lon = pred.shape
    pearson_vals = np.full((lat, lon), np.nan)

    # Mask of valid region grid points
    region_mask = ~np.isnan(regions)

    for i in range(lat):
        for j in range(lon):
            if not region_mask[i, j]:
                continue

            x = pred[:, i, j]
            y = test[:, i, j]

            # Skip if all values are NaN
            if np.all(np.isnan(x)) or np.all(np.isnan(y)):
                continue

            # Valid (non-NaN) indices for both series
            valid = ~np.isnan(x) & ~np.isnan(y)
            if np.sum(valid) >= 2:
                x_valid = x[valid]
                y_valid = y[valid]

                # Check if either time series is constant
                if np.ptp(x_valid) == 0 or np.ptp(y_valid) == 0:
                    continue

                r, _ = pearsonr(x_valid, y_valid)
                pearson_vals[i, j] = r

    return np.nanmean(pearson_vals)
    
            time, lat, lon = pred.shape
            spearman_vals = np.full((lat, lon), np.nan)
        
            # Mask of valid region grid points
            region_mask = ~np.isnan(regions)
        
            for i in range(lat):
                for j in range(lon):
                    if not region_mask[i, j]:
                        continue
        
                    x = pred[:, i, j]
                    y = test[:, i, j]
        
                    # Skip if all values are NaN
                    if np.all(np.isnan(x)) or np.all(np.isnan(y)):
                        continue
        
                    # Check if pred series is constant (excluding NaNs)
                    x_valid = x[~np.isnan(x)]
                    if x_valid.size < 2 or np.ptp(x_valid) == 0:  # constant or too few values
                        continue
        
                    # Valid (non-NaN) indices for both series
                    valid = ~np.isnan(x) & ~np.isnan(y)
                    if np.sum(valid) >= 2:
                        r, _ = spearmanr(x[valid], y[valid])
                        spearman_vals[i, j] = r

            def RMSE_bio_region(pred):
                #return np.sqrt(np.nanmean(((pred*regions)-(test_data*regions))**2))/(np.nanstd(test_data*regions))
                error = (pred * regions) - (test_data * regions)
                iqr = np.nanpercentile(test_data * regions, 75) - np.nanpercentile(test_data * regions, 25)
                return np.sqrt(np.nanmean(error**2)) / iqr

### PSC table

In [ ]:
bio_regions_for_plot.keys()

In [ ]:
"""
Metrics
"""
from scipy.stats import spearmanr,pearsonr
from tqdm import tqdm 
bio_region_used=bio_regions_for_plot.keys()#['Global','Atlantique SS Subtropic North','Tropical Pacific']

#new_bio_regions['Atlantique SS Subtropic North']=Atlantique_Nord_area



"""
On prépare les dictionnaires Micro, Nano, Pico
"""
Micro_pred_concentrations_test={}
Nano_pred_concentrations_test={}
Pico_pred_concentrations_test={}

Micro_pred_concentrations_anom_test={}
Nano_pred_concentrations_anom_test={}
Pico_pred_concentrations_anom_test={}


for name in tqdm(psc_conc_pred_test.keys()):
    Micro_pred_concentrations_test[name]=psc_conc_pred_test[name][:,0]
    Nano_pred_concentrations_test[name]=psc_conc_pred_test[name][:,1]
    Pico_pred_concentrations_test[name]=psc_conc_pred_test[name][:,2]

    Micro_pred_concentrations_anom_test[name]=psc_conc_anom_pred_test[name][:,0]
    Nano_pred_concentrations_anom_test[name]=psc_conc_anom_pred_test[name][:,1]
    Pico_pred_concentrations_anom_test[name]=psc_conc_anom_pred_test[name][:,2]


"""
On prépare les dictionnaires de métriques
"""
metrics_base=['rmse','corr']
metrics_anom=['rmse','corr','acc']

data_names=['Chl','Micro','Nano','Pico']

metric_dict={key:{'base':{'rmse':{},'corr':{}},'anom':{'rmse':{},'corr':{},'acc':{}}} for key in data_names}
psc_dict={'rmse':{},'ce':{}}
perf_psc_dict={'rmse':{},'ce':{}}
perf_dict={key :{'rmse':{},'corr':{}} for key in data_names}
"""
le zip target prediction, pas la meilleure manière mais ça marche
"""

tests_data=[chl_test,chl_anom_test, #Chl
            psc_conc_test[:,0],psc_conc_anom_test[:,0], #Micro
            psc_conc_test[:,1], psc_conc_anom_test[:,1],#Nano
            psc_conc_test[:,2],psc_conc_anom_test[:,2]]#Pico

predictions_data=[chl_pred_test,chl_pred_anom_test,#Chl
                  Micro_pred_concentrations_test,Micro_pred_concentrations_anom_test,#Micro
                  Nano_pred_concentrations_test,Nano_pred_concentrations_anom_test,#Nano
                  Pico_pred_concentrations_test,Pico_pred_concentrations_anom_test] #Pico


#for name,regions in tqdm(new_bio_regions.items()):
for name in tqdm(bio_region_used) :
    regions=bio_regions_for_plot[name]
    for i,(test_data, pred) in tqdm(enumerate(zip(tests_data,predictions_data))):
                    
        data_name=data_names[i//2]

        def RMSE_bio_region(pred):
            return np.sqrt(np.nanmean(((pred*regions)-(test_data*regions))**2,axis=None))
    
        def spearman_chl_region(pred, test=test_data, regions=regions):
            time, lat, lon = pred.shape
            pearson_vals = np.full((lat, lon), np.nan)
        
            # Mask of valid region grid points
            region_mask = ~np.isnan(regions)
        
            for i in range(lat):
                for j in range(lon):
                    if not region_mask[i, j]:
                        continue
        
                    x = pred[:, i, j]
                    y = test[:, i, j]
        
                    # Skip if all values are NaN
                    if np.all(np.isnan(x)) or np.all(np.isnan(y)):
                        continue
        
                    # Valid (non-NaN) indices for both series
                    valid = ~np.isnan(x) & ~np.isnan(y)
                    if np.sum(valid) >= 2:
                        x_valid = x[valid]
                        y_valid = y[valid]
        
                        # Check if either time series is constant
                        if np.ptp(x_valid) == 0 or np.ptp(y_valid) == 0:
                            continue
        
                        r, _ = pearsonr(x_valid, y_valid)
                        pearson_vals[i, j] = r
            return np.nanmean(pearson_vals)


        
        if i%2==1 : #anom
           
            def sign_accuracy(pred, test=test_data, regions=regions):
        
                # Get sign: np.sign returns -1, 0, or 1
                pred_sign = np.sign(pred)
                test_sign = np.sign(test)
            
                # Equal sign => correct prediction
                correct = (pred_sign == test_sign)
            
                # Handle optional region masking
                if regions is not None:
                    region_mask = np.broadcast_to(regions, correct.shape)
                    correct = np.where(region_mask==1, correct, np.nan)
            
                # Ignore positions with NaNs in either input
                valid = ~np.isnan(pred) & ~np.isnan(test)
                correct = np.where(valid, correct, np.nan)
            
                return np.nanmean(correct)
            

        if i%2==0: #base
            metric_dict[data_name]['base']['rmse'][name]={key: RMSE_bio_region(value) for key, value in pred.items()}
            metric_dict[data_name]['base']['corr'][name]={key: spearman_chl_region(value) for key, value in pred.items()}
                        
        else :
            metric_dict[data_name]['anom']['rmse'][name]={key: RMSE_bio_region(value) for key, value in pred.items()}
            metric_dict[data_name]['anom']['corr'][name]={key: spearman_chl_region(value) for key, value in pred.items()}
            metric_dict[data_name]['anom']['acc'][name]={key: sign_accuracy(value) for key, value in pred.items()}


for name in tqdm(bio_region_used):
    regions=bio_regions_for_plot[name]

    def cross_entropy_psc_region(psc_pred, psc_test=psc_test, regions=regions):

        eps = 1e-12
        psc_pred_clipped = np.clip(psc_pred, eps, 1. - eps)*regions # Clip to avoid log(0)
    
        # Cross-entropy: sum over classes
        ce = -np.nansum(psc_test*regions * np.log(psc_pred_clipped), axis=1)  # shape: (time, lat, lon)
        return np.nanmean(ce)


    def RMSE_bio_region(pred, test_data=psc_test):
        return np.sqrt(np.nanmean(((pred*regions)-(test_data*regions))**2,axis=(0,2,3)))

    psc_dict['ce'][name]={key: cross_entropy_psc_region(value) for key, value in psc_pred_test.items()}
    psc_dict['rmse'][name]={key: RMSE_bio_region(value) for key, value in psc_pred_test.items()}



for name in tqdm(bio_region_used):
    regions=bio_regions_for_plot[name]
    for i,(test_data, pred) in enumerate(zip(tests_data,predictions_data)):
        data_name=data_names[i//2]
        if i%2==0:
        
            def perf_rmse_bio_region(rmse_pred):
                value_A=metric_dict[data_name]['base']['rmse'][name]['seasonal']
                return ((value_A-rmse_pred)/value_A)*100
    
            def perf_corr_bio_region(corr_pred):
                value_A=metric_dict[data_name]['base']['corr'][name]['seasonal']
                return ((value_A-corr_pred)/value_A)*100


            perf_dict[data_name]['rmse'][name]={key: perf_rmse_bio_region(value) for key, value in metric_dict[data_name]['base']['rmse'][name].items()}
            perf_dict[data_name]['corr'][name]={key: perf_corr_bio_region(value) for key, value in metric_dict[data_name]['base']['corr'][name].items()}


for name in tqdm(bio_region_used):
    regions=bio_regions_for_plot[name]
    
    def perf_rmse_bio_region(rmse_pred):
        value_A=psc_dict['rmse'][name]['seasonal']
        return ((value_A-rmse_pred)/value_A)
    
    def perf_ce_bio_region(ce_pred):
        value_A=psc_dict['ce'][name]['seasonal']
        return ((value_A-ce_pred)/value_A)*100

    perf_psc_dict['rmse'][name]={key: perf_rmse_bio_region(value) for key, value in psc_dict['rmse'][name].items()}
    perf_psc_dict['ce'][name]={key: perf_ce_bio_region(value) for key, value in psc_dict['ce'][name].items()}



In [ ]:
perf_psc_dict['rmse']['Global'].keys()

In [ ]:

"""
Dataframe
"""

keys=['SmaAt_chlm_avw_variables_winds',
 'SmaAt_chlm_avw_variables_sss',
 'SmaAt_chlm_avw_variables_all',
 'SmaAt_chlm_avw_variables_ssh',
 'SmaAt_chlm_avw_variables_sst',
 'SmaAt_chlm_avw_variables_solar',
 #'SmaAt_chlm_avw_variables_sst_ssh',
 'SmaAt_chlm_avw_variables_mld',
 'SmaAt_chlm_avw_variables_currents',
 #'SmaAt_chlm_avw_variables_sst_ssh_solar',
 #'SmaAt_chlm_avw_variables_sst_solar',
 #'SmaAt_chlm_avw_variables_sst_ssh_solarA',
 #'SmaAt_chlm_avw_variables_alla',
 #'SmaAt_chlm_avw_variables_sstA',
 #'SmaAt_chlm_avw_variables_allA',
 #'SmaAt_chlm_avw_variables_ssta',
 #'SmaAt_chlm_avw_variables_cosin',
 #'SmaAt_chlm_avw_variables_sst_ssh_solar_spatial',
'seasonal',
'train_seasonal']

rename=['E_Winds','E_SSS','E_All','E_SSH','E_SST','E_Cs','E_MLD','E_Currents','Monthly Climatology','Train Climatology'] #'cosin','lat'

data_rmse_bioregions={}
data_corr_bioregions={}
data_rmse_anom_bioregions={}
data_corr_anom_bioregions={}
data_acc_anom_bioregions={}
data_ce_bioregions={}
data_rmse_psc_bioregions={}
data_perf_ce_bioregions={}
data_perf_rmse_psc_bioregions={}
data_perf_rmse_bioregions={}
data_perf_corr_bioregions={}


df_rmse_bioregions={}
df_corr_bioregions={}
df_rmse_anom_bioregions={}
df_corr_anom_bioregions={}
df_acc_anom_bioregions={}
df_ce_bioregions={}
df_rmse_psc_bioregions={}
df_perf_ce_bioregions={}
df_perf_rmse_psc_bioregions={}
df_perf_rmse_bioregions={}
df_perf_corr_bioregions={}

df_sorted_rmse_bioregions={}
df_sorted_corr_bioregions={}
df_sorted_rmse_anom_bioregions={}
df_sorted_corr_anom_bioregions={}
df_sorted_acc_anom_bioregions={}
df_sorted_ce_bioregions={}
df_sorted_rmse_psc_bioregions={}
df_sorted_perf_ce_bioregions={}
df_sorted_perf_rmse_psc_bioregions={}
df_sorted_perf_rmse_bioregions={}
df_sorted_perf_corr_bioregions={}

for data_name in tqdm(data_names):

    data_rmse_bioregions[data_name] = np.stack([[metric_dict[data_name]['base']['rmse'][key1][key2] for key1 in bio_region_used] for key2 in keys])
    data_corr_bioregions[data_name] = np.stack([[metric_dict[data_name]['base']['corr'][key1][key2] for key1 in bio_region_used] for key2 in keys])

    data_perf_rmse_bioregions[data_name] = np.stack([[perf_dict[data_name]['rmse'][key1][key2] for key1 in bio_region_used] for key2 in keys])
    data_perf_corr_bioregions[data_name] = np.stack([[perf_dict[data_name]['corr'][key1][key2] for key1 in bio_region_used] for key2 in keys])
    
    data_rmse_anom_bioregions[data_name] = np.stack([[metric_dict[data_name]['anom']['rmse'][key1][key2] for key1 in bio_region_used] for key2 in keys])
    data_corr_anom_bioregions[data_name] =np.stack([[metric_dict[data_name]['anom']['corr'][key1][key2] for key1 in bio_region_used] for key2 in keys])
    data_acc_anom_bioregions[data_name] = np.stack([[metric_dict[data_name]['anom']['acc'][key1][key2] for key1 in bio_region_used] for key2 in keys])
    

    # Create DataFrame
    df_rmse_bioregions[data_name] = pd.DataFrame(data_rmse_bioregions[data_name], index=keys, columns=bio_region_used)
    df_corr_bioregions[data_name] = pd.DataFrame(data_corr_bioregions[data_name], index=keys, columns=bio_region_used)

    df_perf_rmse_bioregions[data_name] = pd.DataFrame(data_perf_rmse_bioregions[data_name], index=keys, columns=bio_region_used)
    df_perf_corr_bioregions[data_name] = pd.DataFrame(data_perf_corr_bioregions[data_name], index=keys, columns=bio_region_used)
    
    df_rmse_anom_bioregions[data_name] = pd.DataFrame(data_rmse_anom_bioregions[data_name], index=keys, columns=bio_region_used)
    df_corr_anom_bioregions[data_name] = pd.DataFrame(data_corr_anom_bioregions[data_name], index=keys, columns=bio_region_used)
    df_acc_anom_bioregions[data_name] = pd.DataFrame(data_acc_anom_bioregions[data_name], index=keys, columns=bio_region_used)
    
    # rename columns 
    for df in [df_rmse_bioregions[data_name],df_corr_bioregions[data_name],
               df_perf_rmse_bioregions[data_name],df_perf_corr_bioregions[data_name],
               df_rmse_anom_bioregions[data_name],df_corr_anom_bioregions[data_name],df_acc_anom_bioregions[data_name]]:
        df.rename(index={keys[i]:rename[i] for i in range(len(keys))},inplace=True)


    #sort
    if data_name=='Chl': #chl needs to be the first of data_names
        df_sorted_rmse_bioregions[data_name]=df_rmse_bioregions[data_name].sort_values(by='Global', ascending=True)
    else :
        df_sorted_rmse_bioregions[data_name]=df_rmse_bioregions[data_name].loc[df_sorted_rmse_bioregions['Chl'].index]

    df_sorted_corr_bioregions[data_name]= df_corr_bioregions[data_name].loc[df_sorted_rmse_bioregions['Chl'].index]

    df_sorted_perf_rmse_bioregions[data_name]= df_perf_rmse_bioregions[data_name].loc[df_sorted_rmse_bioregions['Chl'].index]
    df_sorted_perf_corr_bioregions[data_name]= df_perf_corr_bioregions[data_name].loc[df_sorted_rmse_bioregions['Chl'].index]
    
    df_sorted_rmse_anom_bioregions[data_name] = df_rmse_anom_bioregions[data_name].loc[df_sorted_rmse_bioregions['Chl'].index]
    df_sorted_corr_anom_bioregions[data_name] = df_corr_anom_bioregions[data_name].loc[df_sorted_rmse_bioregions['Chl'].index]
    df_sorted_acc_anom_bioregions[data_name] = df_acc_anom_bioregions[data_name].loc[df_sorted_rmse_bioregions['Chl'].index]


data_ce_bioregions=np.stack([[psc_dict['ce'][key1][key2] for key1 in bio_region_used] for key2 in keys])
data_rmse_psc_bioregions=np.stack([[psc_dict['rmse'][key1][key2] for key1 in bio_region_used] for key2 in keys])

data_perf_ce_bioregions=np.stack([[perf_psc_dict['ce'][key1][key2] for key1 in bio_region_used] for key2 in keys])
data_perf_rmse_psc_bioregions=np.stack([[perf_psc_dict['rmse'][key1][key2] for key1 in bio_region_used] for key2 in keys])

df_ce_bioregions=pd.DataFrame(data_ce_bioregions, index=keys, columns=bio_region_used)
df_ce_bioregions.rename(index={keys[i]:rename[i] for i in range(len(keys))},inplace=True)
df_sorted_ce_bioregions=df_ce_bioregions.loc[df_sorted_rmse_bioregions['Chl'].index]

df_perf_ce_bioregions=pd.DataFrame(data_perf_ce_bioregions, index=keys, columns=bio_region_used)
df_perf_ce_bioregions.rename(index={keys[i]:rename[i] for i in range(len(keys))},inplace=True)
df_sorted_perf_ce_bioregions=df_perf_ce_bioregions.loc[df_sorted_rmse_bioregions['Chl'].index]


for j,psc_name in enumerate(['Micro','Nano','Pico']):
    df_rmse_psc_bioregions[psc_name]=pd.DataFrame(data_rmse_psc_bioregions[...,j], index=keys, columns=bio_region_used)
    df_rmse_psc_bioregions[psc_name].rename(index={keys[i]:rename[i] for i in range(len(keys))},inplace=True)
    df_sorted_rmse_psc_bioregions[psc_name]=df_rmse_psc_bioregions[psc_name].loc[df_sorted_rmse_bioregions['Chl'].index]

    df_perf_rmse_psc_bioregions[psc_name]=pd.DataFrame(data_perf_rmse_psc_bioregions[...,j], index=keys, columns=bio_region_used)
    df_perf_rmse_psc_bioregions[psc_name].rename(index={keys[i]:rename[i] for i in range(len(keys))},inplace=True)
    df_sorted_perf_rmse_psc_bioregions[psc_name]=df_perf_rmse_psc_bioregions[psc_name].loc[df_sorted_rmse_bioregions['Chl'].index]





In [ ]:
df_sorted_rmse_bioregions['Chl']['Global']

In [ ]:
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'font.size': 5.5})


#plot
fig = plt.figure(figsize=(8, 6), constrained_layout=False, dpi=300)

region='Atlantique SS Subtropic North'
region='Tropical Pacific'
region='Global'


# Define grid specifications and axes
grid_specs = [
    [fig.add_gridspec(nrows=1, ncols=2, left=0.0, right=0.085, wspace=0.05),  # First column
    fig.add_gridspec(nrows=1, ncols=3, left=0.09, right=0.19, wspace=0.05)],  # Second column
    [fig.add_gridspec(nrows=1, ncols=2, left=0.2, right=0.285, wspace=0.05),  # Third column
    fig.add_gridspec(nrows=1, ncols=3, left=0.29, right=0.39, wspace=0.05)],   # Fourth column
    [fig.add_gridspec(nrows=1, ncols=2, left=0.4, right=0.485, wspace=0.05),   # Fifth column
    fig.add_gridspec(nrows=1, ncols=3, left=0.49, right=0.59, wspace=0.05)],   # Sixth column
    [fig.add_gridspec(nrows=1, ncols=2, left=0.6, right=0.685, wspace=0.05),   # Seventh column
    fig.add_gridspec(nrows=1, ncols=3, left=0.69, right=0.79, wspace=0.05)],   # Height column
    fig.add_gridspec(nrows=1, ncols=4, left=0.8, right=1., wspace=0.05),
]

axes = [  
    [fig.add_subplot(gs[0][:, 0]), fig.add_subplot(gs[0][:, 1]), 
     fig.add_subplot(gs[1][:, 0]),fig.add_subplot(gs[1][:, 1]), fig.add_subplot(gs[1][:, 2]) ] for gs in grid_specs[:-1]
]
psc_axe=[fig.add_subplot(grid_specs[-1][:,i]) for i in range(4)]

# Add ghost axes and titles for gs
ghost_axes_grid_specs = [
    fig.add_gridspec(nrows=1, ncols=2, left=0.0, right=0.2, wspace=0.05),
    fig.add_gridspec(nrows=1, ncols=2, left=0.2, right=0.4, wspace=0.05),
    fig.add_gridspec(nrows=1, ncols=2, left=0.4, right=0.6, wspace=0.05),
    fig.add_gridspec(nrows=1, ncols=2, left=0.6, right=0.8, wspace=0.05),
    fig.add_gridspec(nrows=1, ncols=1, left=0.8, right=1., wspace=0.05),
]

ghost_axes=[
    [fig.add_subplot(gs[:,0]), fig.add_subplot(gs[:,1])] for gs in ghost_axes_grid_specs[:-1]
]
ghost_axes_psc=fig.add_subplot(ghost_axes_grid_specs[-1][:])

# Define color maps
color_maps = ["coolwarm", "coolwarm_r", "coolwarm", "coolwarm_r", "coolwarm_r"]
metric_names=['rmse', 'corr', 'rmse', 'corr', 'acc']
title_top1=['Chl-a \nvs \nClimatology','Micro \nvs \nClimatology','Nano \nvs \nClimatology','Pico \nvs \nClimatology']
title_top2=['Chl-a \nanomalies\n','Micro \nanomalies\n','Nano \nanomalies\n','Pico \nanomalies\n']


for i,data_name in enumerate(data_names):
    
    for j,dataframe in enumerate([-df_sorted_perf_rmse_bioregions[data_name],-df_sorted_perf_corr_bioregions[data_name],
                              df_sorted_rmse_anom_bioregions[data_name], df_sorted_corr_anom_bioregions[data_name],
                                  df_sorted_acc_anom_bioregions[data_name]]):
        color_map=color_maps[j]
        unique_sorted = np.sort(np.unique(dataframe[region]))
        vmin = unique_sorted[0]
        vmax =  unique_sorted[-1]
        midpoint = dataframe.loc['Monthly Climatology'][region]
    
        data=dataframe[[region]].drop(labels='Monthly Climatology')

        if j<2:
            #data.loc['Monthly Climatology']=np.nan
            norm = mcolors.TwoSlopeNorm(vcenter=0)
            labels = (np.asarray(["{0:.1f}%".format(value)
                    for value in data.values.flatten()])).reshape(*data.values.shape)
            fmt=""

        elif j==2:
            midpoint = dataframe.loc['Monthly Climatology'][region]
            norm = mcolors.TwoSlopeNorm(vcenter=np.round(midpoint,3),
                                        vmin=np.round(midpoint,3)*0.5,
                                        vmax=np.round(midpoint,3)*1.5)
            labels=True
            fmt=".3f"
            """
            labels = (np.asarray(["{0:.1f}%".format(100*value)
                    for value in data.values.flatten()])).reshape(*data.values.shape)
            fmt=""
            """
            
            ghost_axes[i][1].axis('off')
            ghost_axes[i][1].set_title(title_top2[i])
        
        elif j==3:
            
            midpoint = dataframe.loc['Monthly Climatology'][region]
            norm = mcolors.TwoSlopeNorm(vmin=0, vcenter=0.3, vmax=1)
            labels=True
            fmt=".3f"
            
        else:
            labels=True
            fmt=".3f"            
            vmin = unique_sorted[1]
            norm = mcolors.TwoSlopeNorm(vmin=0., vcenter=0.55, vmax=0.7)
        
        # Create heatmap
        sns.heatmap(data, annot=labels, cmap=color_map, fmt=fmt, linewidths=0.5, ax=axes[i][j],  
                    cbar=False, norm=norm, xticklabels=False)


        ghost_axes[i][0].axis('off')
        ghost_axes[i][0].set_title(title_top1[i])

        axes[i][j].set_xlabel(metric_names[j])

        if i == 0 and j==0 :
            axes[i][j].set_ylabel("Experiment")
            axes[i][j].set_yticklabels(dataframe.drop('Monthly Climatology').index, rotation=0)
        else:
            axes[i][j].get_yaxis().set_ticks([])

#plot psc table

norm = mcolors.TwoSlopeNorm(vcenter=0.5,vmin=0,vmax=1)
data=-df_sorted_perf_ce_bioregions[[region]].drop(labels='Monthly Climatology')
data=df_sorted_ce_bioregions[[region]].drop(labels='Monthly Climatology')


labels = (np.asarray(["{0:.3f}".format(value)
        for value in data.values.flatten()])).reshape(*data.values.shape)

sns.heatmap(data, annot=labels, 
            cmap='coolwarm', 
            fmt="", linewidths=0.5, ax=psc_axe[0],  
            cbar=False, norm=norm, xticklabels=False)
psc_axe[0].get_yaxis().set_ticks([])
psc_axe[0].set_xlabel('ce')
norm = mcolors.TwoSlopeNorm(vcenter=0,vmin=-50,vmax=50)

for j,psc_name in enumerate(['Micro','Nano','Pico']):

    data=-100*df_sorted_perf_rmse_psc_bioregions[psc_name][[region]].drop(labels='Monthly Climatology')
    data=100*df_sorted_rmse_psc_bioregions[psc_name][[region]].drop(labels='Monthly Climatology')
    labels = (np.asarray(["{0:.1f}%".format(value)
            for value in data.values.flatten()])).reshape(*data.values.shape)

    #sns.heatmap(-100*df_sorted_perf_rmse_psc_bioregions[psc_name][[region]].drop(labels='Monthly Climatology'), 
    #            annot=labels, cmap='coolwarm', 
    #            fmt="", linewidths=0.5, ax=psc_axe[j+1],  
    #            cbar=False, norm=norm, xticklabels=False)
    sns.heatmap(100*df_sorted_rmse_psc_bioregions[psc_name][[region]].drop(labels='Monthly Climatology'), 
                annot=labels, cmap='coolwarm', 
                fmt="", linewidths=0.5, ax=psc_axe[j+1],  
                cbar=False, norm=norm, xticklabels=False)
    psc_axe[j+1].get_yaxis().set_ticks([])
    
    psc_axe[j+1].set_xlabel('rmse\n'+psc_name+'%')


ghost_axes_psc.axis('off')
ghost_axes_psc.set_title('PSC%\n')
#plt.savefig(f"/home/luther/Documents/plot/article_1/heatmaps/{region}_4.png", bbox_inches='tight' )
plt.show()

In [ ]:
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'font.size': 7})


#plot
fig = plt.figure(figsize=(8, 6), constrained_layout=False, dpi=300)

region='Atlantique SS Subtropic North'
#region='Tropical Pacific'
region='Global'


# Define grid specifications and axes
grid_specs = [
    [fig.add_gridspec(nrows=1, ncols=2, left=0.0, right=0.095, wspace=0.05),  # First column
    fig.add_gridspec(nrows=1, ncols=3, left=0.1, right=0.24, wspace=0.05)],  # Second column
    [fig.add_gridspec(nrows=1, ncols=2, left=0.25, right=0.345, wspace=0.05),  # Third column
    fig.add_gridspec(nrows=1, ncols=3, left=0.35, right=0.49, wspace=0.05)],   # Fourth column
    [fig.add_gridspec(nrows=1, ncols=2, left=0.5, right=0.595, wspace=0.05),   # Fifth column
    fig.add_gridspec(nrows=1, ncols=3, left=0.6, right=0.74, wspace=0.05)],   # Sixth column
    [fig.add_gridspec(nrows=1, ncols=2, left=0.75, right=0.845, wspace=0.05),   # Seventh column
    fig.add_gridspec(nrows=1, ncols=3, left=0.85, right=0.99, wspace=0.05)],   # Height column
]

axes = [  
    [fig.add_subplot(gs[0][:, 0]), fig.add_subplot(gs[0][:, 1]), 
     fig.add_subplot(gs[1][:, 0]),fig.add_subplot(gs[1][:, 1]), fig.add_subplot(gs[1][:, 2]) ] for gs in grid_specs
]

# Add ghost axes and titles for gs
ghost_axes_grid_specs = [
    fig.add_gridspec(nrows=1, ncols=2, left=0.0, right=0.23, wspace=0.05),
    fig.add_gridspec(nrows=1, ncols=2, left=0.25, right=0.48, wspace=0.05),
    fig.add_gridspec(nrows=1, ncols=2, left=0.5, right=0.72, wspace=0.05),
    fig.add_gridspec(nrows=1, ncols=2, left=0.75, right=0.98, wspace=0.05),
]

ghost_axes=[
    [fig.add_subplot(gs[:,0]), fig.add_subplot(gs[:,1])] for gs in ghost_axes_grid_specs
]

# Define color maps
cmap_1=plt.cm.Greys
cmap_2=plt.cm.Greys_r
colors_1 = cmap_1(np.linspace(0, 1, 256))# Create a new colormap with alpha
colors_2 = cmap_2(np.linspace(0, 1, 256))
colors_1[:, -1],colors_2[:, -1] = 0.3,0.3  # Set alpha to 0.5
from matplotlib.colors import ListedColormap# Create new colormap with modified alpha
new_cmap = ListedColormap(colors_1)
new_cmap_r=ListedColormap(colors_2)
color_maps = ["coolwarm", "coolwarm_r", "coolwarm", "coolwarm_r", "coolwarm_r"]
color_maps = ["coolwarm", "coolwarm_r", new_cmap, new_cmap_r, new_cmap_r]

metric_names=['rmse', 'corr', 'rmse', 'corr', 'acc']
title_top1=['Chl-a \nvs \nClimatology','Micro \nvs \nClimatology','Nano \nvs \nClimatology','Pico \nvs \nClimatology']
title_top2=['Chl-a \nanomalies\n','Micro \nanomalies\n','Nano \nanomalies\n','Pico \nanomalies\n']


for i,data_name in enumerate(data_names):
    
    for j,dataframe in enumerate([-df_sorted_perf_rmse_bioregions[data_name],-df_sorted_perf_corr_bioregions[data_name],
                              df_sorted_rmse_anom_bioregions[data_name], df_sorted_corr_anom_bioregions[data_name],
                                  df_sorted_acc_anom_bioregions[data_name]]):
        color_map=color_maps[j]
        unique_sorted = np.sort(np.unique(dataframe[region]))
        vmin = unique_sorted[0]
        vmax =  unique_sorted[-1]
        midpoint = dataframe.loc['Monthly Climatology'][region]
    
        data=dataframe[[region]].drop(labels='Monthly Climatology')

        if j<2:
            #data.loc['Monthly Climatology']=np.nan
            norm = mcolors.TwoSlopeNorm(vcenter=0)
            labels = (np.asarray(["{0:.1f}%".format(value)
                    for value in data.values.flatten()])).reshape(*data.values.shape)
            fmt=""
            annot_kws=None

        elif j==2:
            midpoint = dataframe.loc['Monthly Climatology'][region]
            norm = mcolors.TwoSlopeNorm(vcenter=np.round(midpoint,3),
                                        vmin=np.round(midpoint,3)*0.95,
                                        vmax=np.round(midpoint,3)*1.05)
            labels=True
            fmt=".3f"
            """
            labels = (np.asarray(["{0:.1f}%".format(100*value)
                    for value in data.values.flatten()])).reshape(*data.values.shape)
            fmt=""
            """
            annot_kws={'color':'black',}
            ghost_axes[i][1].axis('off')
            ghost_axes[i][1].set_title(title_top2[i])
        
        elif j==3:
            
            midpoint = dataframe.loc['Monthly Climatology'][region]
            norm = mcolors.TwoSlopeNorm(vmin=0, vcenter=0.3, vmax=1)
            labels=True
            fmt=".3f"
            annot_kws={'color':'black',}
        else:
            labels=True
            fmt=".3f"            
            vmin = unique_sorted[1]
            norm = mcolors.TwoSlopeNorm(vmin=0., vcenter=0.55, vmax=0.7)
            annot_kws={'color':'black',}
        # Create heatmap
        sns.heatmap(data, annot=labels, cmap=color_map, fmt=fmt, linewidths=0.1, ax=axes[i][j],  
                    cbar=False, norm=norm, xticklabels=False,annot_kws=annot_kws )#linecolor='black')


        ghost_axes[i][0].axis('off')
        ghost_axes[i][0].set_title(title_top1[i])

        axes[i][j].set_xlabel(metric_names[j])

        if i == 0 and j==0 :
            axes[i][j].set_ylabel("Experiment")
            axes[i][j].set_yticklabels(dataframe.drop('Monthly Climatology').index, rotation=0)
        else:
            axes[i][j].get_yaxis().set_ticks([])


 
#plt.savefig(f"/home/luther/Documents/plot/article_1/heatmaps/{region}_5.png", bbox_inches='tight' )
plt.show()

In [ ]:
print(np.nanmean(chl[-156:]*new_bio_regions['Tropical Pacific']))
print(np.nanmean(psc_conc[-156:]*new_bio_regions['Tropical Pacific'],axis=(0,2,3)))
np.nanmean(psc[-156:]*new_bio_regions['Tropical Pacific'],axis=(0,2,3))

In [ ]:
df_acc_anom_bioregions['Pico']['Tropical Pacific']

In [ ]:
fig,ax=plt.subplots(1,1,figsize=(12,5))
time=np.linspace(2011,2023,156)
ax.set_ylim(0,1)
ax.plot(time,np.nanmean(psc[-156:,0]*new_bio_regions['Tropical Pacific'],axis=(1,2)),label='Micro')
ax.plot(time,np.nanmean(psc[-156:,1]*new_bio_regions['Tropical Pacific'],axis=(1,2)),label='Nano')
ax.plot(time,np.nanmean(psc[-156:,2]*new_bio_regions['Tropical Pacific'],axis=(1,2)),label='Pico')

ax.plot(time,np.nanmean(psc_pred_glob['SmaAt_chlm_avw_variables_all'][-156:,0]*new_bio_regions['Tropical Pacific'],axis=(1,2)),label='Micro pred')
ax.plot(time,np.nanmean(psc_pred_glob['SmaAt_chlm_avw_variables_all'][-156:,1]*new_bio_regions['Tropical Pacific'],axis=(1,2)),label='Nano pred')
ax.plot(time,np.nanmean(psc_pred_glob['SmaAt_chlm_avw_variables_all'][-156:,2]*new_bio_regions['Tropical Pacific'],axis=(1,2)),label='Pico pred')
ax_t=ax.twinx()
ax_t.set_ylim(-3,3)
ax_t.plot(time,ENSO)
ax.legend()
plt.show()

### Hovmoller+PSC

In [ ]:


def hovmoller_diagram_2(time, data, ax, cmap='viridis', log_scale=True, xlabel='Years', ylabel='longitude', title='ENSO', colorbar_label='chl',
                     vmax=None, colorbar=True):

    data = np.array(data)
    if log_scale:
        norm = LogNorm(vmin=data.min(), vmax=data.max())
    else:
        if not(vmax):
            boundary=max([np.abs(data.min()),data.max()])
        else : boundary=vmax
        norm=Normalize(vmin=-boundary, vmax=boundary)

    ax.imshow(np.rollaxis(data,1,0), aspect='auto', cmap=cmap, norm=norm, extent=[time.min(), time.max(),0, 120, ])
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    #fig.colorbar(ScalarMappable(norm,cmap=cmap), ax=ax,orientation='vertical', shrink=0.7,label=colorbar_label)

    y_ticks = [0, 30, 80, 120]
    y_labels = ['180°W', '130°W','90°W','120W']
    ax.set_yticks(y_ticks, y_labels)


matplotlib.rcParams.update({'font.size': 6})

fig = plt.figure(figsize=(7, 9), constrained_layout=False)#, dpi=1200)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,  top=1., bottom=0.82, wspace=0.05),  # First column
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.78, bottom=0.6, wspace=0.05),  # Second column
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.55, bottom=0.37, wspace=0.05),  # Third column
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.32, bottom=0.17, wspace=0.05),  # Fourth column
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.15, bottom=0.0, wspace=0.05),  # Fourth column
    fig.add_gridspec(nrows=1, ncols=1, left=0.9, right=1., top=0.32, bottom =0.0, wspace=0.05)
]

axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs[:-1]] 
ghost_axes = fig.add_subplot(grid_specs[-1][:])
ghost_axes.axis('off')
fig.colorbar(ScalarMappable(Normalize(vmin=-0.2, vmax=0.2),cmap=mycmp), ax=ghost_axes,orientation='vertical', shrink=1.,)

import tol_colors as tc
cset = tc.tol_cset('bright')

#PSC
ax=axes[2]
time=np.linspace(2011,2023,156)
ax.set_ylim(0,85)
ax.plot(time,100*np.nanmean(psc[-156:,0]*EL_NINO_area,axis=(1,2)),c=cset[0],label='Micro')
ax.plot(time,100*np.nanmean(psc[-156:,1]*EL_NINO_area,axis=(1,2)),c=cset[1],label='Nano')
ax.plot(time,100*np.nanmean(psc[-156:,2]*EL_NINO_area,axis=(1,2)),c=cset[2],label='Pico')

ax.plot(time,100*np.nanmean(psc_pred_glob['SmaAt_chlm_avw_variables_sst'][-156:,0]*EL_NINO_area,axis=(1,2)),
        c=cset[0], linestyle="--", label='Micro sst')
ax.plot(time,100*np.nanmean(psc_pred_glob['SmaAt_chlm_avw_variables_sst'][-156:,1]*EL_NINO_area,axis=(1,2)),
        c=cset[1],linestyle="--", label='Nano sst')
ax.plot(time,100*np.nanmean(psc_pred_glob['SmaAt_chlm_avw_variables_sst'][-156:,2]*EL_NINO_area,axis=(1,2)),
        c=cset[2],linestyle="--",label='Pico sst')
ax.legend(loc="upper left")
ax.set_xlim([2011,2023])

ax.set_ylabel("PSC %")
ax.set_title('PSC evolution over the Equatorial Pacific')
ax.set_ylim([5,65])

#Chl
ax=axes[1]
ax.plot(time,np.nanmean(chl_pred_glob['SmaAt_chlm_avw_variables_sst'][-156:]*EL_NINO_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl SST')
#ax.plot(time,np.nanmean(chl_pred_glob['SmaAt_chlm_avw_variables_all'][-156:]*EL_NINO_area,axis=(1,2)), c='black',
#        linestyle='dashed',label='Chl ALL')

ax.plot(time,np.nanmean(chl[-156:]*EL_NINO_area,axis=(1,2)),label='chl',c='black')
ax.set_title('CHL evolution over the Equatorial Pacific')
ax.set_xlim([2011,2023])
ax.set_ylabel("mg/m3")

ax.legend()



#ONI
ax_t=axes[0]

pos=np.where(ENSO>0, ENSO,0)
neg=np.where(ENSO<0, ENSO,0)

ax_t.fill_between(time,pos,color='red', alpha=0.3,label='ENSO Index', linestyle='--',)
ax_t.fill_between(time,neg,color='blue', alpha=0.3,label=' ', linestyle='--',)

ax_t.set_ylim([-3, 3])
ax_t.legend(loc="upper right", frameon=False,bbox_transform=ax_t.transAxes)
ax_t.set_xlim([2011,2023])

ax_t.set_ylabel("ENSO Index")

ax_t.set_title('ENSO Index')




#hovmoller chl
#hovmoller_diagram_2(time, chl_el_nino_pred['SmaAt_chlm_avw_variables_all'][60+156:], ax=axes[3],
#                  log_scale=False,cmap=mycmp,vmax=0.2, 
#                  title='Reconstructed CHL all', colorbar_label='monthly anomalies')

hovmoller_diagram_2(time, chl_el_nino_pred['SmaAt_chlm_avw_variables_sst'][60+156:], ax=axes[3],
                  log_scale=False,cmap=mycmp,vmax=0.2, 
                  title='Reconstructed CHL sst', colorbar_label='monthly anomalies',ylabel=None, xlabel=None)

axes[3].set_xticks([])
y_ticks = [30, 80]
y_labels = ['130°W','90°W']
axes[3].set_yticks(y_ticks, y_labels)

hovmoller_diagram_2(time, chl_el_nino_true[156:], 
                  ax=axes[4],log_scale=False, cmap=mycmp,vmax=0.2,
                  title='CHL AVW', colorbar_label='monthly anomalies',ylabel=None)

#plt.savefig('/home/luther/Documents/plot/article_1/ENSO_hov_psc_3.png',bbox_inches = 'tight')

plt.show()

In [ ]:

EL_NINO_area_2=np.zeros((100,360))
EL_NINO_area_2[45:55,45:90]=1
#EL_NINO_area_2[45:55,330:]=1
EL_NINO_area_2[~EL_NINO_area_2.astype(np.bool_)]=np.nan

imshow_area(EL_NINO_area_2, colorbar=False, title="Equatorial Pacific Ocean",)#save="/home/luther/Documents/plot/prez_egu2/ENSO_area.png")
fig,ax=plt.subplots(1,1,figsize=(20,5))
time=np.linspace(2011,2023,156)
ax.plot(time,np.nanmean(psc[-156:,0]*EL_NINO_area_2,axis=(1,2)),c=cset[0], label='Micro')
ax.plot(time,np.nanmean(psc[-156:,1]*EL_NINO_area_2,axis=(1,2)),c=cset[1], label='Nano')
ax.plot(time,np.nanmean(psc[-156:,2]*EL_NINO_area_2,axis=(1,2)),c=cset[2],label='Pico')

ax.plot(time,np.nanmean(psc_pred_glob['SmaAt_chlm_avw_variables_sst'][-156:,0]*EL_NINO_area_2,axis=(1,2)),c=cset[0], linestyle="--", label='Micro sst')
ax.plot(time,np.nanmean(psc_pred_glob['SmaAt_chlm_avw_variables_sst'][-156:,1]*EL_NINO_area_2,axis=(1,2)),c=cset[1],linestyle="--", label='Nano sst')
ax.plot(time,np.nanmean(psc_pred_glob['SmaAt_chlm_avw_variables_sst'][-156:,2]*EL_NINO_area_2,axis=(1,2)),c=cset[2],linestyle="--",label='Pico sst')
ax_t=ax.twinx()
ax_t.set_ylim(-3,3)
ax.legend()
plt.show()

In [ ]:
fig,ax=plt.subplots(1,1,figsize=(12,5))
time=np.linspace(2011,2023,156)
ax.plot(time,np.nanmean(physics[-156:,2]*new_bio_regions['SO-SA-high biomass'],axis=(1,2)))

ax1=ax.twinx()
ax1.plot(time,np.nanmean(psc_pred_glob['SmaAt_chlm_avw_variables_all'][-156:,0]*new_bio_regions['SO-SA-high biomass'],axis=(1,2)))
ax1.plot(time,np.nanmean(psc_pred_glob['SmaAt_chlm_avw_variables_all'][-156:,1]*new_bio_regions['SO-SA-high biomass'],axis=(1,2)))
ax1.plot(time,np.nanmean(psc_pred_glob['SmaAt_chlm_avw_variables_all'][-156:,2]*new_bio_regions['SO-SA-high biomass'],axis=(1,2)))
plt.show()

In [ ]:
plt.figure(figsize=(12,5))
time=np.linspace(2011,2023,156)
plt.plot(time,np.nanmean(psc[-156:,0]*new_bio_regions['SO-SA-high biomass'],axis=(1,2)))
plt.plot(time,np.nanmean(psc[-156:,1]*new_bio_regions['SO-SA-high biomass'],axis=(1,2)))
plt.plot(time,np.nanmean(psc[-156:,2]*new_bio_regions['SO-SA-high biomass'],axis=(1,2)))

plt.plot(time,np.nanmean(psc_pred_glob['SmaAt_chlm_avw_variables_all'][-156:,0]*new_bio_regions['SO-SA-high biomass'],axis=(1,2)))
plt.plot(time,np.nanmean(psc_pred_glob['SmaAt_chlm_avw_variables_all'][-156:,1]*new_bio_regions['SO-SA-high biomass'],axis=(1,2)))
plt.plot(time,np.nanmean(psc_pred_glob['SmaAt_chlm_avw_variables_all'][-156:,2]*new_bio_regions['SO-SA-high biomass'],axis=(1,2)))
plt.show()

# Atlantique Nord 

In [ ]:
def get_dynamic_ylim(arrays, margin=0.05):
    data = np.concatenate([a.flatten() for a in arrays])
    data = data[~np.isnan(data)]
    
    ymin, ymax = data.min(), data.max()
    dy = ymax - ymin
    
    return ymin - margin * dy, ymax + margin * dy

### new atlantique region

In [ ]:

for name_br,br in new_bio_regions.items():

    Atlantique_Nord_area=new_bio_regions[name_br]
    imshow_area(Atlantique_Nord_area, colorbar=False, title=name_br)#,save="/home/luther/Documents/plot/prez_egu2/NA_are.png")
    
    matplotlib.rcParams.update({'font.size': 6})

    fig = plt.figure(figsize=(7, 7), constrained_layout=False, dpi=600)

    # Define grid specifications and axes
    grid_specs = [
        fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,  top=1., bottom=0.8, wspace=0.05),  # First row
        fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95, top=0.8, bottom=0.6),# Seasonal Chl
        fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.6, bottom=0.4, wspace=0.05),  # Second row
        fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.4, bottom=0., wspace=0.05),  # Third row
        #fig.add_gridspec(nrows=1, ncols=1, left=0.6, right=0.9, top=0.34, bottom=0.26), #Seasonal PSC
    ]
    
    axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs] 
    
    import tol_colors as tc
    cset = tc.tol_cset('bright')
    
    
    model_key='SmaAt_chlm_avw_variables_all_ssr'
    
    
    # First row
    #Chl
    ax=axes[0]
    
    time=np.linspace(2011,2023,156)
    
    #ax.plot(time,np.nanmean(chl_pred_glob[model_key][-156:]*Atlantique_Nord_area,axis=(1,2)),c='black', 
    #        linestyle='dashed',label='Chl-a E_All')#,linewidth=0.5)
    
   # ax.plot(time,np.nanmean(chl[-156:]*Atlantique_Nord_area,axis=(1,2)),label='Chl-a',c='black')

    chl_pred_series = np.nanmean(chl_pred_glob[model_key][-156:] * Atlantique_Nord_area, axis=(1,2))
    chl_obs_series  = np.nanmean(chl[-156:] * Atlantique_Nord_area, axis=(1,2))
    
    ax.plot(time, chl_pred_series, c='black', linestyle='dashed', label='Chl-a E_All')
    ax.plot(time, chl_obs_series, c='black', label='Chl-a')

    ax.set_ylim(*get_dynamic_ylim([chl_pred_series, chl_obs_series]))
    #ax.set_title('CHL evolution over the Atlantic Subtropics North')
    ax.set_xlim([2011,2023])
    ax.set_ylabel("mg/m3")
    #ax.set_ylim([0.05,0.4])
    
    ax.legend(loc="upper left")
    
    # Second Row
    # Seasonal Row
    
    ax=axes[1]
    time=np.linspace(2011,2023,156)
    #ax.plot(time,np.nanmean((chl_pred_glob[model_key][-156:]-chl_pred_glob_anom[model_key][-156:])*Atlantique_Nord_area,
                            #axis=(1,2)),c='black', linestyle='dashed',label='Seasonal Climatology Chl E_All')
    #ax.plot(time,np.nanmean((chl[-156:]-chl_anom[-156:])*Atlantique_Nord_area,axis=(1,2)),label='Seasonal Climatology Chl-a',c='black')
    
    chl_pred_season = np.nanmean((chl_pred_glob[model_key][-156:] - chl_pred_glob_anom[model_key][-156:]) * Atlantique_Nord_area, axis=(1,2))
    chl_obs_season  = np.nanmean((chl[-156:] - chl_anom[-156:]) * Atlantique_Nord_area, axis=(1,2))
    
    ax.plot(time, chl_pred_season, c='black', linestyle='dashed',label='Seasonal Climatology Chl E_All')
    ax.plot(time, chl_obs_season, c='black',label='Seasonal Climatology Chl-a')
    ax.legend(loc="upper left")

    #ax.set_ylim(*get_dynamic_ylim([chl_pred_season, chl_obs_season]))
    ax.set_ylim(*get_dynamic_ylim([chl_pred_series, chl_obs_series]))

    ax.set_xlim([2011,2023])
    
    ax.set_ylabel("mg/m3")
    #ax.set_ylim([0.05,0.4])
    yticks = ax.get_yticks()         # Get the tick locations
    new_yticks = yticks[:-1]         # Remove the last tick
    ax.set_yticks(new_yticks)           # Set the new y-ticks
    
    #ax.set_title('Seasonal Chl over the Atlantic Subtropics North')



    # Third row
    #Chl anomalies
    ax=axes[2]
    time=np.linspace(2011,2023,156)
    
    
    #ax.plot(time,np.nanmean(chl_pred_glob_anom[model_key][-156:]*Atlantique_Nord_area,axis=(1,2)),c='black', 
     #       linestyle='dashed',label='Chl-a anomalies E_All')
    
   # ax.plot(time,np.nanmean(chl_anom[-156:]*Atlantique_Nord_area,axis=(1,2)),label='Chl-a anomalies',c='black')
    
    chl_pred_anom = np.nanmean(chl_pred_glob_anom[model_key][-156:] * Atlantique_Nord_area, axis=(1,2))
    chl_obs_anom  = np.nanmean(chl_anom[-156:] * Atlantique_Nord_area, axis=(1,2))
    
    ax.plot(time, chl_pred_anom, linestyle='dashed',c='black',label='Chl-a anomalies E_All')
    ax.plot(time, chl_obs_anom,label='Chl-a anomalies',c='black')
    
    ax.set_ylim(*get_dynamic_ylim([chl_pred_anom, chl_obs_anom]))

    ax.set_xlim([2011,2023])
    ax.set_ylabel("mg/m3")
    #ax.set_ylim([-0.15,0.15])
    yticks = ax.get_yticks()         # Get the tick locations
    new_yticks = yticks[:-1]         # Remove the last tick
    ax.set_yticks(new_yticks)           # Set the new y-ticks
    
    ax.legend(loc="upper left")
    
    
    
    #Fourth row
    #PSC %
    ax=axes[3]
    time=np.linspace(2011,2023,156)
    
    ax.plot(time,100*np.nanmean(psc[-156:,0]*Atlantique_Nord_area,axis=(1,2)),c=cset[0],label='Micro')
    ax.plot(time,100*np.nanmean(psc[-156:,1]*Atlantique_Nord_area,axis=(1,2)),c=cset[1],label='Nano')
    ax.plot(time,100*np.nanmean(psc[-156:,2]*Atlantique_Nord_area,axis=(1,2)),c=cset[2],label='Pico')
        
    ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,0]*Atlantique_Nord_area,axis=(1,2)),
            c=cset[0], linestyle="--", label='Micro% E_All')
    ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,1]*Atlantique_Nord_area,axis=(1,2)),
            c=cset[1],linestyle="--", label='Nano% E_All')
    ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,2]*Atlantique_Nord_area,axis=(1,2)),
            c=cset[2],linestyle="--",label='Pico% E_All')


    micro = 100*np.nanmean(psc[-156:,0]*Atlantique_Nord_area, axis=(1,2))
    nano  = 100*np.nanmean(psc[-156:,1]*Atlantique_Nord_area, axis=(1,2))
    pico  = 100*np.nanmean(psc[-156:,2]*Atlantique_Nord_area, axis=(1,2))
    
    micro_p = 100*np.nanmean(psc_pred_glob[model_key][-156:,0]*Atlantique_Nord_area, axis=(1,2))
    nano_p  = 100*np.nanmean(psc_pred_glob[model_key][-156:,1]*Atlantique_Nord_area, axis=(1,2))
    pico_p  = 100*np.nanmean(psc_pred_glob[model_key][-156:,2]*Atlantique_Nord_area, axis=(1,2))
    
    ax.set_ylim(*get_dynamic_ylim([micro, nano, pico, micro_p, nano_p, pico_p]))

    ax.legend(loc="upper left")
    ax.set_xlim([2011,2023])
    
    ax.set_ylabel("%")
    #ax.set_title('PSC evolution over the Atlantic Subtropics North')
    ax.set_ylim([5,65])
    
    labels = ['a', 'b', 'c', 'd']
    
    for ax, lab in zip(axes, labels):
        ax.text(
            0.98, 0.98, lab,
            transform=ax.transAxes,
            fontsize=10,
            fontweight='bold',
            va='top',
            ha='left'
        )


    #plt.savefig(f"/home/luther/Documents/plot/article_1/figures_finales/biome_temp/{name_br}.png",  bbox_inches='tight')
#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/atlantic_subtrop_north_evolution.pdf", format='pdf',  bbox_inches='tight')

    plt.show()

print()

In [ ]:

Atlantique_Nord={'lat':(-5,5),'lon':(-150,90)}

Atlantique_Nord_area=np.zeros((100,360))
Atlantique_Nord_area=np.where(new_bio_regions['Seasonal strat Subtropics North']==1,1,np.nan)
Atlantique_Nord_area[0:60,0:90]=np.nan
Atlantique_Nord_area[0:60,250:]=np.nan

Atlantique_Nord_area[~Atlantique_Nord_area.astype(np.bool_)]=np.nan

Atlantique_Nord_area=new_bio_regions['Seasonal strat Subtropics Atlantic North']
imshow_area(Atlantique_Nord_area, colorbar=False, title="North Atlantique Ocean")#,save="/home/luther/Documents/plot/prez_egu2/NA_are.png")


In [ ]:
matplotlib.rcParams.update({'font.size': 6})

fig = plt.figure(figsize=(7, 7), constrained_layout=False, dpi=1200)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,  top=1., bottom=0.8, wspace=0.05),  # First row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95, top=0.8, bottom=0.6),# Seasonal Chl
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.6, bottom=0.4, wspace=0.05),  # Second row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.4, bottom=0., wspace=0.05),  # Third row
    #fig.add_gridspec(nrows=1, ncols=1, left=0.6, right=0.9, top=0.34, bottom=0.26), #Seasonal PSC
]

axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs] 

import tol_colors as tc
cset = tc.tol_cset('bright')


model_key='SmaAt_chlm_avw_variables_all'


# First row
#Chl
ax=axes[0]

time=np.linspace(2011,2023,156)

ax.plot(time,np.nanmean(chl_pred_glob[model_key][-156:]*Atlantique_Nord_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl-a E_All')#,linewidth=0.5)

ax.plot(time,np.nanmean(chl[-156:]*Atlantique_Nord_area,axis=(1,2)),label='Chl-a',c='black')
#ax.set_title('CHL evolution over the Atlantic Subtropics North')
ax.set_xlim([2011,2023])
ax.set_ylabel("mg/m3")
ax.set_ylim([0.05,0.4])

ax.legend(loc="upper left")

# Second Row
# Seasonal Row

ax=axes[1]
time=np.linspace(2011,2023,156)
ax.plot(time,np.nanmean((chl_pred_glob[model_key][-156:]-chl_pred_glob_anom[model_key][-156:])*Atlantique_Nord_area,
                        axis=(1,2)),c='black', linestyle='dashed',label='Seasonal Climatology Chl E_All')
ax.plot(time,np.nanmean((chl[-156:]-chl_anom[-156:])*Atlantique_Nord_area,axis=(1,2)),label='Seasonal Climatology Chl-a',c='black')
ax.legend(loc="upper left")
ax.set_xlim([2011,2023])

ax.set_ylabel("mg/m3")
ax.set_ylim([0.05,0.4])
yticks = ax.get_yticks()         # Get the tick locations
new_yticks = yticks[:-1]         # Remove the last tick
ax.set_yticks(new_yticks)           # Set the new y-ticks

#ax.set_title('Seasonal Chl over the Atlantic Subtropics North')



# Third row
#Chl anomalies
ax=axes[2]
time=np.linspace(2011,2023,156)


ax.plot(time,np.nanmean(chl_pred_glob_anom[model_key][-156:]*Atlantique_Nord_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl-a anomalies E_All')

ax.plot(time,np.nanmean(chl_anom[-156:]*Atlantique_Nord_area,axis=(1,2)),label='Chl-a anomalies',c='black')

#ax.set_title('CHL anomalie evolution over the Atlantic Subtropics North')
ax.set_xlim([2011,2023])
ax.set_ylabel("mg/m3")
ax.set_ylim([-0.15,0.15])
yticks = ax.get_yticks()         # Get the tick locations
new_yticks = yticks[:-1]         # Remove the last tick
ax.set_yticks(new_yticks)           # Set the new y-ticks

ax.legend(loc="upper left")



#Fourth row
#PSC %
ax=axes[3]
time=np.linspace(2011,2023,156)
ax.plot(time,100*np.nanmean(psc[-156:,0]*Atlantique_Nord_area,axis=(1,2)),c=cset[0],label='Micro')
ax.plot(time,100*np.nanmean(psc[-156:,1]*Atlantique_Nord_area,axis=(1,2)),c=cset[1],label='Nano')
ax.plot(time,100*np.nanmean(psc[-156:,2]*Atlantique_Nord_area,axis=(1,2)),c=cset[2],label='Pico')

ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,0]*Atlantique_Nord_area,axis=(1,2)),
        c=cset[0], linestyle="--", label='Micro% E_All')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,1]*Atlantique_Nord_area,axis=(1,2)),
        c=cset[1],linestyle="--", label='Nano% E_All')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,2]*Atlantique_Nord_area,axis=(1,2)),
        c=cset[2],linestyle="--",label='Pico% E_All')
ax.legend(loc="upper left")
ax.set_xlim([2011,2023])

ax.set_ylabel("%")
#ax.set_title('PSC evolution over the Atlantic Subtropics North')
ax.set_ylim([5,65])

labels = ['a', 'b', 'c', 'd']

for ax, lab in zip(axes, labels):
    ax.text(
        0.98, 0.98, lab,
        transform=ax.transAxes,
        fontsize=10,
        fontweight='bold',
        va='top',
        ha='left'
    )


plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/atlantic_subtrop_north_evolution_1.png",  bbox_inches='tight')
#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/atlantic_subtrop_north_evolution.pdf", format='pdf',  bbox_inches='tight')

plt.show()

print()

In [ ]:
from scipy.stats import linregress
def linear_trend(y):
    t = np.arange(len(y))

    mask = ~np.isnan(y)
    res = linregress(t[mask], y[mask])
    
    return res #res.slope, res.intercept, res.rvalue, res.pvalue, res.stderr

trend_na=linear_trend(np.nanmean(chl_test*Atlantique_Nord_area,axis=(1,2)))
trend_pna=linear_trend(np.nanmean(chl_pred_test["SmaAt_chlm_avw_variables_all"]*Atlantique_Nord_area,axis=(1,2)))
print(trend_na.slope,trend_pna.slope)

In [ ]:
matplotlib.rcParams.update({'font.size': 6})

fig = plt.figure(figsize=(10, 6), constrained_layout=False)#, dpi=1200)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=1., bottom=0.4, wspace=0.05),  # Second row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.4, bottom=0., wspace=0.05),  # Third row
    #fig.add_gridspec(nrows=1, ncols=1, left=0.6, right=0.9, top=0.34, bottom=0.26), #Seasonal PSC
]

axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs] 

import tol_colors as tc
cset = tc.tol_cset('bright')


model_key='SmaAt_chlm_avw_variables_mld'


#First
#PSC %
ax=axes[0]
time=np.linspace(1998,2023,312)
ax.plot(time,100*np.nanmean(psc[:,0]*Atlantique_Nord_area,axis=(1,2)),c=cset[0],label='Micro')
ax.plot(time,100*np.nanmean(psc[:,1]*Atlantique_Nord_area,axis=(1,2)),c=cset[1],label='Nano')
ax.plot(time,100*np.nanmean(psc[:,2]*Atlantique_Nord_area,axis=(1,2)),c=cset[2],label='Pico')

ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-312:,0]*Atlantique_Nord_area,axis=(1,2)),
        c=cset[0], linestyle="--", label='Micro% E_mld')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-312:,1]*Atlantique_Nord_area,axis=(1,2)),
        c=cset[1],linestyle="--", label='Nano% E_mld')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-312:,2]*Atlantique_Nord_area,axis=(1,2)),
        c=cset[2],linestyle="--",label='Pico% E_mld')
#ax.legend(loc="upper left")
#ax.set_xlim([2011,2023])

ax.set_ylabel("PSC %")
#ax.set_title('PSC evolution over the Atlantic Subtropics North')
ax.set_ylim([5,65])

#Second
#MLD
ax=axes[1]
ax.plot(time,np.nanmean(physics[:,0]*Atlantique_Nord_area,axis=(1,2)),c='darkblue',label='MLD (m)')

ax.legend(loc="upper left")
#ax.set_xlim([2011,2023])

ax.set_ylabel("MLD (m)")

from matplotlib.patches import Patch

split_year = 2011
start_year = 1998
end_year = 2023

# Define colors for shading
train_color = 'lightblue'   
test_color = 'lightyellow'    

for ax in axes:
    # Shaded area for training period
    ax.axvspan(start_year, split_year, facecolor=train_color, alpha=0.5, label='Training Period')
    
    # Shaded area for testing period
    ax.axvspan(split_year, end_year, facecolor=test_color, alpha=0.5, label='Testing Period')

# Optional: Add legend for shaded regions
custom_patches = [
    Patch(facecolor=train_color, edgecolor='none', label='Training Period'),
    Patch(facecolor=test_color, edgecolor='none', label='Testing Period')
]

# Add to the first axis to avoid duplication
axes[0].legend( loc='upper left', fontsize=7, frameon=True)

axes[0].set_xlim([1998,2023])
axes[1].set_xlim([1998,2023])
#plt.savefig("/home/luther/Documents/plot/article_1/mld_north_atlantic_0.png",  bbox_inches='tight')
plt.show()

print()

In [ ]:
psc_v1_roy=np.load("/home/luther/Documents/npy_data/PSC/psc_V1_roy.npy")
psc_v1_roy.shape

In [ ]:
fig, ax = plt.subplots(figsize=(14,7), constrained_layout=False)  # Corrected line

# Time axis
time = np.linspace(1998, 2020, 1012)

# Plot each PSC class
ax.plot(time, 100 * np.nanmean(psc_v1_roy[:, 0] * Atlantique_Nord_area, axis=(1, 2)), 
        color=cset[0], label='Micro')
ax.plot(time, 100 * np.nanmean(psc_v1_roy[:, 1] * Atlantique_Nord_area, axis=(1, 2)), 
        color=cset[1], label='Nano')
ax.plot(time, 100 * np.nanmean(psc_v1_roy[:, 2] * Atlantique_Nord_area, axis=(1, 2)), 
        color=cset[2], label='Pico')

# Axis and legend formatting
ax.legend(loc="upper left")
ax.set_xlim([1998, 2020])
ax.set_ylim([5, 65])
ax.set_ylabel("PSC (%)")
# ax.set_title('PSC evolution over the Atlantic Subtropics North')

plt.show()


### Atlnatique Tropical

In [ ]:

Atlantique_Tropical_area=np.zeros((100,360))
Atlantique_Tropical_area=np.where(new_bio_regions['Tropical Atlantique']==1,1,np.nan)


Atlantique_Tropical_area[~Atlantique_Tropical_area.astype(np.bool_)]=np.nan
imshow_area(Atlantique_Tropical_area, colorbar=False, title="North Atlantique Ocean",)#save="/home/luther/Documents/plot/prez_egu2/ENSO_area.png")


In [ ]:
matplotlib.rcParams.update({'font.size': 6})

fig = plt.figure(figsize=(7, 7), constrained_layout=False)#, dpi=1200)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,  top=1., bottom=0.65, wspace=0.05),  # First row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.6, bottom=0.4, wspace=0.05),  # Second row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.35, bottom=0., wspace=0.05),  # Third row
    fig.add_gridspec(nrows=1, ncols=1, left=0.6, right=0.9, top=0.99, bottom=0.91),# Seasonal Chl
    #fig.add_gridspec(nrows=1, ncols=1, left=0.6, right=0.9, top=0.34, bottom=0.26), #Seasonal PSC
]

axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs] 

import tol_colors as tc
cset = tc.tol_cset('bright')


model_key='SmaAt_chlm_avw_variables_sst'


# First row
#Chl
ax=axes[0]

time=np.linspace(2011,2023,156)

ax.plot(time,np.nanmean(chl_pred_glob[model_key][-156:]*Atlantique_Tropical_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl E_SST')#,linewidth=0.5)

ax.plot(time,np.nanmean(chl[-156:]*Atlantique_Tropical_area,axis=(1,2)),label='Chl',c='black')
ax.set_title('CHL evolution over the Tropical Atlantic')
ax.set_xlim([2011,2023])
ax.set_ylabel("mg/m3")
ax.set_ylim([0.1,0.45])

ax.legend(loc="upper left")

# Seasonal plot on first row
ax=axes[3]
time=np.linspace(1,24,24)
ax.plot(time,np.nanmean((chl_pred_glob[model_key][-24:]-chl_pred_glob_anom[model_key][-24:])*Atlantique_Tropical_area,
                        axis=(1,2)),c='black', linestyle='dashed',label='Chl E_SST')
ax.plot(time,np.nanmean(((chl[-24:]-chl_anom[-24:])*Atlantique_Tropical_area),axis=(1,2)),label='Chl',c='black')
#ax.set_xlim([2021,2023])

ax.set_ylabel("Climato Chl mg/m3")
#ax.set_yticklabels([])
ax.set_xticks([1,5,9,12,17,21,23])
ax.set_xticklabels(labels=['Fev','May','Sept','Nov','Apr','Aug','Nov'],
                   rotation=45, ha="right", rotation_mode="anchor")

#ax.legend(loc="upper left")

# Second row
#Chl anomalies
ax=axes[1]
time=np.linspace(2011,2023,156)


ax.plot(time,np.nanmean(chl_pred_glob_anom[model_key][-156:]*Atlantique_Tropical_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl anom E_SST')

#ax.plot(time,np.nanmean(chl_pred_glob['SmaAt_chlm_avw_variables_all'][-156:]*EL_NINO_area,axis=(1,2)), c='black',
#        linestyle='dashed',label='Chl ALL')

ax.plot(time,np.nanmean(chl_anom[-156:]*Atlantique_Tropical_area,axis=(1,2)),label='Chl',c='black')
ax.set_title('CHL anomalie evolution over the Tropical Atlantic')
ax.set_xlim([2011,2023])
ax.set_ylabel("mg/m3")

ax.legend()



#Third row
#PSC NAO %
ax=axes[2]
time=np.linspace(2011,2023,156)
ax.plot(time,100*np.nanmean(psc[-156:,0]*Atlantique_Tropical_area,axis=(1,2)),c=cset[0],label='Micro')
ax.plot(time,100*np.nanmean(psc[-156:,1]*Atlantique_Tropical_area,axis=(1,2)),c=cset[1],label='Nano')
ax.plot(time,100*np.nanmean(psc[-156:,2]*Atlantique_Tropical_area,axis=(1,2)),c=cset[2],label='Pico')

ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,0]*Atlantique_Tropical_area,axis=(1,2)),
        c=cset[0], linestyle="--", label='Micro sst')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,1]*Atlantique_Tropical_area,axis=(1,2)),
        c=cset[1],linestyle="--", label='Nano sst')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,2]*Atlantique_Tropical_area,axis=(1,2)),
        c=cset[2],linestyle="--",label='Pico sst')
ax.legend(loc="upper left")
ax.set_xlim([2011,2023])

ax.set_ylabel("PSC %")
ax.set_title('PSC evolution over the Tropical Atlantic')
ax.set_ylim([5,65])






# Seasonal Row


#plt.savefig("/home/luther/Documents/plot/article_1/Atlantique_Tropical_area_evolution.png",  bbox_inches='tight')
plt.show()



In [ ]:
#PSC Atlantic Tropical 


fig = plt.figure(figsize=(7, 10), constrained_layout=False)#, dpi=1200)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,  top=1., bottom=0.80, wspace=0.05),  # First row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.75, bottom=0.55, wspace=0.05),  # Second row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.5, bottom=0.3, wspace=0.05),  # Third row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.25, bottom=0., wspace=0.05),  # Fourth row

]

axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs] 

model_key='SmaAt_chlm_avw_variables_sst'


#First Row
#Chl 
ax=axes[0]
ax.plot(time,np.nanmean(chl_pred_glob[model_key][-156:]*Atlantique_Tropical_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl E_SST')
ax.plot(time,np.nanmean(chl[-156:]*Atlantique_Tropical_area,axis=(1,2)),label='chl',c='black')
ax.set_title('CHL evolution over the Tropical Atlantic')
ax.set_xlim([2011,2023])
ax.set_ylabel("mg/m3")

ax.legend()



#Second row
#Chl anomalies
ax=axes[1]
ax.plot(time,np.nanmean(chl_pred_glob_anom[model_key][-156:]*Atlantique_Tropical_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl anom E_SST')
ax.plot(time,np.nanmean(chl_anom[-156:]*Atlantique_Tropical_area,axis=(1,2)),label='chl anom',c='black')

ax.set_title('CHL anomalies evolution over the Tropical Atlantic')
ax.set_xlim([2011,2023])
ax.set_ylabel("mg/m3")

ax.legend()

#Third Row
#PSC %
ax=axes[2]
time=np.linspace(2011,2023,156)
ax.plot(time,100*np.nanmean(psc[-156:,0]*Atlantique_Tropical_area,axis=(1,2)),c=cset[0],label='Micro')
ax.plot(time,100*np.nanmean(psc[-156:,1]*Atlantique_Tropical_area,axis=(1,2)),c=cset[1],label='Nano')
ax.plot(time,100*np.nanmean(psc[-156:,2]*Atlantique_Tropical_area,axis=(1,2)),c=cset[2],label='Pico')

ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,0]*Atlantique_Tropical_area,axis=(1,2)),
        c=cset[0], linestyle="--", label='Micro sst')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,1]*Atlantique_Tropical_area,axis=(1,2)),
        c=cset[1],linestyle="--", label='Nano sst')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,2]*Atlantique_Tropical_area,axis=(1,2)),
        c=cset[2],linestyle="--",label='Pico sst')
ax.legend(loc="upper left")
ax.set_xlim([2011,2023])

ax.set_ylabel("PSC %")
ax.set_title('PSC evolution over the Tropical Atlantic')
ax.set_ylim([5,65])

#Fourth Row
#PSC mg/m3
ax=axes[3]
time=np.linspace(2011,2023,156)
ax.plot(time,100*np.nanmean(psc[-156:,0]*Atlantique_Tropical_area*chl[-156:]
                            ,axis=(1,2)),c=cset[0],label='Micro')
ax.plot(time,100*np.nanmean(psc[-156:,1]*Atlantique_Tropical_area*chl[-156:],
                            axis=(1,2)),c=cset[1],label='Nano')
ax.plot(time,100*np.nanmean(psc[-156:,2]*Atlantique_Tropical_area*chl[-156:],
                            axis=(1,2)),c=cset[2],label='Pico')

ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,0]*Atlantique_Tropical_area*chl_pred_glob[model_key][-156:],
                            axis=(1,2)),
        c=cset[0], linestyle="--", label='Micro sst')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,1]*Atlantique_Tropical_area*chl_pred_glob[model_key][-156:],
                            axis=(1,2)),
        c=cset[1],linestyle="--", label='Nano sst')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,2]*Atlantique_Tropical_area*chl_pred_glob[model_key][-156:],
                            axis=(1,2)),
        c=cset[2],linestyle="--",label='Pico sst')
ax.legend(loc="upper left")
ax.set_xlim([2011,2023])

ax.set_ylabel("PSC mg/m3")
ax.set_title('PSC evolution over the Tropical Atlantic')


#plt.savefig("/home/luther/Documents/plot/article_1/atlantic_trop_evolution_0.png")
plt.show()


##################### amplitude des cycles saisonniers début fin

early_seasonal_cycle=np.nanmean(chl[-156:-120].reshape(3,12,100,360)*Atlantique_Tropical_area,axis=0)
late_seasonal_cycle=np.nanmean(chl[-36:].reshape(3,12,100,360)*Atlantique_Tropical_area,axis=0)

early_seasonal_cycle_pred=np.nanmean(chl_pred_glob[model_key][-156:-120].reshape(3,12,100,360)*Atlantique_Tropical_area,axis=0)
late_seasonal_cycle_pred=np.nanmean(chl_pred_glob[model_key][-36:].reshape(3,12,100,360)*Atlantique_Tropical_area,axis=0)

plt.plot(np.nanmean(early_seasonal_cycle,axis=(1,2)),label='early',c=cset[3])
plt.plot(np.nanmean(late_seasonal_cycle,axis=(1,2)),label='late',c=cset[4])

plt.plot(np.nanmean(early_seasonal_cycle_pred,axis=(1,2)),label='early',c=cset[3],linestyle='dashed')
plt.plot(np.nanmean(late_seasonal_cycle_pred,axis=(1,2)),label='late',c=cset[4],linestyle='dashed')
plt.legend()
plt.show()

print(f"amplitude early seasonal cycle :{np.max(np.nanmean(early_seasonal_cycle,axis=(1,2)))-np.min(np.nanmean(early_seasonal_cycle,axis=(1,2)))}")
print(f"amplitude early seasonal cycle pred :{np.max(np.nanmean(early_seasonal_cycle_pred,axis=(1,2)))-np.min(np.nanmean(early_seasonal_cycle_pred,axis=(1,2)))}")
print(f"amplitude late seasonal cycle :{np.max(np.nanmean(late_seasonal_cycle,axis=(1,2)))-np.min(np.nanmean(late_seasonal_cycle,axis=(1,2)))}")
print(f"amplitude late seasonal cycle pred :{np.max(np.nanmean(late_seasonal_cycle_pred,axis=(1,2)))-np.min(np.nanmean(late_seasonal_cycle_pred,axis=(1,2)))}")

In [ ]:
matplotlib.rcParams.update({'font.size': 6})

fig = plt.figure(figsize=(7, 7), constrained_layout=False)#, dpi=1200)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,  top=1., bottom=0.8, wspace=0.05),  # First row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95, top=0.8, bottom=0.6),# Seasonal Chl
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.6, bottom=0.4, wspace=0.05),  # Second row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.4, bottom=0., wspace=0.05),  # Third row
    #fig.add_gridspec(nrows=1, ncols=1, left=0.6, right=0.9, top=0.34, bottom=0.26), #Seasonal PSC
]

axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs] 

import tol_colors as tc
cset = tc.tol_cset('bright')


model_key='SmaAt_chlm_avw_variables_all'


# First row
#Chl
ax=axes[0]

time=np.linspace(2011,2023,156)

ax.plot(time,np.nanmean(chl_pred_glob[model_key][-156:]*Atlantique_Tropical_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl-a E_All')#,linewidth=0.5)

ax.plot(time,np.nanmean(chl[-156:]*Atlantique_Tropical_area,axis=(1,2)),label='Chl-a',c='black')
#ax.set_title('CHL evolution over the Atlantic Subtropics North')
ax.set_xlim([2011,2023])
ax.set_ylabel("mg/m3")
ax.set_ylim([0.05,0.4])

ax.legend(loc="upper left")

# Second Row
# Seasonal Row

ax=axes[1]
time=np.linspace(2011,2023,156)
ax.plot(time,np.nanmean((chl_pred_glob[model_key][-156:]-chl_pred_glob_anom[model_key][-156:])*Atlantique_Tropical_area,
                        axis=(1,2)),c='black', linestyle='dashed',label='Seasonal Climatology Chl E_All')
ax.plot(time,np.nanmean((chl[-156:]-chl_anom[-156:])*Atlantique_Tropical_area,axis=(1,2)),label='Seasonal Climatology Chl-a',c='black')
ax.legend(loc="upper left")
ax.set_xlim([2011,2023])

ax.set_ylabel("mg/m3")
ax.set_ylim([0.05,0.4])
yticks = ax.get_yticks()         # Get the tick locations
new_yticks = yticks[:-1]         # Remove the last tick
ax.set_yticks(new_yticks)           # Set the new y-ticks

#ax.set_title('Seasonal Chl over the Atlantic Subtropics North')



# Third row
#Chl anomalies
ax=axes[2]
time=np.linspace(2011,2023,156)


ax.plot(time,np.nanmean(chl_pred_glob_anom[model_key][-156:]*Atlantique_Tropical_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl-a anomalies E_All')

ax.plot(time,np.nanmean(chl_anom[-156:]*Atlantique_Tropical_area,axis=(1,2)),label='Chl-a anomalies',c='black')

#ax.set_title('CHL anomalie evolution over the Atlantic Subtropics North')
ax.set_xlim([2011,2023])
ax.set_ylabel("mg/m3")
ax.set_ylim([-0.15,0.15])
yticks = ax.get_yticks()         # Get the tick locations
new_yticks = yticks[:-1]         # Remove the last tick
ax.set_yticks(new_yticks)           # Set the new y-ticks

ax.legend(loc="upper left")



#Fourth row
#PSC %
ax=axes[3]
time=np.linspace(2011,2023,156)
ax.plot(time,100*np.nanmean(psc[-156:,0]*Atlantique_Tropical_area,axis=(1,2)),c=cset[0],label='Micro')
ax.plot(time,100*np.nanmean(psc[-156:,1]*Atlantique_Tropical_area,axis=(1,2)),c=cset[1],label='Nano')
ax.plot(time,100*np.nanmean(psc[-156:,2]*Atlantique_Tropical_area,axis=(1,2)),c=cset[2],label='Pico')

ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,0]*Atlantique_Tropical_area,axis=(1,2)),
        c=cset[0], linestyle="--", label='Micro% E_All')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,1]*Atlantique_Tropical_area,axis=(1,2)),
        c=cset[1],linestyle="--", label='Nano% E_All')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,2]*Atlantique_Tropical_area,axis=(1,2)),
        c=cset[2],linestyle="--",label='Pico% E_All')
ax.legend(loc="upper left")
ax.set_xlim([2011,2023])

ax.set_ylabel("%")
#ax.set_title('PSC evolution over the Atlantic Subtropics North')
ax.set_ylim([5,65])


axes[0].set_title('Tropical Atlantic')

#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/biome_temp/atlantic_trop_evolution.png",  bbox_inches='tight')
#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/biome_temp/atlantic_trop_evolution.pdf", format='pdf',  bbox_inches='tight')

plt.show()

print()

### Indian ocean

In [ ]:

Indian_area=np.zeros((100,360))
Indian_area=np.where(new_bio_regions['Indian ocean']==1,1,np.nan)


Indian_area[~Indian_area.astype(np.bool_)]=np.nan
imshow_area(Indian_area, colorbar=False, title="Indian ocean",)#save="/home/luther/Documents/plot/prez_egu2/ENSO_area.png")


In [ ]:
matplotlib.rcParams.update({'font.size': 6})

fig = plt.figure(figsize=(7, 7), constrained_layout=False)#, dpi=1200)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,  top=1., bottom=0.8, wspace=0.05),  # First row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95, top=0.8, bottom=0.6),# Seasonal Chl
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.6, bottom=0.4, wspace=0.05),  # Second row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.4, bottom=0., wspace=0.05),  # Third row
    #fig.add_gridspec(nrows=1, ncols=1, left=0.6, right=0.9, top=0.34, bottom=0.26), #Seasonal PSC
]

axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs] 

import tol_colors as tc
cset = tc.tol_cset('bright')


model_key='SmaAt_chlm_avw_variables_all'


# First row
#Chl
ax=axes[0]

time=np.linspace(2011,2023,156)

ax.plot(time,np.nanmean(chl_pred_glob[model_key][-156:]*Indian_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl-a E_All')#,linewidth=0.5)

ax.plot(time,np.nanmean(chl[-156:]*Indian_area,axis=(1,2)),label='Chl-a',c='black')
#ax.set_title('CHL evolution over the Atlantic Subtropics North')
ax.set_xlim([2011,2023])
ax.set_ylabel("mg/m3")
ax.set_ylim([0.,0.2])

ax.legend(loc="upper left")

# Second Row
# Seasonal Row

ax=axes[1]
time=np.linspace(2011,2023,156)
ax.plot(time,np.nanmean((chl_pred_glob[model_key][-156:]-chl_pred_glob_anom[model_key][-156:])*Indian_area,
                        axis=(1,2)),c='black', linestyle='dashed',label='Seasonal Climatology Chl E_All')
ax.plot(time,np.nanmean((chl[-156:]-chl_anom[-156:])*Indian_area,axis=(1,2)),label='Seasonal Climatology Chl-a',c='black')
ax.legend(loc="upper left")
ax.set_xlim([2011,2023])

ax.set_ylabel("mg/m3")
ax.set_ylim([0.,0.2])
yticks = ax.get_yticks()         # Get the tick locations
new_yticks = yticks[:-1]         # Remove the last tick
ax.set_yticks(new_yticks)           # Set the new y-ticks

#ax.set_title('Seasonal Chl over the Atlantic Subtropics North')



# Third row
#Chl anomalies
ax=axes[2]
time=np.linspace(2011,2023,156)


ax.plot(time,np.nanmean(chl_pred_glob_anom[model_key][-156:]*Indian_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl-a anomalies E_All')

ax.plot(time,np.nanmean(chl_anom[-156:]*Indian_area,axis=(1,2)),label='Chl-a anomalies',c='black')

#ax.set_title('CHL anomalie evolution over the Atlantic Subtropics North')
ax.set_xlim([2011,2023])
ax.set_ylabel("mg/m3")
ax.set_ylim([-0.1,0.1])
yticks = ax.get_yticks()         # Get the tick locations
new_yticks = yticks[:-1]         # Remove the last tick
ax.set_yticks(new_yticks)           # Set the new y-ticks

ax.legend(loc="upper left")



#Fourth row
#PSC %
ax=axes[3]
time=np.linspace(2011,2023,156)
ax.plot(time,100*np.nanmean(psc[-156:,0]*Indian_area,axis=(1,2)),c=cset[0],label='Micro%')
ax.plot(time,100*np.nanmean(psc[-156:,1]*Indian_area,axis=(1,2)),c=cset[1],label='Nano%')
ax.plot(time,100*np.nanmean(psc[-156:,2]*Indian_area,axis=(1,2)),c=cset[2],label='Pico%')

ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,0]*Indian_area,axis=(1,2)),
        c=cset[0], linestyle="--", label='Micro% E_All')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,1]*Indian_area,axis=(1,2)),
        c=cset[1],linestyle="--", label='Nano% E_All')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,2]*Indian_area,axis=(1,2)),
        c=cset[2],linestyle="--",label='Pico% E_All')
ax.legend(loc="upper left")
ax.set_xlim([2011,2023])

ax.set_ylabel("%")
#ax.set_title('PSC evolution over the Atlantic Subtropics North')
ax.set_ylim([5,65])


#axes[0].set_title('SeStrSub-S')

#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/biome_temp/SeStrSub-S_evolution.png",  bbox_inches='tight')
#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/biome_temp/SeStrSub-S_evolution.pdf", format='pdf',  bbox_inches='tight')

plt.show()

print()

In [ ]:
imshow_area(np.nanmean(physics[:,6],axis=0),cmap=cm.curl,vmax=0.010,vmin=-0.010)

In [ ]:
plt.plot(np.nanmean(physics_anom[:,0]*Indian_area,axis=(1,2)))
plt.show()
plt.plot(np.nanmean(physics_anom[:,6]*Indian_area,axis=(1,2)))
plt.show()
plt.plot(np.nanmean(physics[:,6]*Indian_area,axis=(1,2)))
plt.show()

In [ ]:
matplotlib.rcParams.update({'font.size': 6})

fig = plt.figure(figsize=(10, 6), constrained_layout=False)#, dpi=1200)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=1., bottom=0.4, wspace=0.05),  # Second row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.4, bottom=0., wspace=0.05),  # Third row
    #fig.add_gridspec(nrows=1, ncols=1, left=0.6, right=0.9, top=0.34, bottom=0.26), #Seasonal PSC
]

axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs] 

import tol_colors as tc
cset = tc.tol_cset('bright')


model_key='SmaAt_chlm_avw_variables_mld'


#First
#PSC %
ax=axes[0]
time=np.linspace(1998,2023,312)
ax.plot(time,100*np.nanmean(psc[:,0]*Indian_area,axis=(1,2)),c=cset[0],label='Micro')
ax.plot(time,100*np.nanmean(psc[:,1]*Indian_area,axis=(1,2)),c=cset[1],label='Nano')
ax.plot(time,100*np.nanmean(psc[:,2]*Indian_area,axis=(1,2)),c=cset[2],label='Pico')

ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-312:,0]*Indian_area,axis=(1,2)),
        c=cset[0], linestyle="--", label='Micro% E_mld')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-312:,1]*Indian_area,axis=(1,2)),
        c=cset[1],linestyle="--", label='Nano% E_mld')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-312:,2]*Indian_area,axis=(1,2)),
        c=cset[2],linestyle="--",label='Pico% E_mld')
#ax.legend(loc="upper left")
#ax.set_xlim([2011,2023])

ax.set_ylabel("PSC %")
#ax.set_title('PSC evolution over the Atlantic Subtropics North')
ax.set_ylim([5,65])

#Second
#MLD
ax=axes[1]
ax.plot(time,np.nanmean(physics[:,0]*Indian_area,axis=(1,2)),c='darkblue',label='MLD (m)')

ax.legend(loc="upper left")
#ax.set_xlim([2011,2023])

ax.set_ylabel("MLD (m)")

from matplotlib.patches import Patch

split_year = 2011
start_year = 1998
end_year = 2023

# Define colors for shading
train_color = 'lightblue'   
test_color = 'lightyellow'    

for ax in axes:
    # Shaded area for training period
    ax.axvspan(start_year, split_year, facecolor=train_color, alpha=0.5, label='Training Period')
    
    # Shaded area for testing period
    ax.axvspan(split_year, end_year, facecolor=test_color, alpha=0.5, label='Testing Period')

# Optional: Add legend for shaded regions
custom_patches = [
    Patch(facecolor=train_color, edgecolor='none', label='Training Period'),
    Patch(facecolor=test_color, edgecolor='none', label='Testing Period')
]

# Add to the first axis to avoid duplication
axes[0].legend( loc='upper left', fontsize=7, frameon=True)

axes[0].set_xlim([1998,2023])
axes[1].set_xlim([1998,2023])
#plt.savefig("/home/luther/Documents/plot/article_1/mld_north_atlantic_0.png",  bbox_inches='tight')
plt.show()

print()

In [ ]:
import numpy as np

def compute_linear_trend_with_nan(data, time=None):


    T, nx, ny = data.shape
    if time is None:
        time = np.arange(T)
    time = np.asarray(time)

    slopes = np.full((nx, ny), np.nan)
    intercepts = np.full((nx, ny), np.nan)

    # Loop on grid
    for i in range(nx):
        for j in range(ny):
            y = data[:, i, j]

            # mask NaNs
            mask = ~np.isnan(y)
            if np.sum(mask) < 2:
                continue  # not enough points

            t_valid = time[mask]
            y_valid = y[mask]

            # Fit: y = a*t + b
            A = np.vstack([t_valid, np.ones_like(t_valid)]).T
            coef, _, _, _ = np.linalg.lstsq(A, y_valid, rcond=None)

            slopes[i, j] = coef[0]
            intercepts[i, j] = coef[1]

    return slopes, intercepts


matplotlib.rcParams.update({'font.size': 6})

fig = plt.figure(figsize=(10, 6), constrained_layout=False)#, dpi=1200)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=1., bottom=0.4, wspace=0.05),  # Second row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.4, bottom=0., wspace=0.05),  # Third row
    #fig.add_gridspec(nrows=1, ncols=1, left=0.6, right=0.9, top=0.34, bottom=0.26), #Seasonal PSC
]

axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs] 

import tol_colors as tc
cset = tc.tol_cset('bright')


model_key='SmaAt_chlm_avw_variables_mld'


#First
#PSC %
ax=axes[0]
time=np.linspace(1998,2023,312)
trend_micro=compute_linear_trend(psc_conc_pred_glob_anom[model_key][:,0])
print(np.nanmean(trend_micro*Indian_area))
ax.plot(time,np.nanmean(psc_conc_anom[:,0]*Indian_area,axis=(1,2)),c=cset[0],label='Micro')
"""
ax.plot(time,np.nanmean(psc_conc_anom[:,1]*Indian_area,axis=(1,2)),c=cset[1],label='Nano')
ax.plot(time,np.nanmean(psc_conc_anom[:,2]*Indian_area,axis=(1,2)),c=cset[2],label='Pico')
"""
ax.plot(time,np.nanmean(psc_conc_pred_glob_anom[model_key][-312:,0]*Indian_area,axis=(1,2)),
        c=cset[0], linestyle="--", label='Micro% E_mld')
"""
ax.plot(time,np.nanmean(psc_conc_pred_glob_anom[model_key][-312:,1]*Indian_area,axis=(1,2)),
        c=cset[1],linestyle="--", label='Nano% E_mld')
ax.plot(time,np.nanmean(psc_conc_pred_glob_anom[model_key][-312:,2]*Indian_area,axis=(1,2)),
        c=cset[2],linestyle="--",label='Pico% E_mld')
        """
#ax.legend(loc="upper left")
#ax.set_xlim([2011,2023])

ax.set_ylabel("PSC %")
#ax.set_title('PSC evolution over the Atlantic Subtropics North')
#ax.set_ylim([5,65])

#Second
#MLD
ax=axes[1]
ax.plot(time,np.nanmean(physics_anom[:,0]*Indian_area,axis=(1,2)),c='darkblue',label='MLD (m)')

ax.legend(loc="upper left")
#ax.set_xlim([2011,2023])

ax.set_ylabel("MLD (m)")

from matplotlib.patches import Patch

split_year = 2011
start_year = 1998
end_year = 2023

# Define colors for shading
train_color = 'lightblue'   
test_color = 'lightyellow'    

for ax in axes:
    # Shaded area for training period
    ax.axvspan(start_year, split_year, facecolor=train_color, alpha=0.5, label='Training Period')
    
    # Shaded area for testing period
    ax.axvspan(split_year, end_year, facecolor=test_color, alpha=0.5, label='Testing Period')

# Optional: Add legend for shaded regions
custom_patches = [
    Patch(facecolor=train_color, edgecolor='none', label='Training Period'),
    Patch(facecolor=test_color, edgecolor='none', label='Testing Period')
]

# Add to the first axis to avoid duplication
axes[0].legend( loc='upper left', fontsize=7, frameon=True)

axes[0].set_xlim([1998,2023])
axes[1].set_xlim([1998,2023])
#plt.savefig("/home/luther/Documents/plot/article_1/mld_north_atlantic_0.png",  bbox_inches='tight')
plt.show()

print()

### Pacifique trop

In [ ]:

Pacific_Tropical_area=np.zeros((100,360))
Pacific_Tropical_area=np.where(new_bio_regions['Tropical Pacific']==1,1,np.nan)


Pacific_Tropical_area[~Pacific_Tropical_area.astype(np.bool_)]=np.nan
imshow_area(Pacific_Tropical_area, colorbar=False, title="Tropical Pacific Ocean",)#save="/home/luther/Documents/plot/prez_egu2/ENSO_area.png")


In [ ]:
matplotlib.rcParams.update({'font.size': 6})

fig = plt.figure(figsize=(7, 10), constrained_layout=False)#, dpi=1200)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,  top=1., bottom=0.85, wspace=0.05),  # First row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,  top=0.8, bottom=0.5, wspace=0.05),  # First row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.45, bottom=0.25, wspace=0.05),  # Second row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.2, bottom=0., wspace=0.05),  # Third row
    fig.add_gridspec(nrows=1, ncols=1, left=0.6, right=0.9, top=0.79, bottom=0.71),# Seasonal Chl
    #fig.add_gridspec(nrows=1, ncols=1, left=0.6, right=0.9, top=0.34, bottom=0.26), #Seasonal PSC
]

axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs] 

import tol_colors as tc
cset = tc.tol_cset('bright')


model_key='SmaAt_chlm_avw_variables_sst'

#ONI
time=np.linspace(2011,2023,156)

enso34=np.loadtxt("/home/luther/Documents/npy_data/oni.txt")
ENSO=enso34.flatten()[60+156:]
ax_t=axes[0]

pos=np.where(ENSO>0, ENSO,0)
neg=np.where(ENSO<0, ENSO,0)

ax_t.fill_between(time,pos,color='red', alpha=0.3,label='ENSO Index', linestyle='--',)
ax_t.fill_between(time,neg,color='blue', alpha=0.3,label=' ', linestyle='--',)

ax_t.set_ylim([-3, 3])
ax_t.legend(loc="upper right", frameon=False,bbox_transform=ax_t.transAxes)
ax_t.set_xlim([2011,2023])

ax_t.set_ylabel("ENSO Index")

ax_t.set_title('ENSO Index')


# First row
#Chl
ax=axes[1]

time=np.linspace(2011,2023,156)

ax.plot(time,np.nanmean(chl_pred_glob[model_key][-156:]*Pacific_Tropical_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl E_SST')#,linewidth=0.5)

ax.plot(time,np.nanmean(chl[-156:]*Pacific_Tropical_area,axis=(1,2)),label='Chl',c='black')
ax.set_title('CHL evolution over the Tropical Pacific')
ax.set_xlim([2011,2023])
ax.set_ylabel("mg/m3")
ax.set_ylim([0.1,0.25])

ax.legend(loc="upper left")

# Seasonal plot on first row
ax=axes[4]
time=np.linspace(1,24,24)
ax.plot(time,np.nanmean((chl_pred_glob[model_key][-24:]-chl_pred_glob_anom[model_key][-24:])*Pacific_Tropical_area,
                        axis=(1,2)),c='black', linestyle='dashed',label='Chl E_SST')
ax.plot(time,np.nanmean(((chl[-24:]-chl_anom[-24:])*Pacific_Tropical_area),axis=(1,2)),label='Chl',c='black')
#ax.set_xlim([2021,2023])

ax.set_ylabel("Climato Chl mg/m3")
#ax.set_yticklabels([])
ax.set_xticks([1,5,9,12,17,21,23])
ax.set_xticklabels(labels=['Fev','May','Sept','Nov','Apr','Aug','Nov'],
                   rotation=45, ha="right", rotation_mode="anchor")

#ax.legend(loc="upper left")

# Second row
#Chl anomalies
ax=axes[2]
time=np.linspace(2011,2023,156)


ax.plot(time,np.nanmean(chl_pred_glob_anom[model_key][-156:]*Pacific_Tropical_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl anom E_SST')

#ax.plot(time,np.nanmean(chl_pred_glob['SmaAt_chlm_avw_variables_all'][-156:]*EL_NINO_area,axis=(1,2)), c='black',
#        linestyle='dashed',label='Chl ALL')

ax.plot(time,np.nanmean(chl_anom[-156:]*Pacific_Tropical_area,axis=(1,2)),label='Chl',c='black')
ax.set_title('CHL anomalie evolution over the Tropical Pacific')
ax.set_xlim([2011,2023])
ax.set_ylabel("mg/m3")

ax.legend()



#Third row
#PSC NAO %
ax=axes[3]
time=np.linspace(2011,2023,156)
ax.plot(time,100*np.nanmean(psc[-156:,0]*Pacific_Tropical_area,axis=(1,2)),c=cset[0],label='Micro')
ax.plot(time,100*np.nanmean(psc[-156:,1]*Pacific_Tropical_area,axis=(1,2)),c=cset[1],label='Nano')
ax.plot(time,100*np.nanmean(psc[-156:,2]*Pacific_Tropical_area,axis=(1,2)),c=cset[2],label='Pico')

ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,0]*Pacific_Tropical_area,axis=(1,2)),
        c=cset[0], linestyle="--", label='Micro sst')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,1]*Pacific_Tropical_area,axis=(1,2)),
        c=cset[1],linestyle="--", label='Nano sst')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,2]*Pacific_Tropical_area,axis=(1,2)),
        c=cset[2],linestyle="--",label='Pico sst')
ax.legend(loc="upper left")
ax.set_xlim([2011,2023])

ax.set_ylabel("PSC %")
ax.set_title('PSC evolution over the Tropical Pacific')
ax.set_ylim([5,65])



#plt.savefig("/home/luther/Documents/plot/article_1/Pacific_Tropical_area_evolution.png",  bbox_inches='tight')
plt.show()



In [ ]:
matplotlib.rcParams.update({'font.size': 6})

fig = plt.figure(figsize=(7, 7), constrained_layout=False, dpi=600)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,  top=1., bottom=0.825, wspace=0.05),  # First row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,  top=0.825, bottom=0.65, wspace=0.05),  # First row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.65, bottom=0.475, wspace=0.05),  # Second row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.475, bottom=0.3, wspace=0.05),  # Third row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95, top=0.3, bottom=0.),# Seasonal Chl
    #fig.add_gridspec(nrows=1, ncols=1, left=0.6, right=0.9, top=0.34, bottom=0.26), #Seasonal PSC
]


axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs] 

import tol_colors as tc
cset = tc.tol_cset('bright')


model_key='SmaAt_chlm_avw_variables_all'


# First row
#ONI
time=np.linspace(2011,2023,156)

enso34=np.loadtxt("/home/luther/Documents/npy_data/oni.txt")
ENSO=enso34.flatten()[60+156:]
ax_t=axes[0]

pos=np.where(ENSO>0, ENSO,0)
neg=np.where(ENSO<0, ENSO,0)

ax_t.fill_between(time,pos,color='red', alpha=0.3,label='Oceanic Nino Index', linestyle='--',)
ax_t.fill_between(time,neg,color='blue', alpha=0.3,label=' ', linestyle='--',)

ax_t.set_ylim([-3, 3])
ax_t.legend(loc="upper left", frameon=False,bbox_transform=ax_t.transAxes)
ax_t.set_xlim([2011,2023])

ax_t.set_ylabel("ONI Index")

#ax_t.set_title('ONI Index')

#Chl
ax=axes[1]

time=np.linspace(2011,2023,156)

ax.plot(time,np.nanmean(chl_pred_glob[model_key][-156:]*Pacific_Tropical_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl-a E_all')#,linewidth=0.5)

ax.plot(time,np.nanmean(chl[-156:]*Pacific_Tropical_area,axis=(1,2)),label='Chl-a',c='black')
#ax.set_title('CHL evolution over the Atlantic Subtropics North')
ax.set_xlim([2011,2023])
ax.set_ylabel("mg/m3")
ax.set_ylim([0.1,0.2])

ax.legend(loc="upper left")
yticks = ax.get_yticks()
new_yticks = yticks[:-1]           
ax.set_yticks(new_yticks)          

# Second Row
# Seasonal Row

ax=axes[2]
time=np.linspace(2011,2023,156)
ax.plot(time,np.nanmean((chl_pred_glob[model_key][-156:]-chl_pred_glob_anom[model_key][-156:])*Pacific_Tropical_area,
                        axis=(1,2)),c='black', linestyle='dashed',label='Seasonal Climatology Chl-a E_all')
ax.plot(time,np.nanmean((chl[-156:]-chl_anom[-156:])*Pacific_Tropical_area,axis=(1,2)),label='Seasonal Climatology Chl-a',c='black')
ax.legend(loc="upper left")
ax.set_xlim([2011,2023])

ax.set_ylabel("mg/m3")
ax.set_ylim([0.1,0.2])
# Get current y-ticks and remove the last one
yticks = ax.get_yticks()
new_yticks = yticks[:-1]           # Remove last tick
ax.set_yticks(new_yticks)          # Set new ticks
#ax.set_title('Seasonal Chl over the Atlantic Subtropics North')



# Third row
#Chl anomalies
ax=axes[3]
time=np.linspace(2011,2023,156)


ax.plot(time,np.nanmean(chl_pred_glob_anom[model_key][-156:]*Pacific_Tropical_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl-a anomalies E_all')

ax.plot(time,np.nanmean(chl_anom[-156:]*Pacific_Tropical_area,axis=(1,2)),label='Chl-a anomalies',c='black')
#ax.set_title('CHL anomalie evolution over the Atlantic Subtropics North')
ax.set_xlim([2011,2023])
ax.set_ylabel("mg/m3")
ax.set_ylim([-0.05,0.05])

ax.legend(loc="upper left")



#Fourth row
#PSC %
ax=axes[4]
time=np.linspace(2011,2023,156)
ax.plot(time,100*np.nanmean(psc[-156:,0]*Pacific_Tropical_area,axis=(1,2)),c=cset[0],label='Micro%')
ax.plot(time,100*np.nanmean(psc[-156:,1]*Pacific_Tropical_area,axis=(1,2)),c=cset[1],label='Nano%')
ax.plot(time,100*np.nanmean(psc[-156:,2]*Pacific_Tropical_area,axis=(1,2)),c=cset[2],label='Pico%')

ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,0]*Pacific_Tropical_area,axis=(1,2)),
        c=cset[0], linestyle="--", label='Micro% E_all')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,1]*Pacific_Tropical_area,axis=(1,2)),
        c=cset[1],linestyle="--", label='Nano% E_all')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,2]*Pacific_Tropical_area,axis=(1,2)),
        c=cset[2],linestyle="--",label='Pico% E_all')
ax.legend(loc="upper left")
ax.set_xlim([2011,2023])

ax.set_ylabel("%")
#ax.set_title('PSC evolution over the Atlantic Subtropics North')
ax.set_ylim([5,65])



labels = ['a', 'b', 'c', 'd','e']

for ax, lab in zip(axes, labels):
    ax.text(
        0.98, 0.98, lab,
        transform=ax.transAxes,
        fontsize=10,
        fontweight='bold',
        va='top',
        ha='left'
    )
#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/pacific_trop_evolution_1.png",  bbox_inches='tight')
#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/pacific_trop_evolution.pdf",format='pdf',  bbox_inches='tight')

plt.show()

print()

In [ ]:
matplotlib.rcParams.update({'font.size': 8})

fig = plt.figure(figsize=(7, 6), constrained_layout=False, dpi=600)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,  top=1., bottom=0.5, wspace=0.05),  # First row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.5, bottom=0., wspace=0.05),  # Second row
]


axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs] 

import tol_colors as tc
cset = tc.tol_cset('bright')
cset_pale=tc.tol_cset('pale')
cset_dark=tc.tol_cset('dark')

model_key='SmaAt_chlm_avw_variables_solar'
model_key_2='SmaAt_chlm_avw_variables_all_longer_training'
model_key_3='SmaAt_chlm_avw_variables_sst'

#Chl
"""
ax=axes[0]

time=np.linspace(2011,2023,156)

ax.plot(time,np.nanmean(chl_pred_glob[model_key][-156:]*Pacific_Tropical_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl-a E_sst',linewidth=1.)#,linewidth=0.5)


ax.plot(time[-48:],np.nanmean(chl_pred_glob[model_key_2][-48:]*Pacific_Tropical_area,axis=(1,2)),c='red', 
        linestyle='dotted',label='Chl-a E_all_l')#,linewidth=0.5)

ax.plot(time,np.nanmean(chl[-156:]*Pacific_Tropical_area,axis=(1,2)),label='Chl-a',c='black',linewidth=0.5)
#ax.set_title('CHL evolution over the Atlantic Subtropics North')
ax.set_xlim([2011,2023])
ax.set_ylabel("Chl-a mg/m3")
ax.set_ylim([0.1,0.2])

ax.legend(loc="upper left")
yticks = ax.get_yticks()
new_yticks = yticks[:-1]           
ax.set_yticks(new_yticks)          
"""

#Atlantique_Nord_area
ax=axes[0]
time=np.linspace(2011,2023,156)
ax.plot(time,100*np.nanmean(psc[-156:,0]*Atlantique_Nord_area,axis=(1,2)),c=cset[0],label='Micro%',linewidth=0.5)
ax.plot(time,100*np.nanmean(psc[-156:,1]*Atlantique_Nord_area,axis=(1,2)),c=cset[1],label='Nano%',linewidth=0.5)
ax.plot(time,100*np.nanmean(psc[-156:,2]*Atlantique_Nord_area,axis=(1,2)),c=cset[2],label='Pico%',linewidth=0.5)

ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,0]*Atlantique_Nord_area,axis=(1,2)),
        c=cset[0], linestyle="--", label='Micro% E_cs',linewidth=1.)
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,1]*Atlantique_Nord_area,axis=(1,2)),
        c=cset[1],linestyle="--", label='Nano% E_cs',linewidth=1.)
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,2]*Atlantique_Nord_area,axis=(1,2)),
        c=cset[2],linestyle="--",label='Pico% E_cs',linewidth=1.)

ax.plot(time[-48:],100*np.nanmean(psc_pred_glob[model_key_2][-48:,0]*Atlantique_Nord_area,axis=(1,2)),
        c=cset_dark[0], linestyle="dotted", label='Micro% E_all_longer')
ax.plot(time[-48:],100*np.nanmean(psc_pred_glob[model_key_2][-48:,1]*Atlantique_Nord_area,axis=(1,2)),
        c=cset_dark[1],linestyle="dotted", label='Nano% E_all_longer')
ax.plot(time[-48:],100*np.nanmean(psc_pred_glob[model_key_2][-48:,2]*Atlantique_Nord_area,axis=(1,2)),
        c=cset_dark[2],linestyle="dotted",label='Pico% E_all_longer')
ax.legend(loc="upper left")
ax.set_xlim([2011,2023])

ax.set_ylabel("%")
#ax.set_title('PSC evolution over the Atlantic Subtropics North')
ax.set_ylim([5,65])
#Fourth row
#PSC %
ax=axes[1]
time=np.linspace(2011,2023,156)
ax.plot(time,100*np.nanmean(psc[-156:,0]*Pacific_Tropical_area,axis=(1,2)),c=cset[0],label='Micro%',linewidth=0.5)
ax.plot(time,100*np.nanmean(psc[-156:,1]*Pacific_Tropical_area,axis=(1,2)),c=cset[1],label='Nano%',linewidth=0.5)
ax.plot(time,100*np.nanmean(psc[-156:,2]*Pacific_Tropical_area,axis=(1,2)),c=cset[2],label='Pico%',linewidth=0.5)

ax.plot(time,100*np.nanmean(psc_pred_glob[model_key_3][-156:,0]*Pacific_Tropical_area,axis=(1,2)),
        c=cset[0], linestyle="--", label='Micro% E_sst',linewidth=1.)
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key_3][-156:,1]*Pacific_Tropical_area,axis=(1,2)),
        c=cset[1],linestyle="--", label='Nano% E_sst',linewidth=1.)
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key_3][-156:,2]*Pacific_Tropical_area,axis=(1,2)),
        c=cset[2],linestyle="--",label='Pico% E_sst',linewidth=1.)

ax.plot(time[-48:],100*np.nanmean(psc_pred_glob[model_key_2][-48:,0]*Pacific_Tropical_area,axis=(1,2)),
        c=cset_dark[0], linestyle="dotted", label='Micro% E_all_longer')
ax.plot(time[-48:],100*np.nanmean(psc_pred_glob[model_key_2][-48:,1]*Pacific_Tropical_area,axis=(1,2)),
        c=cset_dark[1],linestyle="dotted", label='Nano% E_all_longer')
ax.plot(time[-48:],100*np.nanmean(psc_pred_glob[model_key_2][-48:,2]*Pacific_Tropical_area,axis=(1,2)),
        c=cset_dark[2],linestyle="dotted",label='Pico% E_all_longer')
ax.legend(loc="upper left")
ax.set_xlim([2011,2023])

ax.set_ylabel("%")
#ax.set_title('PSC evolution over the Atlantic Subtropics North')
ax.set_ylim([5,65])



labels = ['a', 'b']

for ax, lab in zip(axes, labels):
    ax.text(
        0.98, 0.98, lab,
        transform=ax.transAxes,
        fontsize=10,
        fontweight='bold',
        va='top',
        ha='left'
    )

#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/discussion_pacific_trop_3.png",  bbox_inches='tight')
#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/discussion_pacific_trop.pdf",format='pdf',  bbox_inches='tight')

plt.show()

print()

In [ ]:
psc_8_days_pred=np.load("/home/luther/Documents/result/variables/SmaAt_variables_all_ocean/psc_pred_glob.npy")
chl_8_days_pred=np.load("/home/luther/Documents/result/variables/SmaAt_variables_all_ocean/chl_pred_glob.npy")
chl_8_days
chl_8_days.shape,psc_8_days.shape

In [ ]:
matplotlib.rcParams.update({'font.size': 6})

fig = plt.figure(figsize=(7, 7), constrained_layout=False)#, dpi=1200)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,  top=1., bottom=0.825, wspace=0.05),  # First row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.8, bottom=0.4, wspace=0.05),  # Third row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95, top=0.4, bottom=0.),# Seasonal Chl
    #fig.add_gridspec(nrows=1, ncols=1, left=0.6, right=0.9, top=0.34, bottom=0.26), #Seasonal PSC
]


axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs] 

import tol_colors as tc
cset = tc.tol_cset('bright')


chl_8_days=np.load("/home/luther/Documents/result/variables/SmaAt_variables_all_ocean/chl_pred_glob.npy")
psc_8_days=np.load("/home/luther/Documents/result/variables/SmaAt_variables_all_ocean/psc_pred_glob.npy")

# First row
#ONI
time=np.linspace(2011,2023,156)

enso34=np.loadtxt("/home/luther/Documents/npy_data/oni.txt")
ENSO=enso34.flatten()[60+156:]
ax_t=axes[0]

pos=np.where(ENSO>0, ENSO,0)
neg=np.where(ENSO<0, ENSO,0)

ax_t.fill_between(time,pos,color='red', alpha=0.3,label='Oceanic Nino Index', linestyle='--',)
ax_t.fill_between(time,neg,color='blue', alpha=0.3,label=' ', linestyle='--',)

ax_t.set_ylim([-3, 3])
ax_t.legend(loc="upper left", frameon=False,bbox_transform=ax_t.transAxes)
ax_t.set_xlim([2011,2023])

ax_t.set_ylabel("ONI Index")

#ax_t.set_title('ONI Index')

#Chl
ax=axes[1]

time=np.linspace(2011,2023,598)

ax.plot(time,np.nanmean(chl_8_days[-598:]*Pacific_Tropical_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl-a E_all')#,linewidth=0.5)

ax.plot(time,np.nanmean(chl[-156:]*Pacific_Tropical_area,axis=(1,2)),label='Chl-a',c='black')
#ax.set_title('CHL evolution over the Atlantic Subtropics North')
ax.set_xlim([2011,2023])
ax.set_ylabel("Chl-a mg/m3")
ax.set_ylim([0.1,0.2])

ax.legend(loc="upper left")
yticks = ax.get_yticks()
new_yticks = yticks[:-1]           
ax.set_yticks(new_yticks)          

# Second Row
# Seasonal Row

ax=axes[2]
time=np.linspace(2011,2023,156)
ax.plot(time,np.nanmean((chl_pred_glob[model_key][-156:]-chl_pred_glob_anom[model_key][-156:])*Pacific_Tropical_area,
                        axis=(1,2)),c='black', linestyle='dashed',label='Seasonal Climatology Chl-a E_all')
ax.plot(time,np.nanmean((chl[-156:]-chl_anom[-156:])*Pacific_Tropical_area,axis=(1,2)),label='Seasonal Climatology Chl-a',c='black')
ax.legend(loc="upper left")
ax.set_xlim([2011,2023])

ax.set_ylabel("Seasonal Climatology Chl-a mg/m3")
ax.set_ylim([0.1,0.2])
# Get current y-ticks and remove the last one
yticks = ax.get_yticks()
new_yticks = yticks[:-1]           # Remove last tick
ax.set_yticks(new_yticks)          # Set new ticks
#ax.set_title('Seasonal Chl over the Atlantic Subtropics North')



# Third row
#Chl anomalies
ax=axes[3]
time=np.linspace(2011,2023,156)


ax.plot(time,np.nanmean(chl_pred_glob_anom[model_key][-156:]*Pacific_Tropical_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl-a anomalies E_all')

ax.plot(time,np.nanmean(chl_anom[-156:]*Pacific_Tropical_area,axis=(1,2)),label='Chl-a anomalies',c='black')
#ax.set_title('CHL anomalie evolution over the Atlantic Subtropics North')
ax.set_xlim([2011,2023])
ax.set_ylabel("mg/m3")
ax.set_ylim([-0.05,0.05])

ax.legend(loc="upper left")



#Fourth row
#PSC %
ax=axes[4]
time=np.linspace(2011,2023,156)
ax.plot(time,100*np.nanmean(psc[-156:,0]*Pacific_Tropical_area,axis=(1,2)),c=cset[0],label='Micro%')
ax.plot(time,100*np.nanmean(psc[-156:,1]*Pacific_Tropical_area,axis=(1,2)),c=cset[1],label='Nano%')
ax.plot(time,100*np.nanmean(psc[-156:,2]*Pacific_Tropical_area,axis=(1,2)),c=cset[2],label='Pico%')

ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,0]*Pacific_Tropical_area,axis=(1,2)),
        c=cset[0], linestyle="--", label='Micro% E_all')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,1]*Pacific_Tropical_area,axis=(1,2)),
        c=cset[1],linestyle="--", label='Nano% E_all')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][-156:,2]*Pacific_Tropical_area,axis=(1,2)),
        c=cset[2],linestyle="--",label='Pico% E_all')
ax.legend(loc="upper left")
ax.set_xlim([2011,2023])

ax.set_ylabel("PSC %")
#ax.set_title('PSC evolution over the Atlantic Subtropics North')
ax.set_ylim([5,65])




#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/pacific_trop_evolution.png",  bbox_inches='tight')
#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/pacific_trop_evolution.pdf",format='pdf',  bbox_inches='tight')

plt.show()

print()

In [ ]:
matplotlib.rcParams.update({'font.size': 6})

fig = plt.figure(figsize=(7, 7), constrained_layout=False)#, dpi=1200)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,  top=1., bottom=0.825, wspace=0.05),  # First row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,  top=0.825, bottom=0.65, wspace=0.05),  # First row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.65, bottom=0.475, wspace=0.05),  # Second row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95,   top=0.475, bottom=0.3, wspace=0.05),  # Third row
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.95, top=0.3, bottom=0.),# Seasonal Chl
    #fig.add_gridspec(nrows=1, ncols=1, left=0.6, right=0.9, top=0.34, bottom=0.26), #Seasonal PSC
]


axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs] 

import tol_colors as tc
cset = tc.tol_cset('bright')


model_key='SmaAt_chlm_avw_variables_sst'


# First row
#ONI
time=np.linspace(1993,1998,60)

enso34=np.loadtxt("/home/luther/Documents/npy_data/oni.txt")
ENSO=enso34.flatten()[:60]
ax_t=axes[0]

pos=np.where(ENSO>0, ENSO,0)
neg=np.where(ENSO<0, ENSO,0)

ax_t.fill_between(time,pos,color='red', alpha=0.3,label='Oceanic Nino Index', linestyle='--',)
ax_t.fill_between(time,neg,color='blue', alpha=0.3,label=' ', linestyle='--',)

ax_t.set_ylim([-3, 3])
ax_t.legend(loc="upper left", frameon=False,bbox_transform=ax_t.transAxes)
ax_t.set_xlim([1993,1998])

ax_t.set_ylabel("ONI Index")

#ax_t.set_title('ONI Index')

#Chl
ax=axes[1]


ax.plot(time,np.nanmean(chl_pred_glob[model_key][:60]*Pacific_Tropical_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl-a E_SST')#,linewidth=0.5)

#ax.set_title('CHL evolution over the Atlantic Subtropics North')
ax.set_xlim([1993,1998])
ax.set_ylabel("Chl-a mg/m3")
ax.set_ylim([0.1,0.2])

ax.legend(loc="upper left")
yticks = ax.get_yticks()
new_yticks = yticks[:-1]           
ax.set_yticks(new_yticks)          

# Second Row
# Seasonal Row

ax=axes[2]
ax.plot(time,np.nanmean((chl_pred_glob[model_key][:60]-chl_pred_glob_anom[model_key][:60])*Pacific_Tropical_area,
                        axis=(1,2)),c='black', linestyle='dashed',label='Seasonal Climatology Chl-a E_SST')
ax.legend(loc="upper left")
ax.set_xlim([1993,1998])

ax.set_ylabel("Seasonal Climatology Chl-a mg/m3")
ax.set_ylim([0.1,0.2])
# Get current y-ticks and remove the last one
yticks = ax.get_yticks()
new_yticks = yticks[:-1]           # Remove last tick
ax.set_yticks(new_yticks)          # Set new ticks
#ax.set_title('Seasonal Chl over the Atlantic Subtropics North')



# Third row
#Chl anomalies
ax=axes[3]


ax.plot(time,np.nanmean(chl_pred_glob_anom[model_key][:60]*Pacific_Tropical_area,axis=(1,2)),c='black', 
        linestyle='dashed',label='Chl-a anomalies E_SST')

#ax.set_title('CHL anomalie evolution over the Atlantic Subtropics North')
ax.set_xlim([1993,1998])
ax.set_ylabel("mg/m3")
ax.set_ylim([-0.05,0.05])

ax.legend(loc="upper left")



#Fourth row
#PSC %
ax=axes[4]

ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][:60,0]*Pacific_Tropical_area,axis=(1,2)),
        c=cset[0], linestyle="--", label='Micro% E_sst')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][:60,1]*Pacific_Tropical_area,axis=(1,2)),
        c=cset[1],linestyle="--", label='Nano% E_sst')
ax.plot(time,100*np.nanmean(psc_pred_glob[model_key][:60,2]*Pacific_Tropical_area,axis=(1,2)),
        c=cset[2],linestyle="--",label='Pico% E_sst')
ax.legend(loc="upper left")
ax.set_xlim([1993,1998])

ax.set_ylabel("PSC %")
#ax.set_title('PSC evolution over the Atlantic Subtropics North')
ax.set_ylim([5,65])




#plt.savefig("/home/luther/Documents/plot/article_1/pacific_trop_evolution_1.png",  bbox_inches='tight')
plt.show()

print()

### Multi plot

In [ ]:

# ----------------------------
# Legend 1: Experiments (Markers)
# ----------------------------
keys=['SmaAt_chlm_avw_variables_winds',
 'SmaAt_chlm_avw_variables_sss',
 'SmaAt_chlm_avw_variables_all',
 'SmaAt_chlm_avw_variables_ssh',
 'SmaAt_chlm_avw_variables_sst',
 'SmaAt_chlm_avw_variables_solar',
 #'SmaAt_chlm_avw_variables_sst_ssh',
 'SmaAt_chlm_avw_variables_mld',
 'SmaAt_chlm_avw_variables_currents',
 #'SmaAt_chlm_avw_variables_sst_ssh_solar',
 #'SmaAt_chlm_avw_variables_sst_solar',
 #'SmaAt_chlm_avw_variables_sst_ssh_solarA',
 #'SmaAt_chlm_avw_variables_alla',
 #'SmaAt_chlm_avw_variables_sstA',
 #'SmaAt_chlm_avw_variables_allA',
 #'SmaAt_chlm_avw_variables_ssta',
 #'SmaAt_chlm_avw_variables_cosin',
 #'SmaAt_chlm_avw_variables_sst_ssh_solar_spatial',
'seasonal',]

rename=['E_Winds','E_SSS','E_All','E_SSH','E_SST','E_Cs','E_MLD','E_Currents','Monthly Climatology'] #'cosin','lat'

experiment_name_map = dict(zip(keys, rename))

experiment_handles = [
    Line2D(
        [0], [0],
        marker=marker_styles[i],
        color='gray',
        linestyle='None',
        markersize=8,
        markerfacecolor='gray',
        label=experiment_name_map[e_name]
    ) for i,(e_name,phy_index) in enumerate(index_experiments.items())
]

legend1 = plt.legend(
    handles=experiment_handles,
    title="Experiments",
    loc='upper left',
    bbox_to_anchor=(1.01, 1),
    fontsize=9
)
plt.gca().add_artist(legend1)  # Add first legend manually

In [ ]:
chl_pred_glob.keys()

In [ ]:

fig = plt.figure(figsize=(12, 5), constrained_layout=False)#, dpi=1200)


# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=1,  top=1., bottom=0., wspace=0.05),  # First row
]

axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs] 

import tol_colors as tc
cset = tc.tol_cset('bright')

marker_styles = ['o', 's', 'D', '^', 'v', '<', '>', 'p', '*', 'X']

# First row
#Chl
ax=axes[0]

time=np.linspace(1998,2023,312)
ax.plot(time,100*np.nanmean(psc[:,2]*Atlantique_Nord_area,axis=(1,2)),c=cset[2],label='Pico%')
keys=['SmaAt_chlm_avw_variables_winds',
 'SmaAt_chlm_avw_variables_sss',
 'SmaAt_chlm_avw_variables_all',
 'SmaAt_chlm_avw_variables_ssh',
 'SmaAt_chlm_avw_variables_sst',
 'SmaAt_chlm_avw_variables_solar',
 #'SmaAt_chlm_avw_variables_sst_ssh',
 'SmaAt_chlm_avw_variables_mld',
 'SmaAt_chlm_avw_variables_currents',
 #'SmaAt_chlm_avw_variables_sst_ssh_solar',
 'SmaAt_chlm_avw_variables_sst_solar',
 #'SmaAt_chlm_avw_variables_sst_ssh_solarA',
 #'SmaAt_chlm_avw_variables_alla',
 #'SmaAt_chlm_avw_variables_sstA',
 #'SmaAt_chlm_avw_variables_allA',
 #'SmaAt_chlm_avw_variables_ssta',
 #'SmaAt_chlm_avw_variables_cosin',
 #'SmaAt_chlm_avw_variables_sst_ssh_solar_spatial',
]
for i,model_key in enumerate(keys):


    year_max=np.nanmax(np.nanmean(psc_pred_glob[model_key][60:,2]*Atlantique_Nord_area,axis=(1,2)).reshape(26,12),axis=1)
    time_scatter=[time[j] for j in range(0,312,12)]
    ax.scatter(time_scatter,100*year_max,c=cset[2],marker=marker_styles[i % len(marker_styles)],
                edgecolor='black', s=80, alpha=0.85)

ax.legend(loc="upper left")
ax.set_xlim([1998,2023])

ax.set_ylabel("PSC %")
#ax.set_title('PSC evolution over the Atlantic Subtropics North')
ax.set_ylim([30,55])



# ----------------------------
# Legend 1: Experiments (Markers)
# ----------------------------


rename=['E_Winds','E_SSS','E_All','E_SSH','E_SST','E_Cs','E_MLD','E_Currents','E_sst_Cs'] #'cosin','lat'

experiment_name_map = dict(zip(keys, rename))

experiment_handles = [
    Line2D(
        [0], [0],
        marker=marker_styles[i],
        color='green',
        linestyle='None',
        markersize=8,
        markerfacecolor='gray',
        label=experiment_name_map[e_name]
    ) for i,(e_name) in enumerate(keys)
]

legend1 = plt.legend(
    handles=experiment_handles,
    title="Experiments",
    loc='upper left',
    #bbox_to_anchor=(1.01, 1),
    fontsize=9
)
plt.gca().add_artist(legend1)  # Add first legend manually


plt.show()

### Plot formal

In [ ]:
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'font.size': 18})


fig = plt.figure(figsize=(20, 4), constrained_layout=False)#,dpi=800)

# Define grid specifications and axes
grid_specs = [
    [fig.add_gridspec(nrows=1, ncols=2, left=0.0, right=0.085, wspace=0.05),  # First column
    fig.add_gridspec(nrows=1, ncols=2, left=0.09, right=0.19, wspace=0.05)],  # Second column
    [fig.add_gridspec(nrows=1, ncols=2, left=0.2, right=0.285, wspace=0.05),  # Third column
    fig.add_gridspec(nrows=1, ncols=2, left=0.29, right=0.39, wspace=0.05)],   # Fourth column
    [fig.add_gridspec(nrows=1, ncols=2, left=0.4, right=0.485, wspace=0.05),   # Fifth column
    fig.add_gridspec(nrows=1, ncols=2, left=0.49, right=0.59, wspace=0.05)],   # Sixth column
    [fig.add_gridspec(nrows=1, ncols=2, left=0.6, right=0.685, wspace=0.05),   # Seventh column
    fig.add_gridspec(nrows=1, ncols=2, left=0.69, right=0.79, wspace=0.05)],   # Height column
]

axes = [  
    [fig.add_subplot(gs[0][:, 0]), fig.add_subplot(gs[0][:, 1]), 
     fig.add_subplot(gs[1][:, 0]),fig.add_subplot(gs[1][:, 1])] for gs in grid_specs
]

# Add ghost axes and titles for gs
ghost_axes = [
    fig.add_subplot(fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=0.2, wspace=0.05)[:]),
    fig.add_subplot(fig.add_gridspec(nrows=1, ncols=1, left=0.2, right=0.4, wspace=0.05)[:]),
    fig.add_subplot(fig.add_gridspec(nrows=1, ncols=1, left=0.4, right=0.6, wspace=0.05)[:]),
    fig.add_subplot(fig.add_gridspec(nrows=1, ncols=1, left=0.6, right=0.8, wspace=0.05)[:]),
]



# Define color maps
color_maps = ["coolwarm", "coolwarm_r", "coolwarm", "coolwarm_r", "Blues"]
metric_names=['rmse', 'r2', 'rmse', 'r2', 'acc anom']
region='Global'



for i,data_name in enumerate(data_names):
    
    for j,dataframe in enumerate([-df_sorted_perf_rmse_bioregions[data_name],-df_sorted_perf_corr_bioregions[data_name],
                              df_sorted_rmse_anom_bioregions[data_name], df_sorted_corr_anom_bioregions[data_name],
                                  ]):
        color_map=color_maps[j]
        unique_sorted = np.sort(np.unique(dataframe[region]))
        vmin = unique_sorted[0]
        vmax =  unique_sorted[-1]
        midpoint = dataframe.loc['Monthly Climatology'][region]
    
        data=dataframe[[region]].drop(labels='Monthly Climatology').loc[['SST','All']]

        if j<2:
            #data.loc['Monthly Climatology']=np.nan
            norm = mcolors.TwoSlopeNorm(vcenter=0)
            labels = (np.asarray(["{0:.1f}%".format(value)
                    for value in data.values.flatten()])).reshape(*data.values.shape)
            fmt=""

        elif j==2:
            midpoint = dataframe.loc['Monthly Climatology'][region]
            norm = mcolors.TwoSlopeNorm(vcenter=midpoint)#,vmin=midpoint-(np.abs(midpoint)/2),  vmax=midpoint+(np.abs(midpoint)/2))
            labels=True
            fmt=".3f"

        elif j==3:
            midpoint = dataframe.loc['Monthly Climatology'][region]
            #print(midpoint)
            norm = mcolors.TwoSlopeNorm(vmin=0, vcenter=0.5, vmax=1)
            labels=True
            fmt=".3f"
            
        else:
            labels=True
            fmt=".4f"            
            vmin = unique_sorted[1]
            norm=None#mcolors.TwoSlopeNorm(vmin=vmin, vcenter=(vmax+vmin)/2, vmax=vmax)
            #data.loc['Monthly Climatology']=np.nan
        #print(i,j)
        # Create heatmap
        sns.heatmap(data, annot=labels, cmap=color_map, fmt=fmt, linewidths=0.5, ax=axes[i][j],  
                    cbar=False, norm=norm, xticklabels=False)


        ghost_axes[i].axis('off')
        ghost_axes[i].set_title(data_name)

        #ghost_axes_2[i].axis('off')
        #ghost_axes_2[i].set_title('vs seasonal     anomalies')

        
        axes[i][j].set_xlabel(metric_names[j])

        if i == 0 and j==0 :
            axes[i][j].set_ylabel("Experiment")
            axes[i][j].set_yticklabels(dataframe.drop('Monthly Climatology').loc[['SST','All']].index, rotation=0)
        else:
            axes[i][j].get_yaxis().set_ticks([])

#plot psc table

plt.savefig("/home/luther/Documents/plot/formal/heatmap_monthly_avw_psc_global_0.png", bbox_inches='tight' )

plt.show()

# RMSE Pacific investigation

In [ ]:
timestep=len(chl)
shape=chl.shape
Climato_Season=np.concatenate([np.nanmean(chl.reshape(timestep//12,12,*shape[1:]),axis=0)]*(timestep//12))
#Climato_Season_1=np.concatenate([np.nanmean(chl_decomposed.S.reshape(timestep//12,12,*shape[1:]),axis=0)]*(timestep//12))

a=chl_pred_glob['SmaAt_chlm_avw_variables_sst'][-156:]*new_bio_regions['Tropical Pacific']
anom=chl_pred_glob_anom['SmaAt_chlm_avw_variables_sst'][-156:]*new_bio_regions['Tropical Pacific']
c=Climato_Season[-156:]*new_bio_regions['Tropical Pacific']
d=chl[-156:]*new_bio_regions['Tropical Pacific']


year=np.linspace(2012,2023,156)
fig,ax=plt.subplots(1,1,figsize=(12,5))
ax.plot(year,np.nanmean(a,axis=(1,2)),label='chl_pred')
ax.plot(year,np.nanmean(d,axis=(1,2)),label='chl')

#plt.plot(error_anom,label='error chl_anom')
ax.plot(year,np.nanmean(c,axis=(1,2)),label='season')
ax.plot(year,np.nanmean(a-anom,axis=(1,2)),label='season_pred')

ax1=ax.twinx()

ax1.plot(time,np.nanmean(psc[-156:,0]*new_bio_regions['Tropical Pacific'],axis=(1,2)))
ax1.plot(time,np.nanmean(psc[-156:,1]*new_bio_regions['Tropical Pacific'],axis=(1,2)))
ax1.plot(time,np.nanmean(psc[-156:,2]*new_bio_regions['Tropical Pacific'],axis=(1,2)))

ax1.plot(time,np.nanmean(psc_pred_glob['SmaAt_chlm_avw_variables_all'][-156:,0]*new_bio_regions['Tropical Pacific'],axis=(1,2)))
ax1.plot(time,np.nanmean(psc_pred_glob['SmaAt_chlm_avw_variables_all'][-156:,1]*new_bio_regions['Tropical Pacific'],axis=(1,2)))
ax1.plot(time,np.nanmean(psc_pred_glob['SmaAt_chlm_avw_variables_all'][-156:,2]*new_bio_regions['Tropical Pacific'],axis=(1,2)))

plt.legend()
plt.show()
"""
plt.figure(figsize=(12,5))
plt.plot(year,np.sqrt(np.nanmean((d-c)**2,axis=(1,2))),label='error anom')
plt.plot(year,np.sqrt(np.nanmean((d-c)**2+(((a-anom)-(c))**2),axis=(1,2))),label='error season pred vs season + error anom')
plt.plot(year,np.sqrt(np.nanmean((a-(d))**2,axis=(1,2))),label='error chl_pred vs chl')

plt.legend()
plt.show()


plt.figure(figsize=(12,5))
plt.plot(year,np.sqrt(np.nanmean((d-c)**2,axis=(1,2))),label='error season vs chl ')
plt.plot(year,np.sqrt(np.nanmean((a-(d))**2,axis=(1,2))),label='error chl_pred vs chl')

plt.legend()
plt.show()

plt.figure(figsize=(12,5))
plt.plot(year,np.sqrt(np.nanmean((c-np.nanmean(c,axis=0))**2,axis=(1,2))),label='error season vs chl ')
plt.plot(year,np.sqrt(np.nanmean((a-(d))**2,axis=(1,2))),label='error chl_pred vs chl')

plt.legend()
plt.show()

plt.figure(figsize=(12,5))
plt.plot(year,np.sqrt(np.nanmean((d-c)**2,axis=(1,2))),label='distance de season par rapport à la moyenne')
plt.plot(year,np.sqrt(np.nanmean((a-(d))**2,axis=(1,2))),label='error chl_pred vs chl')

plt.legend()
plt.show()


error_a=np.sqrt(np.nanmean((a-d)**2))
error_season=np.sqrt(np.nanmean(((d-c)**2)))

error_anom=np.sqrt(np.nanmean((anom-(d-c))**2))
error_season_pred=np.sqrt(np.nanmean(((a-anom)-c)**2))

corrchl, _ = pearsonr(np.nanmean(a,axis=(1,2)), np.nanmean(d,axis=(1,2)))
corrseason, _ = pearsonr(np.nanmean(a-anom,axis=(1,2)), np.nanmean(c,axis=(1,2)))
corranom, _ = pearsonr(np.nanmean(anom,axis=(1,2)), np.nanmean(d-c,axis=(1,2)))

corrchl_season, _ = pearsonr(np.nanmean(c,axis=(1,2)), np.nanmean(d,axis=(1,2)))

print(f"error chl pred : {np.mean(error_a)}, error season : {np.mean(error_season)}")
print(f"error chl anom pred : {np.mean(error_anom)},error season pred : {np.mean(error_season_pred)}")

print(f"corr chl pred vs chl: {corrchl}, season pred vs season : {corrseason},\nanom pred vs anom : {corranom},season vs chl:{corrchl_season}")
"""
print()

In [ ]:
timestep=len(chl)
shape=chl.shape
Climato_Season=np.concatenate([np.nanmean(chl.reshape(timestep//12,12,*shape[1:]),axis=0)]*(timestep//12))
#Climato_Season_1=np.concatenate([np.nanmean(chl_decomposed.S.reshape(timestep//12,12,*shape[1:]),axis=0)]*(timestep//12))

a=chl_pred_glob['SmaAt_chlm_avw_variables_sst'][-156:,50,50]
anom=chl_pred_glob_anom['SmaAt_chlm_avw_variables_sst'][-156:,50,50]
c=Climato_Season[-156:,50,50]
d=chl[-156:,50,50]


year=np.linspace(2012,2023,156)
plt.figure(figsize=(12,5))
plt.plot(year,a,label='chl_pred')
plt.plot(year,d,label='chl')

#plt.plot(error_anom,label='error chl_anom')
plt.plot(year,c,label='season')
plt.plot(year,a-anom,label='season_pred')

plt.legend()
plt.show()

plt.figure(figsize=(12,5))
plt.plot(year,np.sqrt((d-c)**2),label='error anom')
plt.plot(year,np.sqrt((d-c)**2+(((a-anom)-(c))**2)),label='error season pred vs season + error anom')
plt.plot(year,np.sqrt((a-(d))**2),label='error chl_pred vs chl')

plt.legend()
plt.show()


plt.figure(figsize=(12,5))
plt.plot(year,np.sqrt((d-c)**2),label='error season vs chl ')
plt.plot(year,np.sqrt((a-(d))**2,),label='error chl_pred vs chl')

plt.legend()
plt.show()

plt.figure(figsize=(12,5))
plt.plot(year,np.sqrt((c-np.nanmean(c,axis=0))**2),label='error season vs chl ')
plt.plot(year,np.sqrt((a-(d))**2),label='error chl_pred vs chl')

plt.legend()
plt.show()

plt.figure(figsize=(12,5))
plt.plot(year,np.sqrt((d-c)**2),label='distance de season par rapport à la moyenne')
plt.plot(year,np.sqrt((a-(d))**2),label='error chl_pred vs chl')

plt.legend()
plt.show()




In [ ]:
timestep=len(chl)
shape=chl.shape
Climato_Season=np.concatenate([np.nanmean(chl.reshape(timestep//12,12,*shape[1:]),axis=0)]*(timestep//12))
#Climato_Season_1=np.concatenate([np.nanmean(chl_decomposed.S.reshape(timestep//12,12,*shape[1:]),axis=0)]*(timestep//12))

a=chl_pred_glob['SmaAt_chlm_avw_variables_sst']*new_bio_regions['Indian ocean']
anom=chl_pred_glob_anom['SmaAt_chlm_avw_variables_sst']*new_bio_regions['Indian ocean']
c=Climato_Season*new_bio_regions['Indian ocean']
d=chl*new_bio_regions['Indian ocean']


year=np.linspace(1998,2023,312)
plt.figure(figsize=(12,5))
plt.plot(year,np.nanmean(a[60:],axis=(1,2)),label='chl_pred')
plt.plot(year,np.nanmean(d,axis=(1,2)),label='chl')

#plt.plot(error_anom,label='error chl_anom')
plt.plot(year,np.nanmean(c,axis=(1,2)),label='season')
plt.plot(year,np.nanmean(a[60:]-anom[60:],axis=(1,2)),label='season_pred')

plt.legend()
plt.show()

plt.figure(figsize=(12,5))
plt.plot(year,np.sqrt(np.nanmean((anom[60:]-(d-c))**2,axis=(1,2))),label='error chl_anom_pred')
plt.plot(year,np.sqrt(np.nanmean(((a[60:]-anom[60:])-(c))**2,axis=(1,2))),label='error chl_season_pred')

plt.legend()
plt.show()





error_a=np.sqrt(np.nanmean((a[216:]-d[156:])**2))
error_season=np.sqrt(np.nanmean(((d[156:]-c[156:])**2)))

error_anom=np.sqrt(np.nanmean((anom[216:]-(d[156:]-c[156:]))**2))
error_season_pred=np.sqrt(np.nanmean(((a[216:]-anom[216:])-c[156:])**2))

print(f"error chl pred : {np.mean(error_a)}, error season : {np.mean(error_season)}")
print(f"error chl anom pred : {np.mean(error_anom)},error season pred : {np.mean(error_season_pred)}")
print()

In [ ]:
"""
plt.figure(figsize=(12,5))
#plt.plot(year,np.sqrt(np.nanmean((a[60:]-c)**2,axis=(1,2))),label='chl_pred-season')
plt.plot(year,np.sqrt(np.nanmean((a[60:]-d)**2,axis=(1,2))),label='chl_pred-chl')

plt.plot(year,np.sqrt(np.nanmean((d-c)**2,axis=(1,2))),label='chl-season')

#plt.plot(error_anom,label='error chl_anom')
#plt.plot(year,np.nanmean(c,axis=(1,2)),label='season')
plt.legend()
plt.show()


plt.figure(figsize=(12,5))
#plt.plot(year,np.sqrt(np.nanmean((a[60:]-c)**2,axis=(1,2))),label='chl_pred-season')
plt.plot(year,(np.nanmean((a[60:]-d),axis=(1,2))),label='chl_pred-chl')

plt.plot(year,(np.nanmean((d-c),axis=(1,2))),label='chl-season')

#plt.plot(error_anom,label='error chl_anom')
#plt.plot(year,np.nanmean(c,axis=(1,2)),label='season')
plt.legend()
plt.show()

plt.figure(figsize=(12,5))
plt.plot(year,np.nanmean(anom[60:],axis=(1,2)),label='chl_pred_anom')
plt.plot(year,np.nanmean(d-c,axis=(1,2)),label='chl anom')

#plt.plot(error_anom,label='error chl_anom')
#plt.plot(year,np.nanmean(c,axis=(1,2)),label='season')
plt.legend()
plt.show()




error_a=np.sqrt(np.nanmean((a[216:]-d[156:])**2))
error_anom=np.sqrt(np.nanmean((anom[216:]-(d[156:]-c[156:]))**2))
error_anom_season=np.sqrt(np.nanmean((d[156:]-c[156:])**2))
error_season=np.sqrt(np.nanmean(((d[156:]-c[156:])**2)))

print(np.mean(error_a),np.mean(error_season))
print(np.mean(error_anom),np.mean(error_anom_season))


a=chl_pred_glob['SmaAt_chlm_avw_variables_sst']

error_season=np.sqrt(np.nanmean((Climato_Season[-156:]-chl[-156:])**2,axis=0))
error=np.sqrt(np.nanmean((a[-156:]-chl[-156:])**2,axis=0))
carte_test=error-error_season
imshow_area(error-error_season,vmin=-0.01,vmax=0.01,cmap='coolwarm')

test_area=np.zeros((100,360))
test_area[45:50,0:90]=1
test_area[45:50,330:]=1
test_area[~test_area.astype(np.bool_)]=np.nan
test_area_2=np.where(carte_test<0,1,np.nan)
test_area=test_area_2*test_area
imshow_area(test_area)
"""
print()

# Trend learn comparaison

In [ ]:
import cartopy.crs as ccrs
from utils.temporal_decomposition_monthly import temporal_decomposition_monthly



def trend_plot(name):
    #trend whole serie
    #decomposition chl pred and chl true
    chl_decomposed=temporal_decomposition_monthly(np.squeeze(chl))
    chl_pred_decomposed=temporal_decomposition(np.squeeze(chl_pred_glob[name]))
    
    #trend over all (except boundaries years)
    chl_trend=compute_trend_iav(chl_decomposed.T[12:-12])
    chl_pred_trend=compute_trend_iav(chl_pred_decomposed.T[12:-12]) 

    #trend pred over 1998-2010
    chl_pred_trend_learned=compute_trend_iav(chl_pred_decomposed.T[12:156])
    chl_trend_learned=compute_trend_iav(chl_decomposed.T[12:156]) 


    imshow_area(12*chl_trend,vmin=-5e-3,vmax=5e-3, #fig=fig, ax=axes[0,0],
                cmap=mycmp,title='Chlorophyll-a avw Trend 1998-2023',colorbar="mg/m3*an",
                save="/home/luther/Documents/plot/article_1/test_trend/chl_avw_trend_glob.png")
    
    imshow_area(12*chl_pred_trend,vmin=-5e-3,vmax=5e-3, #fig=fig, ax=axes[1,0],
                cmap=mycmp,title=f'Chlorophyll-a {name} avw Trend 1998-2023',colorbar="mg/m3*an",
                save=f"/home/luther/Documents/plot/article_1/test_trend/chl_avw_{name}_glob.png")

    
    imshow_area(12*chl_trend_learned,vmin=-5e-3,vmax=5e-3, #fig=fig, ax=axes[0,1],
                cmap=mycmp,title='Chlorophyll-a avw Trend learned 1998-2010',colorbar="mg/m3*an",
                save="/home/luther/Documents/plot/article_1/test_trend/chl_avw_trend_learned_glob.png")

    imshow_area(12*chl_pred_trend_learned,vmin=-5e-3,vmax=5e-3, #fig=fig, ax=axes[1,1],
                cmap=mycmp,title=f'Chlorophyll-a {name} Trend 1998-2010',colorbar="mg/m3*an",
                save=f"/home/luther/Documents/plot/article_1/test_trend/chl_avw_{name}_learned_glob.png")

#for key in chl_pred_glob.keys():
#    trend_plot(key)

In [ ]:

import cartopy.crs as ccrs

def trend_plot(name):

    #trend whole serie
    #decomposition chl pred and chl true
    chl_decomposed=temporal_decomposition_monthly(np.squeeze(chl))
    
    weekly_mean=np.concatenate([np.nanmean(chl_decomposed.S.reshape(len(chl_decomposed.S)//12,12,*chl_decomposed.S.shape[1:]),axis=0)]*(len(chl_decomposed.S)//12))
    delta_seasonal = chl_decomposed.S-weekly_mean

    chl_trend=compute_trend_iav(chl_decomposed.T[12:-12]+delta_seasonal[12:-12])
    chl_trend_learned=compute_trend_iav(chl_decomposed.T[12:156]+delta_seasonal[12:156]) 

    #trend over all (except boundaries years)
    chl_pred_decomposed=temporal_decomposition(np.squeeze(chl_pred_glob[name]))
    
    weekly_mean=np.concatenate([np.nanmean(chl_pred_decomposed.S.reshape(len(chl_pred_decomposed.S)//12,12,*chl_pred_decomposed.S.shape[1:]),axis=0)]*(len(chl_pred_decomposed.S)//12))
    delta_seasonal = chl_pred_decomposed.S-weekly_mean

    chl_pred_trend=compute_trend_iav(chl_pred_decomposed.T[12:-12]+delta_seasonal[12:-12]) 
    chl_pred_trend_learned=compute_trend_iav(chl_pred_decomposed.T[12:156]+delta_seasonal[12:156])

    #trend pred over 1998-2010

    imshow_area(12*chl_trend,vmin=-5e-3,vmax=5e-3, #fig=fig, ax=axes[0,0],
                cmap=mycmp,title='Chlorophyll-a avw Trend 1998-2023',colorbar="mg/m3*an",
                save="/home/luther/Documents/plot/article_1/test_trend/chl_avw_trend_glob_1.png")
    
    imshow_area(12*chl_pred_trend,vmin=-5e-3,vmax=5e-3, #fig=fig, ax=axes[1,0],
                cmap=mycmp,title=f'Chlorophyll-a {name} avw Trend 1998-2023',colorbar="mg/m3*an",
                save=f"/home/luther/Documents/plot/article_1/test_trend/chl_avw_{name}_glob_1.png")

    
    imshow_area(12*chl_trend_learned,vmin=-5e-3,vmax=5e-3, #fig=fig, ax=axes[0,1],
                cmap=mycmp,title='Chlorophyll-a avw Trend learned 1998-2010',colorbar="mg/m3*an",
                save="/home/luther/Documents/plot/article_1/test_trend/chl_avw_trend_learned_glob_1.png")

    imshow_area(12*chl_pred_trend_learned,vmin=-5e-3,vmax=5e-3, #fig=fig, ax=axes[1,1],
                cmap=mycmp,title=f'Chlorophyll-a {name} Trend 1998-2010',colorbar="mg/m3*an",
                save=f"/home/luther/Documents/plot/article_1/test_trend/chl_avw_{name}_learned_glob_1.png")

trend_plot(key)
trend_plot(key1)
trend_plot(key2)

### Trend impact study 

In [ ]:
sst_multiannual.shape

In [ ]:
sst_decomposed=temporal_decomposition_monthly(physics[:,1])

timestep=len(sst_decomposed.S)#we suppose that t is the first dim
shape=sst_decomposed.S.shape
weekly_mean=np.concatenate([np.nanmean(sst_decomposed.S.reshape(timestep//12,12,*shape[1:]),axis=0)]*(timestep//12))

delta_seasonal = sst_decomposed.S-weekly_mean
sst_multiannual= sst_decomposed.T+delta_seasonal

sst_trend=compute_trend_iav(sst_multiannual[12:-12]) 
imshow_area(12*sst_trend,cmap='seismic', title='global trend sst')

sst_trend_learned=compute_trend_iav(sst_multiannual[12:156]) 
imshow_area(12*sst_trend_learned,cmap='seismic', title='learned trend sst')

sst_trend_test=compute_trend_iav(sst_multiannual[156:-12]) 
imshow_area(12*sst_trend_test,cmap='seismic', title='test trend sst')

# Plot introduction

In [ ]:
chl1=xr.open_dataset("/home/luther/Documents/npy_data/chl/monthly_avw/chl_avw_m_glob_lat50_1998_2023.nc")['CHL1_mean'].values

#psc_roy
psc1=xr.open_dataset("/home/luther/Documents/npy_data/PSC/chl_psc_avw_roy_monthly_glob_100km_1998_2023_1.nc")[['Micro','Nano','Pico']].to_array(dim='variables').data[...,40:-40,:]
psc1 = np.swapaxes(psc1, 0, 1) if len(psc1) < 10 else ps1  # Swap axes only for .nc files, in nc files the variables are on the first axis instead of the second one
psc_conc1=psc1*chl1[:,None]


In [ ]:
import cartopy.feature as cfeature
import cartopy.crs as ccrs
from matplotlib.colors import LogNorm, Normalize
matplotlib.rcParams.update({'font.size': 6})

proj=ccrs.Robinson(central_longitude=200)

#plot
fig = plt.figure(figsize=(7, 7), constrained_layout=False, dpi=600)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=0.45,  top=0.8, bottom=0.4, wspace=0.05),  # Chl
    fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=0.45,  top=0.5, bottom=0.1, wspace=0.05),  # Micro
    fig.add_gridspec(nrows=1, ncols=1, left=0.55, right=1.,  top=0.8, bottom=0.4, wspace=0.05),  # Nano
    fig.add_gridspec(nrows=1, ncols=1, left=0.55, right=1.,  top=0.5, bottom=0.1, wspace=0.05),  # Pico
    fig.add_gridspec(nrows=1, ncols=1, left=0., right=0.45,  top=0.5, bottom=0.4, wspace=0.05),  # colorbar chl
    fig.add_gridspec(nrows=1, ncols=1, left=0., right=0.45,  top=0.2, bottom=0.1, wspace=0.05),  # colorbar Micro
    fig.add_gridspec(nrows=1, ncols=1, left=0.55, right=1.,  top=0.5, bottom=0.4, wspace=0.05),  # colorbar Nano
    fig.add_gridspec(nrows=1, ncols=1, left=0.55, right=1.,  top=0.2, bottom=0.1, wspace=0.05),  # colorbar Pico

]

axes=[fig.add_subplot(gs[0,0],projection=proj) for gs in grid_specs[:-4]]  
colorbar_chl=fig.add_subplot(grid_specs[-4][0,0])
colorbars_psc=[fig.add_subplot(grid_specs[i][0,0]) for i in range(-3,0)]

imshow_area(np.nanmean(chl1,axis=0), fig=fig, ax=axes[0], cmap='viridis', vmin=-3,vmax=1
            , colorbar=False, log=True, title='Mean Chl (mg/m3)')

imshow_area(np.nanmean(psc[:,0],axis=0), fig=fig, ax=axes[1], cmap='GnBu', 
            vmin=0.1,vmax=0.9, colorbar=False, log=False, title='Mean Micro : %')
imshow_area(np.nanmean(psc[:,1],axis=0), fig=fig, ax=axes[2], cmap='GnBu', 
            vmin=0.1,vmax=0.9, colorbar=False, log=False, title='Mean Nano : %')
imshow_area(np.nanmean(psc[:,2],axis=0), fig=fig, ax=axes[3], cmap='GnBu', 
            vmin=0.1,vmax=0.9, colorbar=False, log=False, title='Mean Pico : %')

for cb in colorbars_psc :
    cb.axis('off')
    fig.colorbar(
        ScalarMappable(norm=Normalize(vmin=0., vmax=100), cmap='GnBu'),
        ax=cb,
        orientation='horizontal',
        shrink=0.8,
        fraction=1.,
        extend='both',
        aspect=30,
        label='%'
    )

colorbar_chl.axis('off')
fig.colorbar(
    ScalarMappable(norm=LogNorm(vmin=1e-3, vmax=1), cmap='viridis'),
    ax=colorbar_chl,
    orientation='horizontal',
    shrink=0.8,
    fraction=1.,
    extend='both',
    aspect=30,
    label='mg/m3'
)

labels = ['(a)', '(b)', '(c)', '(d)']

for ax, lab in zip(axes, labels):
    ax.text(-0.05, 1.05, lab, transform=ax.transAxes, fontsize=10, fontweight='bold')
plt.savefig("/home/luther/Documents/plot/article_1/fig1_intro_mean_4.png", bbox_inches='tight')
#plt.savefig("/home/luther/Documents/plot/article_1/fig1_intro_mean_3.pdf", format="pdf", bbox_inches="tight")
plt.show()
print()

## Plot IA Spatial 

In [ ]:
### code stuff for stations
def coord_to_index(lon, lat):
    
    idx_lon=lon+180
    idx_lat=50-lat    
    return (idx_lat, idx_lon)

#stations keerthi
stations_coord={
'SWAtl':(-54,-43),
'NEAtl':(-15,45),
'WEqPac':(171,0),
'SIO':(81,-25),
'SCTR':(80,-6),
}

stations_coord={
'SWAtl':(-55,-43),
'NAtl':(-37,36),
'NPac':(156,24),
'SIO':(60,-32),
'SCTR':(80,-3),
}


stations_array_idx={}

for station in stations_coord.keys():
    stations_array_idx[station]=coord_to_index(*stations_coord[station])

In [ ]:
##plot stations
import cartopy
import cartopy.feature as cfeature
import cartopy.crs as ccrs

axes_label=[-179.5, 179.5, -49.5, 49.5] 
lon, lat = np.mgrid[axes_label[0]:axes_label[1]+0.5,
                    axes_label[2]:axes_label[3]+0.5] #ça marche parce qu'on a 1 degré de lib
proj=ccrs.Mercator()
fig=plt.figure(figsize=(20,15))
ax=plt.subplot(1,1,1, projection=proj)
ax.stock_img()
ax.coastlines(alpha=0.5)
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,linewidth=0.5, color='gray', alpha=0.2, linestyle='solid')
ax.set_extent(axes_label, crs=ccrs.PlateCarree())

for station in ['NAtl']:#stations_coord.keys():
    ax.text(*stations_coord[station],station,horizontalalignment='right',transform=ccrs.PlateCarree(),fontsize='large')
    ax.scatter(*stations_coord[station], c='r', transform=ccrs.PlateCarree())

ax.set(xlabel='Longitude', ylabel='Latitude')
ax.set_title('Stations')

#plt.savefig("/home/luther/Documents/plot/formal//station_loc.png",bbox_inches='tight')

plt.show()

for station in stations_array_idx.keys():
    temp=psc[:,:,stations_array_idx[station][0],stations_array_idx[station][1]]
    print(f"percentage of PSC nan values at {station} station, on test set")
    print(f"{100*np.round((np.count_nonzero([np.isnan(temp)]))/temp.size,2)}%")

In [ ]:


matplotlib.rcParams.update({'font.size': 7})
matplotlib.rcParams.update({'xtick.labelsize': 7})


#plot
fig = plt.figure(figsize=(9, 6), constrained_layout=False, dpi=300)
# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.99,  top=1., bottom=0.7, wspace=0.05),  # First column
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.99,   top=0.65, bottom=0.35, wspace=0.05),  # Second column
    fig.add_gridspec(nrows=1, ncols=1, left=0.01, right=0.99,   top=0.3, bottom=0., wspace=0.05),  # Third column
]

axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs]


import tol_colors as tc
cset = tc.tol_cset('bright')

station='NAtl'
lon_station,lat_station=stations_array_idx[station][0],stations_array_idx[station][1]

#PSC
"""
ax=axes[0]
time=np.linspace(2011,2023,156)

ax.plot(time,100*psc[-156:,0,lon_station,lat_station],c=cset[1],label='Micro')
ax.plot(time,100*psc[-156:,1,lon_station,lat_station],c=cset[0],label='Nano')
ax.plot(time,100*psc[-156:,2,lon_station,lat_station],c=cset[3],label='Pico')

ax.plot(time,100*psc_pred_glob['SmaAt_chlm_avw_variables_sst'][-156:,0,lon_station,lat_station],
        c=cset[1], linestyle="--", label='Micro sst')
ax.plot(time,100*psc_pred_glob['SmaAt_chlm_avw_variables_sst'][-156:,0,lon_station,lat_station],
        c=cset[0],linestyle="--", label='Nano sst')
ax.plot(time,100*psc_pred_glob['SmaAt_chlm_avw_variables_sst'][-156:,0,lon_station,lat_station],
        c=cset[3],linestyle="--",label='Pico sst')

ax.legend(loc="upper left")
ax.set_xlim([2011,2023])
ax.set_ylim(5,75)
ax.set_ylabel("PSC %")
ax.set_title('PSC')
"""
#Chl
ax=axes[0]
#ax.plot(time,chl_pred_glob['SmaAt_chlm_avw_variables_sst'][-156:,lon_station,lat_station],
#        c='black', linestyle='dashed',label='E_SST')
#ax.plot(time,np.nanmean(chl_pred_glob['SmaAt_chlm_avw_variables_all'][-156:]*EL_NINO_area,axis=(1,2)), c='black',
#        linestyle='dashed',label='Chl ALL')

ax.plot(time,chl[-156:,lon_station,lat_station],label='Chl-a',c='black')


ax.set_xlim([2011,2023])
ax.set_ylabel("mg/m3")

ax.legend()

ax=axes[1]
#ax.plot(time,chl_pred_glob['SmaAt_chlm_avw_variables_sst'][-156:,lon_station,lat_station],
#        c='black', linestyle='dashed',label='E_SST')
#ax.plot(time,np.nanmean(chl_pred_glob['SmaAt_chlm_avw_variables_all'][-156:]*EL_NINO_area,axis=(1,2)), c='black',
#        linestyle='dashed',label='Chl ALL')

ax.plot(time,chl[-156:,lon_station,lat_station]-chl_anom[-156:,lon_station,lat_station],
        label='Seasonal Climatology Chl-a',c='black',linestyle='dashed')

ax.set_xlim([2011,2023])
ax.set_ylabel("mg/m3")

ax.legend()

#SST
ax=axes[2]

ax.plot(time,physics[-156:,1,lon_station,lat_station],label='SST',c=cset[1])
ax.set_xlim([2011,2023])
ax.set_ylabel("°C")

ax.legend()

plt.savefig('/home/luther/Documents/presentations/CSI_2/explication_climato.png',bbox_inches='tight')
plt.show()

# Correlation chl physique season

In [ ]:
#anomalies computing
def compute_anomalies(array):
    timestep=len(array)#we suppose that t is the first dim
    shape=array.shape
    weekly_mean=np.concatenate([np.nanmean(array.reshape(timestep//12,12,*shape[1:]),axis=0)]*(timestep//12))
    return array-weekly_mean

from scipy.stats import pearsonr,spearmanr
def compute_corr_map(A, B):
    time, lon, lat = A.shape
    corr_map = np.full((lon, lat), np.nan)

    for i in range(lon):
        for j in range(lat):
            a_series = A[:, i, j]
            b_series = B[:, i, j]

            # Handle missing values (NaNs) if any
            mask = np.isfinite(a_series) & np.isfinite(b_series)
            if np.sum(mask) > 1:  # Need at least 2 points for correlation
                #corr_map[i, j], _ = pearsonr(a_series[mask], b_series[mask])
                corr_map[i, j], _ = spearmanr(a_series[mask], b_series[mask])
    return corr_map
    
physics_anom=np.zeros_like(physics)
#anomalies
for i in range(physics.shape[1]):
    physics_anom[:,i]=compute_anomalies(physics[:,i])
    
corr_map=np.zeros_like(physics[0])
physics_ratio=np.zeros_like(physics[0])

for i in range(physics.shape[1]): 
    corr_map[i]=compute_corr_map(chl-chl_anom,physics[:,i]-physics_anom[:,i])
    physics_ratio[i]=np.nanstd(physics[:,i]-physics_anom[:,i],axis=0)/np.nanstd(physics_anom[:,i],axis=0)


In [ ]:
 #0 mld, 1 SST, 2 Salinité, 3 SSH, 4 Solar, 5 currents, 6 winds
for i in range(physics.shape[1]): 
    imshow_area(corr_map[i], title=f'corr {i}',cmap='coolwarm',vmin=-1,vmax=1)
    imshow_area(physics_ratio[i], title=f'ratio {i}',cmap='coolwarm',vmin=-1,vmax=1,log=True)

In [ ]:
mean_corr_br=np.zeros((len(bio_regions_for_plot.keys()),physics.shape[1]))
corr_mean_br=np.zeros((len(bio_regions_for_plot.keys()),physics.shape[1]))

for j,(name_br,br) in enumerate(bio_regions_for_plot.items()):
    chl_climato_br=np.nanmean((chl-chl_anom)*br,axis=(1,2))
    for i in range(physics.shape[1]): 
        physics_climato_br=np.nanmean((physics[:,i]-physics_anom[:,i])*br,axis=(1,2))
        mask = np.isfinite(physics_climato_br) & np.isfinite(chl_climato_br)
        corr_br, _=pearsonr(chl_climato_br[mask], physics_climato_br[mask])
        mean_corr_br[j,i]=np.nanmean(corr_map[i]*br)
        corr_mean_br[j,i]=corr_br
        #print(f'correl of mean {name_br} + {i} : {corr_br}')
        #print(f'mean of correl {name_br} + {i} : {np.nanmean(corr_map[i]*br)}')

fig = plt.figure(figsize=(7, 7), constrained_layout=False)#,dpi=800)
#norm = mcolors.TwoSlopeNorm(vmin=midpoint-(np.abs(midpoint)/2), vcenter=midpoint, vmax=midpoint+(np.abs(midpoint)/2))
#norm = mcolors.TwoSlopeNorm(vcenter=midpoint)

# moyenne des correlations
#sns.heatmap(mean_corr_br, annot=True, cmap='coolwarm', fmt=".4f", linewidths=0.5)#, ax=axes[i][j],  
        #cbar=False, norm=norm, xticklabels=False)
#plt.show()
fig, ax = plt.subplots(figsize=(7, 7), constrained_layout=False)#,dpi=800)
sns.heatmap(corr_mean_br, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5, ax=ax) 

ax.set_ylabel("bio regions")
ax.set_yticklabels(new_bio_regions.keys(), rotation=0)

ax.set_xlabel("")
ax.set_xticklabels(['mld','SST','SSS','SSH','Solar','currents','winds'],rotation=0)

#plt.savefig("/home/luther/Documents/plot/article_1/corr_phy_climato.png", bbox_inches='tight')
plt.show()

In [ ]:
index_experiments={'SmaAt_chlm_avw_variables_winds':6,'SmaAt_chlm_avw_variables_currents':5,
                   'SmaAt_chlm_avw_variables_mld':0,'SmaAt_chlm_avw_variables_sss':2,
                   'SmaAt_chlm_avw_variables_ssh':3,'SmaAt_chlm_avw_variables_ssr':4,
                   'SmaAt_chlm_avw_variables_sst':1,}

def NRMSE_chl(chl_pred, region, axis=None):
    return np.sqrt(np.nanmean((region*chl_pred-region*chl_test)**2,axis=axis))


def NRMSE_chl_season(chl_pred, region, axis=None):
    seasonal_pred=chl_pred-compute_anomalies(chl_pred)
    seasonal=chl_test-compute_anomalies(chl_test)
    return 1-np.sqrt(np.nanmean((region*seasonal_pred-region*seasonal)**2,axis=axis))/np.nanmean(region*seasonal)

def NRMSE_chl_anom(chl_pred, region, axis=None):
    anom_pred=compute_anomalies(chl_pred)[-156:]
    anom=compute_anomalies(chl)[-156:]
    return np.sqrt(np.nanmean(((region*anom_pred)-(region*anom))**2,axis=axis))

NRMSE_experiments={}
NRMSE_experiments_anom={}

for j,(name_br,br) in enumerate(bio_regions_for_plot.items()):
    NRMSE_experiments[name_br]={}
    NRMSE_experiments_anom[name_br]={}
    for network_name,physic_index in index_experiments.items():
        NRMSE_experiments[name_br][network_name]=NRMSE_chl(chl_pred_test[network_name],region=br)  
        NRMSE_experiments_anom[name_br][network_name]=NRMSE_chl_anom(chl_pred_glob[network_name],region=br)      
                                          

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tol_colors as tc

colors =  plt.cm.tab20c( np.linspace(0,1,20 ))
colors =  tc.tol_cmap('rainbow_discrete')( np.linspace(0,1,20 ))

bio_region_color={'Subtropics North':colors[0], #weak
                  'Tropical Pacific':colors[3], #weak
                  'Tropical Atlantique':colors[5], #weak 
                  'Subtropics South':colors[6],#medium
                  'Indian ocean':colors[8], #medium
                  'SO-SA-high biomass':colors[10], #medium
                  'Global':'black', #medium
                  'Seasonal strat Subtropics Pacific North':colors[12], #strong
                  'Seasonal strat Subtropics South':colors[14], #strong
                  'Seasonal strat Subtropics Atlantic North':colors[16], #strong
}
bio_region_color={'Subtropics North':'darkorange', #1
                  'Tropical Pacific':'crimson', #2
                  'Tropical Atlantique':'yellowgreen', #3 
                  'Subtropics South':'gold',#1
                  'Indian ocean':'darkgreen', #3
                  'SO-SA-high biomass':'darkturquoise', #4
                  'Global':'black', 
                  'Seasonal strat Subtropics Pacific North':'royalblue', #4
                  'Seasonal strat Subtropics South':'powderblue', #4
                  'Seasonal strat Subtropics Atlantic North':'navy', #4
}
biome_colors=bio_region_color

bio_region_name={'Subtropics North':'Subtropics North', #weak
                  'Tropical Pacific':'Tropical Pacific', #weak
                  'Tropical Atlantique':'Tropical Atlantic', #weak 
                  'Subtropics South':'Subtropics South',#medium
                  'Indian ocean':'Indian ocean', #medium
                  'SO-SA-high biomass':'SO-SA-high biomass', #medium
                  'Global':'Global', #medium
                  'Seasonal strat Subtropics Pacific North':'Seasonally Stratified Subtropics North Pacific (SeStrSub-NP)', #strong
                  'Seasonal strat Subtropics South':'Seasonally Stratified Subtropics South (SeStrSub-S)', #strong
                  'Seasonal strat Subtropics Atlantic North':'Seasonally Stratified Subtropics North Atlantic (SeStrSub-NA)', #strong
}



# Define color maps
"""
cmap=plt.cm.Greys
colors = cmap_1(np.linspace(0, 1, 256))# Create a new colormap with alpha
colors[:, -1]=0.3  # Set alpha to 0.5
from matplotlib.colors import ListedColormap# Create new colormap with modified alpha
new_cmap = ListedColormap(colors)
"""

plt.scatter(np.arange(20),np.ones(20), c=colors, s=180)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

bio_regions_for_plot_2=bio_regions_for_plot
if 'Seasonal strat Subtropics North' in bio_regions_for_plot_2.keys():
    bio_regions_for_plot_2.pop('Seasonal strat Subtropics North')
# Marker styles (one per experiment)
marker_styles = ['o', 's', 'D', '^', 'v', '<', '>', 'p', '*', 'X']

# Colors for bioregions
#colors = plt.cm.tab20c(np.linspace(0, 1, n_bioregions))
colors =  tc.tol_cmap('rainbow_discrete')( np.linspace(0,1,20 ))

# roup mapping for bioregions
# Define how each bioregion is grouped (you can replace this logic with your own)
group_mapping = {}
colors_mapping = {}
colors_counter=[0,6,12]
for name_br,br in bio_regions_for_plot_2.items():
    if np.nanmean(chl_snr*br)>1.2:
        group_mapping[name_br] = 'Strong Chl-a seasonal signal (Ratio>1.2)'

    elif np.nanmean(chl_snr*br)<0.8:
        group_mapping[name_br] = 'Weak  Chl-a seasonal signal (Ratio<0.8)'

    else:
        group_mapping[name_br] = 'Medium  Chl-a seasonal signal'


# Sort bioregions by group to ensure consistent legend order
sorted_bioregions = sorted(bio_regions_for_plot_2.items(), key=lambda item: group_mapping[item[0]])
# Sort bioregions by group to ensure consistent legend order
selected_bioregions = {'Global':bio_regions_for_plot_2['Global'],
                       'Subtropics North':bio_regions_for_plot_2['Subtropics North'],
                       'Tropical Pacific': bio_regions_for_plot_2['Tropical Pacific'],
                       'Tropical Atlantique':bio_regions_for_plot_2['Tropical Atlantique'],
                       'Seasonal strat Subtropics Atlantic North':bio_regions_for_plot_2['Seasonal strat Subtropics Atlantic North'],
                      }
colors_mapping['Global']='black'
# ----------------------------
# Plotting
# ----------------------------

plt.figure(figsize=(12, 7))

# Plot each data point with the corresponding experiment marker and bioregion color
for j,(name_br,br) in enumerate(selected_bioregions.items()):
    vmax=np.nanmax(list(NRMSE_experiments[name_br].values()))
    vmin=np.nanmin(list(NRMSE_experiments[name_br].values()))
    for i,(e_name,phy_index) in enumerate(index_experiments.items()):
        points=(NRMSE_experiments[name_br][e_name]-vmin)/(vmax-vmin)
        plt.scatter(
            points,
            np.nanmean(physics_ratio[phy_index]*br),
            marker=marker_styles[i % len(marker_styles)],
            color=bio_region_color[name_br],#colors[j],
            edgecolor='black',
            s=150,
            alpha=0.85
        )
        
    if name_br in ['Global','Seasonal strat Subtropics Atlantic North','Tropical Pacific']:
        x_points=[]
        y_points=[]
        for i,(e_name,phy_index) in enumerate(index_experiments.items()):
            x_points.append((NRMSE_experiments[name_br][e_name]-vmin)/(vmax-vmin))
            y_points.append(np.nanmean(physics_ratio[phy_index]*br))
        x_points,y_points=zip(*sorted(zip(x_points,y_points)))
        plt.plot(
            list(x_points),
            list(y_points),
            color=bio_region_color[name_br],#colors[j],
            alpha=0.5,
            linestyle='dashed'
        )
        
plt.yscale('log')
plt.ylim([1e-1,10e1])
#plt.xlim([0,1])
# ----------------------------
# Legend 1: Experiments (Markers)
# ----------------------------
keys=['SmaAt_chlm_avw_variables_winds',
 'SmaAt_chlm_avw_variables_sss',
 'SmaAt_chlm_avw_variables_all_ssr',
 'SmaAt_chlm_avw_variables_ssh',
 'SmaAt_chlm_avw_variables_sst',
 'SmaAt_chlm_avw_variables_ssr',
 #'SmaAt_chlm_avw_variables_sst_ssh',
 'SmaAt_chlm_avw_variables_mld',
 'SmaAt_chlm_avw_variables_currents',
 #'SmaAt_chlm_avw_variables_sst_ssh_solar',
 #'SmaAt_chlm_avw_variables_sst_solar',
 #'SmaAt_chlm_avw_variables_sst_ssh_solarA',
 #'SmaAt_chlm_avw_variables_alla',
 #'SmaAt_chlm_avw_variables_sstA',
 #'SmaAt_chlm_avw_variables_allA',
 #'SmaAt_chlm_avw_variables_ssta',
 #'SmaAt_chlm_avw_variables_cosin',
 #'SmaAt_chlm_avw_variables_sst_ssh_solar_spatial',
'seasonal',]

rename=['E_Winds','E_SSS','E_All','E_SSH','E_SST','E_Cs','E_MLD','E_Currents','Monthly Climatology'] #'cosin','lat'

experiment_name_map = dict(zip(keys, rename))

experiment_handles = [
    Line2D(
        [0], [0],
        marker=marker_styles[i],
        color='gray',
        linestyle='None',
        markersize=8,
        markerfacecolor='gray',
        label=experiment_name_map[e_name]#+'\n'
    ) for i,(e_name,phy_index) in enumerate(index_experiments.items())
]

legend1 = plt.legend(
    handles=experiment_handles,
    title="Experiments",
    loc='upper center',
    #bbox_to_anchor=(1.01, 1),
    fontsize=9
)
plt.gca().add_artist(legend1)  # Add first legend manually

# ----------------------------
# Legend 2: Bioregions (Colors)
# ----------------------------

# Bioregion legend handles grouped under >>1, <<1, ~1
bioregion_handles = []

last_group = None
for j, (name_br, br) in enumerate(selected_bioregions.items()):
    #group = group_mapping[name_br]

    """
    # Add a group header if this is the first of its group
    if group != last_group:
        bioregion_handles.append(
            Line2D(
                [0], [0],
                linestyle='None',
                marker='',
                label=f"{group}"
            )
        )
        last_group = group
    """
    bioregion_handles.append(
        Line2D(
            [0], [0],
            marker='o',
            color=bio_region_color[name_br],
            linestyle='None',
            markersize=8,
            label=bio_region_name[name_br],
            markeredgecolor='black'
        )
    )


legend2 = plt.legend(
    handles=bioregion_handles,
    title="Bioregions",
    loc='upper right',
    #bbox_to_anchor=(1.01, 0.35),
    fontsize=8
)

# ----------------------------
# Plot Formatting
# ----------------------------

plt.xlabel("NRMSE", fontsize=12)
plt.yticks(fontsize=12)
plt.ylabel("Seasonal Ratio \n(climatology std \ anomalies std)" , fontsize=12)
plt.title("      Chl-a NRMSE vs Seasonal Ratio \nfor each single input Experiment and Biomes", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# Show plot
#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/discussion_seasonality.png",bbox_inches="tight")
#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/discussion_seasonality.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
import cartopy.crs as ccrs
from matplotlib.colors import LinearSegmentedColormap

#PLOT

matplotlib.rcParams.update({'font.size': 7})
proj=ccrs.Robinson(central_longitude=200)
fig = plt.figure(figsize=(8, 4), constrained_layout=False, dpi=300)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=0.7,  top=1., bottom=0.),  # bioregion
    fig.add_gridspec(nrows=1, ncols=1, left=0.7, right=1.,  top=1., bottom=0.),  #legend
]

axes = [fig.add_subplot(grid_specs[0][0,0], projection=proj),
       fig.add_subplot(grid_specs[1][0,0])]



plot_bio_reg=np.zeros_like(bio_regions_for_plot_2['Global'])
color_list=[[0,[1,1,1]]]
for i,name in enumerate(bio_regions_for_plot_2.keys()):
    if name!='Global' :
        plot_bio_reg=np.where(bio_regions_for_plot_2[name]==1,i+1,plot_bio_reg)
        color_list.append([(i+1)/10,bio_region_color[name]])


cmap = LinearSegmentedColormap.from_list('test', color_list, N=11)

imshow_area(plot_bio_reg,cmap=cmap,title='Biomes', colorbar=False, fig=fig, ax=axes[0])

labels =  list(bio_regions_for_plot_2.keys())
#bio_region_color['Other']='white'
labels.remove('Global')

# Create color patches for legend
patches = [mpatches.Patch(color=bio_region_color[label], label=label) for label in labels]

bioregion_handles = []

last_group = None
for j, (name_br, br) in enumerate(sorted_bioregions):
    if name_br!='Global':
        group = '\n'+group_mapping[name_br]
        
        # Add a group header if this is the first of its group
        """
        if group != last_group:
            bioregion_handles.append(
                Line2D(
                    [0], [0],
                    linestyle='None',
                    marker='',
                    label=f"{group}"
                )
            )
            last_group = group
        """
        bioregion_handles.append(
            mpatches.Patch(
                color=bio_region_color[name_br],
                label=bio_region_name[name_br],
            )
        )






# Plot an empty figure with just the legend
axes[1].axis('off')  # Hide axes
legend = axes[1].legend(handles=bioregion_handles, loc='center left', frameon=False)
plt.tight_layout()
#plt.savefig('/home/luther/Documents/plot/article_1/figures_finales/bio_reg.png',bbox_inches='tight' )
#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/bio_reg.pdf", format="pdf", bbox_inches="tight")

plt.show()

In [ ]:
bio_region_name={'Subtropics North':'Subtropics North', #weak
                  'Tropical Pacific':'Tropical Pacific', #weak
                  'Tropical Atlantique':'Tropical Atlantique', #weak 
                  'Subtropics South':'Subtropics South',#medium
                  'Indian ocean':'Indian ocean', #medium
                  'SO-SA-high biomass':'SO-SA-high biomass', #medium
                  'Global':'Global', #medium
                  'Seasonal strat Subtropics Pacific North':'SeStrSub-NP', #strong
                  'Seasonal strat Subtropics South':'SeStrSub-S', #strong
                  'Seasonal strat Subtropics Atlantic North':'SeStrSub-NA', #strong
}

bio_region_name={'Subtropics North':'Subtropics North', #weak
                  'Tropical Pacific':'Tropical Pacific', #weak
                  'Tropical Atlantique':'Tropical Atlantic', #weak 
                  'Subtropics South':'Subtropics South',#medium
                  'Indian ocean':'Indian ocean', #medium
                  'SO-SA-high biomass':'SO-SA-high biomass', #medium
                  'Global':'Global', #medium
                  'Seasonal strat Subtropics Pacific North':'Seasonally Stratified Subtropics \nNorth Pacific (SeStrSub-NP)', #strong
                  'Seasonal strat Subtropics South':'Seasonally Stratified Subtropics \nSouth (SeStrSub-S)', #strong
                  'Seasonal strat Subtropics Atlantic North':'Seasonally Stratified Subtropics \nNorth Atlantic (SeStrSub-NA)', #strong
}


#biome_colors = {
#    name: plt.cm.tab20(i % 20)   # or any custom color list
#    for i, name in enumerate(biome_names)
#}
def compute_snr(signal):
    """
    inputs : signal_decomposed (needs to have S and T argumen
    """
    timestep=len(signal)#we suppose that t is the first dim
    shape=signal.shape
    weekly_mean=np.concatenate([np.nanmean(signal.reshape(timestep//12,12,*shape[1:]),axis=0)]*(timestep//12))
    anomalie_power = np.nanstd(signal-weekly_mean,axis=0)
    noise_power = np.nanstd(weekly_mean,axis=0)
    snr =  noise_power /anomalie_power
    return snr, noise_power, anomalie_power

#chl_snr, A, B=compute_snr(chl)



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

matplotlib.rcParams.update({'font.size': 10})

# biomes: dict[str, np.ndarray]   # shape (100, 360), boolean masks
# A, B: np.ndarray of shape (100, 360)
fig = plt.figure(figsize=(8, 4), constrained_layout=False, dpi=600)

A=chl-chl_anom
B=chl_anom+np.nanmean(chl,axis=0)

biome_names = []#['Global']
temp=list(bio_regions_for_plot_2.keys())
temp.remove('Global')
biome_names+=temp
# Example: one color per biome
biome_colors = {
    name: plt.cm.tab20(i % 20)   # or any custom color list
    for i, name in enumerate(biome_names)
}
# Collect masked values
#A_vals = [A[~np.isnan(bio_regions_for_plot_2[name])] for name in biome_names]
#B_vals = [B[~np.isnan(bio_regions_for_plot_2[name])] for name in biome_names]

A_vals=[np.nanmean(A*bio_regions_for_plot_2[name],axis=(1,2)) for name in biome_names]
B_vals=[np.nanmean(B*bio_regions_for_plot_2[name],axis=(1,2)) for name in biome_names]

#A_vals=[np.nanstd(A*bio_regions_for_plot_2[name],axis=0).flatten() for name in biome_names]
#B_vals=[np.nanstd(B*bio_regions_for_plot_2[name],axis=0).flatten() for name in biome_names]


A_vals=[A_vals[i][~np.isnan(A_vals[i])] for i in range(len(biome_names))]
B_vals=[B_vals[i][~np.isnan(B_vals[i])] for i in range(len(biome_names))]


# Build positions so that each biome has two adjacent boxes
positions_A = np.arange(len(biome_names)) * 3.0 - 0.4
positions_B = np.arange(len(biome_names)) * 3.0 + 0.4

plt.figure(figsize=(14, 6))

categories = [(positions_A[0]-0.4, positions_B[1]+0.4, 'lightyellow'), 
              (positions_A[2]-0.4, positions_B[2]+0.4, 'lightpink'),
             (positions_A[3]-0.4, positions_B[4]+0.4, 'lightgreen'),
             (positions_A[5]-0.4, positions_B[-1]+0.4, 'lightblue')]

# Add shaded regions for different periods
for start, end, color in categories:
    plt.axvspan(start, end, facecolor=color, alpha=0.5)


# Plot A
plt.boxplot(
    A_vals,
    positions=positions_A,
    widths=0.6,
    patch_artist=True,
    showfliers=False,          # ← remove outlier dots
    boxprops=dict(facecolor="white"),
    medianprops=dict(color="black")
)

# Plot B
plt.boxplot(
    B_vals,
    positions=positions_B,
    widths=0.6,
    patch_artist=True,
    showfliers=False,          # ← remove outlier dots
    boxprops=dict(facecolor="gray"),
    medianprops=dict(color="black")
)

# Fix x-axis labels
plt.xticks(np.arange(len(biome_names)) * 3.0, [bio_region_name[biome_names[i]] for i in range(len(biome_names))], rotation=45, ha='right')

plt.ylabel("Chl-a mg.m-3")
plt.title("Amplitude comparison of signals")

import matplotlib.patches as patches

ax = plt.gca()


#xs[2].legend(handles=axs[2].get_lines() + proxies, title='Legend')



for i, name in enumerate(biome_names):
    x = i * 3.0   # your tick positions
    color = bio_region_color[name]

    # Add a small color square at y = slightly below lowest data
    ax.add_patch(
        patches.Rectangle(
            (x - 0.3, ax.get_ylim()[0] - 0.05*(ax.get_ylim()[1]-ax.get_ylim()[0])),
            1.,                                      # width
            0.03*(ax.get_ylim()[1]-ax.get_ylim()[0]), # height (relative)
            facecolor=color,
            transform=ax.transData,
            clip_on=False
        )
    )
plt.tick_params(axis='x', pad=15)


A_patch = mpatches.Patch(facecolor="white", label="seasonal", edgecolor='black')
B_patch = mpatches.Patch(facecolor="gray", label="non seasonal", edgecolor='black')

plt.legend(handles=[A_patch, B_patch], loc="upper right")
plt.tight_layout()
#plt.savefig("/home/luther/Documents/plot/article_1/review/seasonal_nonseasonal_.png",bbox_inches="tight")
#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/seasonal_nonseasonal_.pdf", format="pdf", bbox_inches="tight")

plt.show()


In [ ]:
print("""
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

bio_regions_for_plot_2={'Global':bio_regions_for_plot['Global']}
if 'Seasonal strat Subtropics North' in bio_regions_for_plot_2.keys():
    bio_regions_for_plot_2.pop('Seasonal strat Subtropics North')
# Marker styles (one per experiment)
marker_styles = ['o', 's', 'D', '^', 'v', '<', '>', 'p', '*', 'X']

# Colors for bioregions
#colors = plt.cm.tab20c(np.linspace(0, 1, n_bioregions))
colors =  tc.tol_cmap('rainbow_discrete')( np.linspace(0,1,20 ))

# roup mapping for bioregions
# Define how each bioregion is grouped (you can replace this logic with your own)
group_mapping = {}
colors_mapping = {}
colors_counter=[0,6,12]
for name_br,br in bio_regions_for_plot_2.items():
    if np.nanmean(chl_snr*br)>1.2:
        group_mapping[name_br] = 'Strong Chl-a seasonal signal (Ratio>1.2)'

    elif np.nanmean(chl_snr*br)<0.8:
        group_mapping[name_br] = 'Weak  Chl-a seasonal signal (Ratio<0.8)'

    else:
        group_mapping[name_br] = 'Medium  Chl-a seasonal signal'


# Sort bioregions by group to ensure consistent legend order
sorted_bioregions = sorted(bio_regions_for_plot_2.items(), key=lambda item: group_mapping[item[0]])

colors_mapping['Global']='black'
# ----------------------------
# Plotting
# ----------------------------

plt.figure(figsize=(12, 7))

# Plot each data point with the corresponding experiment marker and bioregion color
for j,(name_br,br) in enumerate(bio_regions_for_plot_2.items()):
    vmax=np.nanmax(list(NRMSE_experiments[name_br].values()))
    vmin=np.nanmin(list(NRMSE_experiments[name_br].values()))
    for i,(e_name,phy_index) in enumerate(index_experiments.items()):
        points=(NRMSE_experiments[name_br][e_name]-vmin)/(vmax-vmin)
        plt.scatter(
            points,
            np.nanmean(physics_ratio[phy_index]*br),
            marker=marker_styles[i % len(marker_styles)],
            color=bio_region_color[name_br],#colors[j],
            edgecolor='black',
            s=150,
            alpha=0.85
        )

plt.yscale('log')
plt.ylim([1e-1,10e1])
#plt.xlim([0,1])
# ----------------------------
# Legend 1: Experiments (Markers)
# ----------------------------
keys=['SmaAt_chlm_avw_variables_winds',
 'SmaAt_chlm_avw_variables_sss',
 'SmaAt_chlm_avw_variables_all',
 'SmaAt_chlm_avw_variables_ssh',
 'SmaAt_chlm_avw_variables_sst',
 'SmaAt_chlm_avw_variables_solar',
 #'SmaAt_chlm_avw_variables_sst_ssh',
 'SmaAt_chlm_avw_variables_mld',
 'SmaAt_chlm_avw_variables_currents',
 #'SmaAt_chlm_avw_variables_sst_ssh_solar',
 #'SmaAt_chlm_avw_variables_sst_solar',
 #'SmaAt_chlm_avw_variables_sst_ssh_solarA',
 #'SmaAt_chlm_avw_variables_alla',
 #'SmaAt_chlm_avw_variables_sstA',
 #'SmaAt_chlm_avw_variables_allA',
 #'SmaAt_chlm_avw_variables_ssta',
 #'SmaAt_chlm_avw_variables_cosin',
 #'SmaAt_chlm_avw_variables_sst_ssh_solar_spatial',
'seasonal',]

rename=['E_Winds','E_SSS','E_All','E_SSH','E_SST','E_Cs','E_MLD','E_Currents','Monthly Climatology'] #'cosin','lat'

experiment_name_map = dict(zip(keys, rename))

experiment_handles = [
    Line2D(
        [0], [0],
        marker=marker_styles[i],
        color='gray',
        linestyle='None',
        markersize=8,
        markerfacecolor='gray',
        label=experiment_name_map[e_name]#+'\n'
    ) for i,(e_name,phy_index) in enumerate(index_experiments.items())
]

legend1 = plt.legend(
    handles=experiment_handles,
    title="Experiments",
    loc='upper center',
    #bbox_to_anchor=(1.01, 1),
    fontsize=9
)
plt.gca().add_artist(legend1)  # Add first legend manually

# ----------------------------
# Legend 2: Bioregions (Colors)
# ----------------------------

# Bioregion legend handles grouped under >>1, <<1, ~1
bioregion_handles = []

last_group = None
for j, (name_br, br) in enumerate(sorted_bioregions):
    group = group_mapping[name_br]
    
    # Add a group header if this is the first of its group
    if group != last_group:
        bioregion_handles.append(
            Line2D(
                [0], [0],
                linestyle='None',
                marker='',
                label=f"{group}"
            )
        )
        last_group = group
    
    bioregion_handles.append(
        Line2D(
            [0], [0],
            marker='o',
            color=bio_region_color[name_br],
            linestyle='None',
            markersize=8,
            label=bio_region_name[name_br],
            markeredgecolor='black'
        )
    )


legend2 = plt.legend(
    handles=bioregion_handles,
    title="Bioregions",
    loc='upper right',
    #bbox_to_anchor=(1.01, 0.35),
    fontsize=8
)

# ----------------------------
# Plot Formatting
# ----------------------------

plt.xlabel("NRMSE", fontsize=12)
plt.yticks(fontsize=12)
plt.ylabel("Input ratio between seasonal climatology std \nand \nanomalies std" , fontsize=12)
plt.title("                     NRMSE vs Input ratio (seasonal climatology std on anomalies std) \nby Experiment and Bioregion", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# Show plot
#plt.savefig("/home/luther/Documents/plot/article_1/discussion_plot_4.png")
#plt.savefig("/home/luther/Documents/presentations/CSI_2/discussion_plot_global.png")
#plt.savefig("/home/luther/Documents/plot/article_1/discussion_plot_4.pdf", format="pdf", bbox_inches="tight")
plt.show()
""")

## BR SNR

### Std moyenne

In [ ]:
def compute_snr(signal):
    """
    inputs : signal_decomposed (needs to have S and T argumen
    """
    timestep=len(signal)#we suppose that t is the first dim
    shape=signal.shape
    weekly_mean=np.concatenate([np.nanmean(signal.reshape(timestep//12,12,*shape[1:]),axis=0)]*(timestep//12))
    anomalie_power = np.nanstd(signal-weekly_mean,axis=0)
    noise_power = np.nanstd(weekly_mean,axis=0)
    snr =  noise_power /anomalie_power
    return snr, noise_power, anomalie_power

chl_snr, chl_season_var, chl_anom_var=compute_snr(chl)

for name_br,br in new_bio_regions.items():
    print(f"{name_br}: Seasonal variance = {np.nanmean(chl_season_var*br):.2e}, Anomaly variance = {np.nanmean(chl_anom_var*br):.2e}")
    print(f"{name_br}: mean chl-a value = {np.nanmean(chl*br):.2e}")

### std de la moyenne 

In [ ]:
def compute_snr(signal):
    """
    inputs : signal_decomposed (needs to have S and T argumen
    """
    timestep=len(signal)#we suppose that t is the first dim
    shape=signal.shape
    weekly_mean=np.concatenate([np.nanmean(signal.reshape(timestep//12,12,*shape[1:]),axis=0)]*(timestep//12))
    anomalie_power = np.nanstd(signal-weekly_mean,axis=0)
    noise_power = np.nanstd(weekly_mean,axis=0)
    snr =  noise_power /anomalie_power
    return snr, noise_power, anomalie_power

chl_snr, chl_season_var, chl_anom_var=compute_snr(chl)

for name_br,br in new_bio_regions.items():

    timestep=len(chl)#we suppose that t is the first dim
    shape=chl.shape
    weekly_mean=np.concatenate([np.nanmean(chl.reshape(timestep//12,12,*shape[1:]),axis=0)]*(timestep//12))
    anomalie_power = np.nanstd(np.nanmean((chl-weekly_mean)*br,axis=(1,2)),axis=0)
    noise_power = np.nanstd(np.nanmean(br*weekly_mean,axis=(1,2)),axis=0)
    
    print(f"{name_br}: Seasonal variance = {noise_power:.2e}, Anomaly variance = {anomalie_power:.2e}")
    print(f"{name_br}: mean chl-a value = {np.nanmean(chl*br):.2e}")

## Climato train 

In [ ]:
### rmse_vals_best construction
keys=['SmaAt_chlm_avw_variables_winds',
 'SmaAt_chlm_avw_variables_sss',
 'SmaAt_chlm_avw_variables_all',
 'SmaAt_chlm_avw_variables_all_ssr', #solar
 'SmaAt_chlm_avw_variables_ssh',
 'SmaAt_chlm_avw_variables_sst',
 'SmaAt_chlm_avw_variables_ssr', #solar
 'SmaAt_chlm_avw_variables_solar', 
 'SmaAt_chlm_avw_variables_mld',
 'SmaAt_chlm_avw_variables_currents',
 'seasonal']

biome_names = ['Global']
temp=list(bio_regions_for_plot_2.keys())
temp.remove('Global')
biome_names+=temp

def RMSE(target,true , axis=None):
    return np.sqrt(np.nanmean((target-true)**2,axis=axis))

rmse_chl={}
rmse_chl_anom={}
rmse_chl_seas={}
rmse_psc={}
rmse_psc_conc={}
rmse_psc_conc_anom={}
rmse_psc_conc_seas={}
rmse_psc_anom={}
rmse_psc_seas={}

relative_log_chl={}
relative_log_chl_anom={} #sym log ? #on ajoute la moyenne de la chl ?je pense on fait ça plutôt
relative_log_chl_seas={}
relative_log_psc_conc={}
relative_log_psc_conc_anom={}
relative_log_psc_conc_seas={}




for br in biome_names:
    rmse_chl[br]={}
    rmse_chl_anom[br]={}
    rmse_chl_seas[br]={}
    rmse_psc[br]={}
    rmse_psc_conc[br]={}
    rmse_psc_conc_anom[br]={}
    rmse_psc_conc_seas[br]={}
    rmse_psc_anom[br]={}
    rmse_psc_seas[br]={}

    relative_log_chl[br]={}
    relative_log_chl_anom[br]={} #sym log ? #on ajoute la moyenne de la chl ?je pense on fait ça plutôt
    relative_log_chl_seas[br]={}
    relative_log_psc_conc[br]={}
    relative_log_psc_conc_anom[br]={}
    relative_log_psc_conc_seas[br]={}

    for key in keys :

        if key!='seasonal':

            rmse_chl[br][key]=RMSE(bio_regions_for_plot_2[br]*chl_test,bio_regions_for_plot_2[br]*chl_pred_test[key])
            rmse_chl_anom[br][key]=RMSE(bio_regions_for_plot_2[br]*chl_anom_test,bio_regions_for_plot_2[br]*chl_pred_anom_test[key])
            rmse_chl_seas[br][key]=RMSE(bio_regions_for_plot_2[br]*(chl_test-chl_anom_test),bio_regions_for_plot_2[br]*(chl_pred_test[key]-chl_pred_anom_test[key]))
            
            rmse_psc[br][key]=RMSE(bio_regions_for_plot_2[br]*psc_test,bio_regions_for_plot_2[br]*psc_pred_test[key],axis=(0,2,3))
            rmse_psc_conc[br][key]=RMSE(bio_regions_for_plot_2[br]*psc_conc_test,bio_regions_for_plot_2[br]*psc_conc_pred_test[key],axis=(0,2,3))

            rmse_psc_conc_anom[br][key]=RMSE(bio_regions_for_plot_2[br]*psc_conc_anom_test,bio_regions_for_plot_2[br]*psc_conc_anom_pred_test[key],axis=(0,2,3))
            rmse_psc_conc_seas[br][key]=RMSE(bio_regions_for_plot_2[br]*(psc_conc_test-psc_conc_anom_test),bio_regions_for_plot_2[br]*(psc_conc_pred_test[key]-psc_conc_anom_pred_test[key]),axis=(0,2,3))

            rmse_psc_anom[br][key]=RMSE(bio_regions_for_plot_2[br]*psc_anom[-156:],bio_regions_for_plot_2[br]*psc_pred_glob_anom[key][-156:],axis=(0,2,3))
            rmse_psc_seas[br][key]=RMSE(bio_regions_for_plot_2[br]*(psc_test[-156:]-psc_anom[-156:]),bio_regions_for_plot_2[br]*(psc_pred_test[key][-156:]-psc_pred_glob_anom[key][-156:]),axis=(0,2,3))

            relative_log_psc_conc[br][key]=np.nanmean(np.log10(bio_regions_for_plot_2[br] * psc_conc_pred_test[key]) /np.log10(bio_regions_for_plot_2[br] * psc_conc_test),axis=(0,2,3))
            relative_log_psc_conc_anom[br][key] = np.nanmean(np.log10(bio_regions_for_plot_2[br] * (psc_conc_anom_pred_test[key] + np.nanmean(psc_conc_pred_glob[key], axis=0))) /np.log10(bio_regions_for_plot_2[br] * (psc_conc_anom_test + np.nanmean(psc_conc, axis=0))),axis=(0,2,3))
            relative_log_psc_conc_seas[br][key] = np.nanmean(np.log10(bio_regions_for_plot_2[br] * (psc_conc_pred_test[key] - psc_conc_anom_pred_test[key])) /np.log10(bio_regions_for_plot_2[br] * (psc_conc_test - psc_conc_anom_test)),axis=(0,2,3))

            relative_log_chl[br][key]=np.nanmean(np.log10(bio_regions_for_plot_2[br]*chl_pred_test[key])/np.log10(bio_regions_for_plot_2[br]*chl_test))
            relative_log_chl_anom[br][key]=np.nanmean(np.log10(bio_regions_for_plot_2[br]*(chl_pred_anom_test[key]+np.nanmean(chl_pred_glob[key],axis=0)))/np.log10(bio_regions_for_plot_2[br]*(chl_anom_test+np.nanmean(chl,axis=0))))
            relative_log_chl_seas[br][key]=np.nanmean(np.log10(bio_regions_for_plot_2[br]*(chl_pred_test[key]-chl_pred_anom_test[key]))/np.log10(bio_regions_for_plot_2[br]*(chl_test-chl_anom_test)))
            



chl_rmse_vals_best = []
chl_rmse_names_best = []

Micro_rmse_vals_best = []
Micro_rmse_names_best = []

Nano_rmse_vals_best = []
Nano_rmse_names_best = []

Pico_rmse_vals_best = []
Pico_rmse_names_best = []

EXCLUDED = {'seasonal', 'SmaAt_chlm_avw_variables_all'}

for b in biome_names:

    chl_list = []
    Micro_list = []
    Nano_list = []
    Pico_list = []

    for network_name in rmse_chl[b].keys():
        if network_name in EXCLUDED or network_name not in keys:
            continue

        chl_value   = rmse_chl[b][network_name]
        Micro_value = rmse_psc[b][network_name][0]
        Nano_value  = rmse_psc[b][network_name][1]
        Pico_value  = rmse_psc[b][network_name][2]

        chl_list.append((chl_value, network_name))
        Micro_list.append((Micro_value, network_name))
        Nano_list.append((Nano_value, network_name))
        Pico_list.append((Pico_value, network_name))

    # sort and keep only 2 best
    chl_top2   = sorted(chl_list)[:2]
    Micro_top2 = sorted(Micro_list)[:2]
    Nano_top2  = sorted(Nano_list)[:2]
    Pico_top2  = sorted(Pico_list)[:2]

    chl_rmse_vals_best.append([v for v, n in chl_top2])
    chl_rmse_names_best.append([n for v, n in chl_top2])

    Micro_rmse_vals_best.append([v for v, n in Micro_top2])
    Micro_rmse_names_best.append([n for v, n in Micro_top2])

    Nano_rmse_vals_best.append([v for v, n in Nano_top2])
    Nano_rmse_names_best.append([n for v, n in Nano_top2])

    Pico_rmse_vals_best.append([v for v, n in Pico_top2])
    Pico_rmse_names_best.append([n for v, n in Pico_top2])



In [ ]:
for i in rmse_chl.keys():
    print(
        'biomes : ' + i +
        ', Experiment : ' + 'SmaAt_chlm_avw_variables_all' +
        f", rmse: {rmse_chl[i]['SmaAt_chlm_avw_variables_all']}"
    )
    print(
        'biomes : ' + i +
        ', Experiment : ' + 'SmaAt_chlm_avw_variables_all_ssr' +
        f", rmse: {rmse_chl[i]['SmaAt_chlm_avw_variables_all_ssr']}"
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
biome_names = ['Global']
temp=list(bio_regions_for_plot_2.keys())
temp.remove('Global')
biome_names+=temp

n = len(bio_regions_for_plot_2.keys())

# Extract RMSE values in the same biome order
rmse_vals_all = np.array([rmse_chl[b]['SmaAt_chlm_avw_variables_all_ssr'] for b in biome_names])
rmse_vals_season = np.array([rmse_chl[b]['seasonal'] for b in biome_names])
rmse_vals_best = np.array(chl_rmse_vals_best)[:,0]
labels_best = np.array(chl_rmse_names_best)[:,0]
# Bar positions
x = np.arange(n)
width = 0.25

plt.figure(figsize=(14,6))

plt.bar(x - width,         rmse_vals_season, width, label='Error Seasonal signal', color='white', edgecolor='black')
plt.bar(x, rmse_vals_all, width, label='Error E_all', color='grey', edgecolor='black')
plt.bar(x + width, rmse_vals_best, width, label='Error E_Best_var',color='black', edgecolor='black')


# ---- ADD LABEL ABOVE EACH BLACK BAR ----
# ---- ADD LABEL ABOVE EACH BLACK BAR ----
for xi, height, label in zip(x + width, rmse_vals_best, labels_best):
    plt.text(xi, height + 0.01,   # position slightly above the bar
             experiment_name_map[label],
             ha='center', va='bottom', color='black', fontsize=10)


plt.xticks(x, biome_names, rotation=45, ha='right')
plt.ylabel("RMSE Chl-a mg/m3")
#plt.ylim([0,0.4])
plt.title("Error Chl-a by Biome")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n = len(bio_regions_for_plot_2.keys())

# Extract RMSE values in the same biome order
rmse_vals_all = np.array([rmse_chl[b]['SmaAt_chlm_avw_variables_all'] for b in biome_names])
rmse_vals_season = np.array([rmse_chl_seas[b]['SmaAt_chlm_avw_variables_all'] for b in biome_names])
rmse_vals_anom =  np.array([rmse_chl_anom[b]['SmaAt_chlm_avw_variables_all'] for b in biome_names])

# Bar positions
x = np.arange(n)
width = 0.25

plt.figure(figsize=(14,6))

plt.bar(x - width,         rmse_vals_season, width, label='Error Seasonal signal', color='white', edgecolor='black')
plt.bar(x, rmse_vals_all, width, label='Error E_all', color='grey', edgecolor='black')
plt.bar(x + width, rmse_vals_anom, width, label='Error anom',color='black', edgecolor='black')



plt.xticks(x, biome_names, rotation=45, ha='right')
plt.ylabel("RMSE Chl-a mg/m3")
#plt.ylim([0,0.4])
plt.title("Error Chl-a by Biome")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.patches as patches

def add_biome_color_patches(ax, biome_names, biome_colors):
    """
    Adds a small color rectangle under each xtick.
    biome_colors can be a dict {biome: color} or a list aligned with biome_names.
    """

    # Ensure drawing has happened (so ax.get_ylim() is valid)
    ax.figure.canvas.draw()

    # Get y-range to position patches below the axis
    y0, y1 = ax.get_ylim()
    height = 0.03 * (y1 - y0)     # height relative to data range
    y_pos = y0 - 0.06 * (y1 - y0) # vertical position slightly below bottom

    # x tick positions (shared axis)
    tick_positions = ax.get_xticks()

    for x, biome in zip(tick_positions, biome_names):
        color = biome_colors[biome]

        rect = patches.Rectangle(
            (x - 0.3, y_pos),   # bottom-left corner
            0.6,                # width
            height,             # height
            facecolor=color,
            edgecolor='none',
            clip_on=False,
            transform=ax.transData,
            zorder=2
        )
        ax.add_patch(rect)

    # Increase space between ticks and labels
    ax.tick_params(axis='x', pad=15)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Number of biomes
n = len(biome_names)
x = np.arange(n)
width = 0.25
categories = [(x[1]-1.5*width, x[2]+1.5*width, 'lightyellow'), 
              (x[3]-1.5*width, x[3]+1.5*width, 'lightpink'),
             (x[4]-1.5*width, x[5]+1.5*width, 'lightgreen'),
             (x[6]-1.5*width, x[-1]+1.5*width, 'lightblue')]

# Create the figure
fig, axes = plt.subplots(4, 1, sharex=True, figsize=(10, 12))
fig.subplots_adjust(hspace=0.25)   # spacing between plots

# -----------------------------
# DEFINE A FUNCTION TO DRAW ONE PANEL
# -----------------------------
def plot_panel(ax, rmse_vals_all, rmse_vals_best, labels_best,title, y_lim=[0,15],y_label='%'):
    # Add shaded regions for different periods
    for start, end, color in categories:
        ax.axvspan(start, end, facecolor=color, alpha=0.5)
    # Bars
    ax.bar(x - width, rmse_vals_all, width,
           label='Error E_all', color='white', edgecolor='black')
    ax.bar(x, rmse_vals_best[:,0], width,
           label='Error E_var_best 1', color='grey', edgecolor='black')
    ax.bar(x + width, rmse_vals_best[:,1], width,
           label='Error E_var_best 2', color='black', edgecolor='black')

    # Vertical labels inside black bars
    for i, (xi, label) in enumerate(zip(x, labels_best)):
        if  rmse_vals_best[i,0]>0.15*y_lim[1]:
            text_color='white'
            text_height_denominateur=2
            add=0
        else :
            text_color='black'
            text_height_denominateur=1
            add=0.15*y_lim[1]
            
        ax.text(xi, add+rmse_vals_best[i,0]/text_height_denominateur , experiment_name_map[label[0]],
                ha='center', va='center',
                rotation=90, color=text_color, fontsize=10)
        ax.text(xi + width, add+rmse_vals_best[i,1]/text_height_denominateur, experiment_name_map[label[1]],
                ha='center', va='center',
                rotation=90, color=text_color, fontsize=10)

    ax.set_title(title)
    ax.set_ylabel(y_label)
    ax.set_ylim(y_lim)
    ax.grid(axis='y', linestyle='--', alpha=0.4)




# -----------------------------
# PLOT YOUR FOUR PANELS
# -----------------------------
plot_panel(axes[0], np.array([rmse_chl[b]['SmaAt_chlm_avw_variables_all_ssr'] for b in biome_names]),
           np.array(chl_rmse_vals_best), chl_rmse_names_best, "Chl error",y_lim=[0.0,0.35],y_label='mg/m3')

plot_panel(axes[1],100*np.array([rmse_psc[b]['SmaAt_chlm_avw_variables_all_ssr'][0] for b in biome_names]), 
           100*np.array(Micro_rmse_vals_best), Micro_rmse_names_best, "Micro% error")

plot_panel(axes[2], 100*np.array([rmse_psc[b]['SmaAt_chlm_avw_variables_all_ssr'][1] for b in biome_names]),
           100*np.array(Nano_rmse_vals_best), Nano_rmse_names_best, "Nano% error")

plot_panel(axes[3], 100*np.array([rmse_psc[b]['SmaAt_chlm_avw_variables_all_ssr'][2] for b in biome_names]),
           100*np.array(Pico_rmse_vals_best), Pico_rmse_names_best, "Pico% error")



# Set xticklabels on bottom subplot
biome_names_2=['Global',
 'Subtropics North',
 'Subtropics South',
 'Tropical Pacific',
 'Tropical Atlantique',
 'Indian ocean',
 'Seasonal strat Subtropics \nSouth (SeStrSub-S)',
 'SO-SA-high biomass',
 'Seasonal strat Subtropics \nNorth Atlantic (SeStrSub-NA)',
 'Seasonal strat Subtropics \nNorth Pacific (SeStrSub-NP)']

bio_region_name={'Subtropics North':'Subtropics North', #weak
                  'Tropical Pacific':'Tropical Pacific', #weak
                  'Tropical Atlantique':'Tropical Atlantic', #weak 
                  'Subtropics South':'Subtropics South',#medium
                  'Indian ocean':'Indian ocean', #medium
                  'SO-SA-high biomass':'SO-SA-high biomass', #medium
                  'Global':'Global', #medium
                  'Seasonal strat Subtropics Pacific North':'Seasonally Stratified Subtropics \nNorth Pacific (SeStrSub-NP)', #strong
                  'Seasonal strat Subtropics South':'Seasonally Stratified Subtropics \nSouth (SeStrSub-S)', #strong
                  'Seasonal strat Subtropics Atlantic North':'Seasonally Stratified Subtropics \nNorth Atlantic (SeStrSub-NA)', #strong
}
axes[-1].set_xticks(x)
axes[-1].set_xticklabels(biome_names_2, rotation=45, ha='right')
add_biome_color_patches(axes[-1], biome_names, bio_region_color)

# ---- ADD COLOR PATCHES ON BOTTOM AXIS ----
add_biome_color_patches(axes[0], biome_names, bio_region_color)
add_biome_color_patches(axes[1], biome_names, bio_region_color)
add_biome_color_patches(axes[2], biome_names, bio_region_color)







# Global legend (optional)
handles, labels = axes[0].get_legend_handles_labels()
#fig.legend(handles, labels, loc='upper right')
axes[0].legend(handles, labels, loc='upper left')
#axes[1].legend(handles, labels, loc='upper left')
#axes[2].legend(handles, labels, loc='upper left')
#axes[3].legend(handles, labels, loc='upper left')


plt.tight_layout()
#plt.savefig("/home/luther/Documents/plot/article_1/review/result_rmse_br_tot_best_ssr.png",bbox_inches="tight")
#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/result_rmse_br_tot_best.pdf",format="pdf", bbox_inches="tight")

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
#### computing Occcci rmse

key_occci=['SmaAt_chlm_occci_only_variables_sst',
           'SmaAt_chlm_occci_only_variables_all', 
           'SmaAt_chlm_occci_only_variables_winds', 
           'SmaAt_chlm_occci_only_variables_currents', 
           'SmaAt_chlm_occci_only_variables_mld', 
           'SmaAt_chlm_occci_only_variables_sss', 
           'SmaAt_chlm_occci_only_variables_ssh', 
           'SmaAt_chlm_occci_only_variables_solar']

experiment_name_map_occci={'SmaAt_chlm_occci_only_variables_winds': 'E_Winds',
 'SmaAt_chlm_occci_only_variables_sss': 'E_SSS',
 'SmaAt_chlm_occci_only_variables_all': 'E_All',
 'SmaAt_chlm_occci_only_variables_ssh': 'E_SSH',
 'SmaAt_chlm_occci_only_variables_sst': 'E_SST',
 'SmaAt_chlm_occci_only_variables_solar': 'E_Cs',
 'SmaAt_chlm_occci_only_variables_mld': 'E_MLD',
 'SmaAt_chlm_occci_only_variables_currents': 'E_Currents',}

rmse_occci = {}
t={}
rmse_occci_best_two={}

for br_name,br in bio_regions_for_plot_2.items():
    rmse_occci[br_name]={
        key: RMSE(chl_pred_glob_occci[key][-36:]*br, chl_occci[-36:]*br)
    for key in key_occci}

    t[br_name] = {key: value for key, value in rmse_occci[br_name].items() if key != 'E_all'}

    # sort by RMSE (ascending = best)
    best_two = sorted(t[br_name].items(), key=lambda x: x[1])[:2]
    rmse_occci_best_two[br_name] = dict(best_two)



# Number of biomes
n = len(biome_names)
x = np.arange(n)
width = 0.25
categories = [(x[1]-1.5*width, x[2]+1.5*width, 'lightyellow'), 
              (x[3]-1.5*width, x[3]+1.5*width, 'lightpink'),
             (x[4]-1.5*width, x[5]+1.5*width, 'lightgreen'),
             (x[6]-1.5*width, x[-1]+1.5*width, 'lightblue')]

# Create the figure
fig, axes = plt.subplots(1, 1, sharex=True, figsize=(10, 4))
fig.subplots_adjust(hspace=0.25)   # spacing between plots

# -----------------------------
# DEFINE A FUNCTION TO DRAW ONE PANEL
# -----------------------------
def plot_panel(ax, rmse_vals_all, rmse_vals_best, labels_best,title, y_lim=[0,15],y_label='%'):
    # Add shaded regions for different periods
    for start, end, color in categories:
        ax.axvspan(start, end, facecolor=color, alpha=0.5)
    # Bars
    ax.bar(x - width, rmse_vals_all, width,
           label='Error E_all', color='white', edgecolor='black')
    ax.bar(x, rmse_vals_best[:,0], width,
           label='Error E_var_best 1', color='grey', edgecolor='black')
    ax.bar(x + width, rmse_vals_best[:,1], width,
           label='Error E_var_best 2', color='black', edgecolor='black')

    # Vertical labels inside black bars
    for i, (xi, label) in enumerate(zip(x, labels_best)):
        if  rmse_vals_best[i,0]>0.2*y_lim[1]:
            text_color='white'
            text_height_denominateur=2
            add=0
        else :
            text_color='black'
            text_height_denominateur=1
            add=0.15*y_lim[1]
            
        ax.text(xi, add+rmse_vals_best[i,0]/text_height_denominateur , experiment_name_map_occci[label[0]],
                ha='center', va='center',
                rotation=90, color=text_color, fontsize=10)
        ax.text(xi + width, add+rmse_vals_best[i,1]/text_height_denominateur, experiment_name_map_occci[label[1]],
                ha='center', va='center',
                rotation=90, color=text_color, fontsize=10)

    ax.set_title(title)
    ax.set_ylabel(y_label)
    ax.set_ylim(y_lim)
    ax.grid(axis='y', linestyle='--', alpha=0.4)




# -----------------------------
# PLOT YOUR FOUR PANELS
# -----------------------------
plot_panel(axes, np.array([rmse_occci[b]['SmaAt_chlm_occci_only_variables_all'] for b in biome_names]),
           np.array([list(rmse_occci_best_two[b].values()) for b in biome_names]), 
        np.array([list(rmse_occci_best_two[b].keys()) for b in biome_names]), "Chl oc-cci error",y_lim=[0.0,0.3],y_label='mg/m3')


# Set xticklabels on bottom subplot
biome_names_2=['Global',
 'Subtropics North',
 'Subtropics South',
 'Tropical Pacific',
 'Tropical Atlantique',
 'Indian ocean',
 'Seasonal strat Subtropics \nSouth',
 'SO-SA-high biomass',
 'Seasonal strat Subtropics \nAtlantic North',
 'Seasonal strat Subtropics \nPacific North']
axes.set_xticks(x)
axes.set_xticklabels(biome_names_2, rotation=45, ha='right')
add_biome_color_patches(axes, biome_names, bio_region_color)


# Global legend (optional)
handles, labels = axes.get_legend_handles_labels()
#fig.legend(handles, labels, loc='upper right')
axes.legend(handles, labels, loc='upper left')
#axes[1].legend(handles, labels, loc='upper left')
#axes[2].legend(handles, labels, loc='upper left')
#axes[3].legend(handles, labels, loc='upper left')


plt.tight_layout()
#plt.savefig("/home/luther/Documents/plot/article_1/result_rmse_br_tot_best_chl_occci.png",bbox_inches="tight")
#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/result_rmse_br_tot_best.pdf",format="pdf", bbox_inches="tight")

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Number of biomes
n = len(biome_names)
x = np.arange(n)
width = 0.25
categories = [(x[1]-width, x[2]+width, 'lightyellow'), 
              (x[3]-width, x[3]+width, 'lightpink'),
             (x[4]-width, x[5]+width, 'lightgreen'),
             (x[6]-width, x[-1]+width, 'lightblue')]
"""
# Create the figure
fig = plt.figure(figsize=(10, 12), constrained_layout=False)#, dpi=1200)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=1.,  top=1., bottom=0.77, wspace=0.05,hspace=0.05),  # Chl
    fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=1.,   top=0.75, bottom=0.52, wspace=0.05,hspace=0.05),  # Micro
    fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=1.,   top=0.5, bottom=0.27, wspace=0.05,hspace=0.05),  # Nano
    fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=1., top=0.25, bottom=0.,hspace=0.05),# Pico
]

axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs] 
"""
# Create the figure
fig, axes = plt.subplots(4, 1, sharex=True, figsize=(10, 12))
fig.subplots_adjust(hspace=0.25) 

A_chl=chl-chl_anom
B_chl=chl_anom+np.nanmean(chl,axis=0)

A_psc=psc_conc-psc_conc_anom
B_psc=psc_conc_anom+np.nanmean(psc_conc,axis=0)

A_vals_chl=[np.nanstd(np.nanmean(A_chl*bio_regions_for_plot_2[name],axis=(1,2))) for name in biome_names]
B_vals_chl=[np.nanstd(np.nanmean(B_chl*bio_regions_for_plot_2[name],axis=(1,2))) for name in biome_names]

A_vals_psc=[np.nanstd(np.nanmean(A_psc*bio_regions_for_plot_2[name],axis=(2,3)),axis=0) for name in biome_names]
B_vals_psc=[np.nanstd(np.nanmean(B_psc*bio_regions_for_plot_2[name],axis=(2,3)),axis=0) for name in biome_names]

stacked_bar_chl={'seasonal':np.array(A_vals_chl),'non seasonal':np.array(B_vals_chl)} 
stacked_bar_Micro={'seasonal':np.array(A_vals_psc)[:,0],'non seasonal':np.array(B_vals_psc)[:,0]} 
stacked_bar_Nano={'seasonal':np.array(A_vals_psc)[:,1],'non seasonal':np.array(B_vals_psc)[:,1]} 
stacked_bar_Pico={'seasonal':np.array(A_vals_psc)[:,2],'non seasonal':np.array(B_vals_psc)[:,2]} 

def add_pie_chart(ax, top_ax, bot_ax, stacked_bar):
    pie_width=0.03
    for i in range(1,len(stacked_bar['seasonal'])):
        x_center = 0.09*i+0.13
        new_grid_specs=fig.add_gridspec(nrows=1,ncols=1,left=x_center - pie_width / 2,right=x_center + pie_width / 2,top=top_ax,bottom=bot_ax)
        new_axe=fig.add_subplot(new_grid_specs[0,0])
        new_axe.pie([stacked_bar['seasonal'][i],stacked_bar['non seasonal'][i]], colors=['white','grey'], wedgeprops={"linewidth": 0.1, "edgecolor": "black"})
        # Hide axes ticks
        new_axe.grid(False)
        #new_axe.set_xticks([])
        #new_axe.set_yticks([])    
        new_axe.axis('off')
        
# -----------------------------
# DEFINE A FUNCTION TO DRAW ONE PANEL
# -----------------------------
def plot_panel(ax, rmse_vals_season, rmse_vals_anom, stacked_bar, top, bot, title, y_lim=[0,0.4], y_label='mg/m3'):
    # Add shaded regions for different periods
    for start, end, color in categories:
        ax.axvspan(start, end, facecolor=color, alpha=0.5)

    # Bars
    ax.bar(x - width/2, rmse_vals_season, width,
           label='Seasonal error', color='white', edgecolor='black')
    ax.bar(x+width/2, rmse_vals_anom, width,
           label='Non-seasonal error', color='grey', edgecolor='black')
    #stacked bars
    add_pie_chart(ax, top, bot, stacked_bar)
    #ax.bar(x+width,stacked_bar['seasonal'], width, label=None, color='white', edgecolor='black')
    #ax.bar(x+width, stacked_bar['non seasonal'], width, bottom=stacked_bar['seasonal'],label=None, color='grey', edgecolor='black')
    pie_chart_handles=[]
    pie_chart_handles.append(
        Line2D(
            [0], [0],
            marker='o',
            color='white',
            linestyle='None',
            markersize=8,
            label='Seasonal',
            markeredgecolor='black'
        )
    )
    pie_chart_handles.append(
        Line2D(
            [0], [0],
            marker='o',
            color='grey',
            linestyle='None',
            markersize=8,
            label='Non Seasonal',
            markeredgecolor='black'
        )
    )
    legend1 = ax.legend(
        handles=pie_chart_handles,
        title="% of the Signal",
        loc='upper left',
        #labelcolor=['white','grey'],
        fontsize=9)
    ax.add_artist(legend1)  # Add first legend manually


    ax.legend(title=title,loc='upper right')
    ax.set_ylabel(y_label)
    ax.set_ylim(y_lim)
    ax.grid(axis='y', linestyle='--', alpha=0.4)




# -----------------------------
# PLOT YOUR FOUR PANELS
# -----------------------------
plot_panel(axes[0], np.array([rmse_chl_seas[b]['SmaAt_chlm_avw_variables_all'] for b in biome_names]),
           np.array([rmse_chl_anom[b]['SmaAt_chlm_avw_variables_all'] for b in biome_names]),
           stacked_bar_chl, top=0.95, bot =0.85, y_lim=[0,0.35],
           title="Chl-a")

plot_panel(axes[1], np.array([rmse_psc_conc_seas[b]['SmaAt_chlm_avw_variables_all'][0] for b in biome_names]),
           np.array([rmse_psc_conc_anom[b]['SmaAt_chlm_avw_variables_all'][0] for b in biome_names]), 
           stacked_bar_Micro,top=0.85, bot =0.5, y_lim=[0,0.2],
           title="Micro")

plot_panel(axes[2], np.array([rmse_psc_conc_seas[b]['SmaAt_chlm_avw_variables_all'][1] for b in biome_names]),
           np.array([rmse_psc_conc_anom[b]['SmaAt_chlm_avw_variables_all'][1] for b in biome_names]), 
           stacked_bar_Nano,top=0.7, bot =0.25,y_lim=[0,0.15],
           title="Nano")

plot_panel(axes[3],  np.array([rmse_psc_conc_seas[b]['SmaAt_chlm_avw_variables_all'][2] for b in biome_names]),
           np.array([rmse_psc_conc_anom[b]['SmaAt_chlm_avw_variables_all'][2] for b in biome_names]), 
           stacked_bar_Pico,top=0.5, bot =0.,y_lim=[0,0.1],
           title="Pico")



# Set xticklabels on bottom subplot
axes[-1].set_xticks(x)
biome_names_2=['Global',
 'Subtropics North',
 'Subtropics South',
 'Tropical Pacific',
 'Tropical Atlantique',
 'Indian ocean',
 'Seasonal strat Subtropics \nSouth',
 'SO-SA-high biomass',
 'Seasonal strat Subtropics \nAtlantic North',
 'Seasonal strat Subtropics \nPacific North']
axes[-1].set_xticklabels(biome_names_2, rotation=45, ha='right')

add_biome_color_patches(axes[-1], biome_names, bio_region_color)

# ---- ADD COLOR PATCHES ON BOTTOM AXIS ----
add_biome_color_patches(axes[0], biome_names, bio_region_color)
add_biome_color_patches(axes[1], biome_names, bio_region_color)
add_biome_color_patches(axes[2], biome_names, bio_region_color)


# Global legend (optional)
#handles, labels = axes[0].get_legend_handles_labels()
#fig.legend(handles, labels, loc='upper right')
#axes[0].legend(handles, labels, loc='upper left')
#axes[1].legend(handles, labels, loc='upper left')
#axes[2].legend(handles, labels, loc='upper left')
#axes[3].legend(handles, labels, loc='upper left')

plt.tight_layout()
#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/result_rmse_br_anom_season.png",bbox_inches="tight")
#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/result_rmse_br_anom_season.pdf",format="pdf", bbox_inches="tight")

plt.show()


### RRMSE

In [ ]:
def RRMSE(pred,target,axis=0):
    top=np.sqrt(np.nansum((pred-target)**2,axis=axis))
    bot=np.sqrt(np.nansum((target)**2,axis=axis))
    return top/bot

rrmse_chl_map={}
for exp_name,exp in chl_pred_test.items():
    rrmse_chl_map[exp_name]=RRMSE(exp,chl_test)

rrmse_chl_br_0={}
rrmse_chl_br_1={}

rrmse_psc_br_0={}
rrmse_psc_br_1={}

for br_name,br in bio_regions_for_plot_2.items():
    rrmse_chl_br_0[br_name]={}
    rrmse_chl_br_1[br_name]={}
    rrmse_psc_br_0[br_name]={}
    rrmse_psc_br_1[br_name]={}
    for exp_name,exp in chl_pred_test.items():
        rrmse_chl_br_0[br_name][exp_name]=RRMSE(exp*br,chl_test*br,axis=None)
        rrmse_chl_br_1[br_name][exp_name]=np.nanmean(RRMSE(exp*br,chl_test*br))
        
        rrmse_psc_br_0[br_name][exp_name]=RRMSE(exp[:,None]*psc_pred_test[exp_name]*br,psc_test*chl_test[:,None]*br,axis=(0,2,3))
        rrmse_psc_br_1[br_name][exp_name]=np.nanmean(RRMSE(exp[:,None]*psc_pred_test[exp_name]*br,psc_test*chl_test[:,None]*br,axis=0),axis=(1,2))


In [ ]:
imshow_area(rrmse_chl_map['SmaAt_chlm_avw_variables_all'],cmap='cividis',vmin=0,vmax=1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# --------------------------------------------------
# Data
# --------------------------------------------------
n = len(biome_names)

rrmse_vals_0 = np.array([
    rrmse_chl_br_0[b]['SmaAt_chlm_avw_variables_all'] for b in biome_names
])
rrmse_vals_1 = np.array([
    rrmse_chl_br_1[b]['SmaAt_chlm_avw_variables_all'] for b in biome_names
])

# x positions (grouped like second plot)
x_center = np.arange(n) * 3.0
width = 0.6

# --------------------------------------------------
# Figure
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 6), dpi=600)

# --------------------------------------------------
# Background biome categories (axvspan)
# --------------------------------------------------
categories = [
    (x_center[1]-1.0, x_center[2]+1.0, 'lightyellow'),
    (x_center[3]-1.0, x_center[3]+1.0, 'lightpink'),
    (x_center[4]-1.0, x_center[5]+1.0, 'lightgreen'),
    (x_center[6]-1.0, x_center[-1]+1.0, 'lightblue')
]

for start, end, color in categories:
    ax.axvspan(start, end, facecolor=color, alpha=0.4, zorder=0)

# --------------------------------------------------
# Bars
# --------------------------------------------------
ax.bar(
    x_center - width/2,
    rrmse_vals_0,
    width,
    label='Error 0',
    facecolor='white',
    edgecolor='black',
    zorder=2
)

ax.bar(
    x_center + width/2,
    rrmse_vals_1,
    width,
    label='Error 1',
    facecolor='black',
    edgecolor='black',
    zorder=2
)

# --------------------------------------------------
# Axes labels & title
# --------------------------------------------------
ax.set_ylabel("RRMSE Chl-a ")
ax.set_title("Chl-a RRMSE by Biome")

# --------------------------------------------------
# X ticks (biome names)
# --------------------------------------------------
ax.set_xticks(x_center)
ax.set_xticklabels(
    [bio_region_name[b] for b in biome_names],
    rotation=45,
    ha='right'
)
ax.tick_params(axis='x', pad=15)

# --------------------------------------------------
# Biome color patches under x-axis
# --------------------------------------------------
ymin, ymax = ax.get_ylim()
y_patch = ymin - 0.08 * (ymax - ymin)
h_patch = 0.03 * (ymax - ymin)

for i, name in enumerate(biome_names):
    ax.add_patch(
        mpatches.Rectangle(
            (x_center[i] - 0.8, y_patch),
            1.6,
            h_patch,
            facecolor=bio_region_color[name],
            transform=ax.transData,
            clip_on=False
        )
    )

# --------------------------------------------------
# Legend
# --------------------------------------------------
error_0_patch = mpatches.Patch(facecolor="white", edgecolor="black", label="Relative RMSE")
error_1_patch = mpatches.Patch(facecolor="black", edgecolor="black", label="Mean of Spatial Relative RMSE")

ax.legend(handles=[error_0_patch, error_1_patch], loc="upper right")

# --------------------------------------------------
plt.tight_layout()
#plt.savefig("/home/luther/Documents/plot/article_1/RRMSE_0.png")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Number of biomes
n = len(biome_names)
x = np.arange(n)
width = 0.25
categories = [(x[1]-width, x[2]+width, 'lightyellow'), 
              (x[3]-width, x[3]+width, 'lightpink'),
             (x[4]-width, x[5]+width, 'lightgreen'),
             (x[6]-width, x[-1]+width, 'lightblue')]
"""
# Create the figure
fig = plt.figure(figsize=(10, 12), constrained_layout=False)#, dpi=1200)

# Define grid specifications and axes
grid_specs = [
    fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=1.,  top=1., bottom=0.77, wspace=0.05,hspace=0.05),  # Chl
    fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=1.,   top=0.75, bottom=0.52, wspace=0.05,hspace=0.05),  # Micro
    fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=1.,   top=0.5, bottom=0.27, wspace=0.05,hspace=0.05),  # Nano
    fig.add_gridspec(nrows=1, ncols=1, left=0.0, right=1., top=0.25, bottom=0.,hspace=0.05),# Pico
]

axes=[fig.add_subplot(gs[0,0]) for gs in grid_specs] 
"""
# Create the figure
fig, axes = plt.subplots(4, 1, sharex=True, figsize=(10, 12), dpi=600)
fig.subplots_adjust(hspace=0.25) 

        
# -----------------------------
# DEFINE A FUNCTION TO DRAW ONE PANEL
# -----------------------------
def plot_panel(ax, rrmse_vals_0, rrmse_vals_1, title, y_lim=[0,1], y_label='mg/m3'):
    # Add shaded regions for different periods
    for start, end, color in categories:
        ax.axvspan(start, end, facecolor=color, alpha=0.5)

    # Bars
    ax.bar(
        x - width/2,
        rrmse_vals_0,
        width,
        label='Relative RMSE',
        facecolor='white',
        edgecolor='black',
        zorder=2
    )
    
    ax.bar(
        x + width/2,
        rrmse_vals_1,
        width,
        label='Spatial averaged Relative RMSE',
        facecolor='black',
        edgecolor='black',
        zorder=2
    )
    

    ax.legend(title=title,loc='upper right')
    ax.set_ylabel(y_label)
    ax.set_ylim(y_lim)
    #ax.grid(axis='y', linestyle='--', alpha=0.4)




# -----------------------------
# PLOT YOUR FOUR PANELS
# -----------------------------
plot_panel(axes[0], np.array([rrmse_chl_br_0[b]['SmaAt_chlm_avw_variables_all'] for b in biome_names]),
           np.array([rrmse_chl_br_1[b]['SmaAt_chlm_avw_variables_all'] for b in biome_names]),
           title="Chl-a")

plot_panel(axes[1], np.array([rrmse_psc_br_0[b]['SmaAt_chlm_avw_variables_all'][0] for b in biome_names]),
           np.array([rrmse_psc_br_1[b]['SmaAt_chlm_avw_variables_all'][0] for b in biome_names]), 
           title="Micro")

plot_panel(axes[2], np.array([rrmse_psc_br_0[b]['SmaAt_chlm_avw_variables_all'][1] for b in biome_names]),
           np.array([rrmse_psc_br_1[b]['SmaAt_chlm_avw_variables_all'][1] for b in biome_names]), 
           title="Nano")

plot_panel(axes[3],  np.array([rrmse_psc_br_0[b]['SmaAt_chlm_avw_variables_all'][2] for b in biome_names]),
           np.array([rrmse_psc_br_1[b]['SmaAt_chlm_avw_variables_all'][2] for b in biome_names]), 
           title="Pico")


# Set xticklabels on bottom subplot
axes[-1].set_xticks(x)
biome_names_2=['Global',
               'Subtropics North',
               'Subtropics South',
               'Tropical Pacific',
               'Tropical Atlantique',
               'Indian ocean',
               'Seasonal strat Subtropics \nSouth',
               'SO-SA-high biomass',
               'Seasonal strat Subtropics \nAtlantic North',
               'Seasonal strat Subtropics \nPacific North']
axes[-1].set_xticklabels(biome_names_2, rotation=45, ha='right')

add_biome_color_patches(axes[-1], biome_names, bio_region_color)

# ---- ADD COLOR PATCHES ON BOTTOM AXIS ----
add_biome_color_patches(axes[0], biome_names, bio_region_color)
add_biome_color_patches(axes[1], biome_names, bio_region_color)
add_biome_color_patches(axes[2], biome_names, bio_region_color)


# Global legend (optional)
#handles, labels = axes[0].get_legend_handles_labels()
#fig.legend(handles, labels, loc='upper right')
#axes[0].legend(handles, labels, loc='upper left')
#axes[1].legend(handles, labels, loc='upper left')
#axes[2].legend(handles, labels, loc='upper left')
#axes[3].legend(handles, labels, loc='upper left')
labels = ['(a)', '(b)', '(c)', '(d)']

for ax, lab in zip(axes, labels):
    ax.text(0.05, 0.9, lab, transform=ax.transAxes, fontsize=12, fontweight='bold')
plt.tight_layout()
#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/rrmse_chl_psc.png",bbox_inches="tight")
#plt.savefig("/home/luther/Documents/plot/article_1/figures_finales/result_rmse_br_anom_season.pdf",format="pdf", bbox_inches="tight")

plt.show()


### Calcul inégalité triangulaire 

In [ ]:
def rmse(y_true, y_pred,axis=None):
    return np.sqrt(np.nanmean((y_pred - y_true) ** 2,axis=axis))

def compute_residu(signal_tot_true,signal_tot_pred,
                   signal_seas_true,signal_seas_pred,
                   signal_nonseas_true,signal_nonseas_pred,axis=None):
    a=rmse(signal_tot_true,signal_tot_pred,axis=axis)
    b=rmse(signal_seas_true,signal_seas_pred)+rmse(signal_nonseas_true,signal_nonseas_pred,axis=axis)
    return np.abs(a-b),a,b

for br_name,br in bio_regions_for_plot_2.items():
    signal_seas_true=br*(chl_test-chl_anom_test)
    signal_seas_pred=br*(chl_pred_test['SmaAt_chlm_avw_variables_all']-chl_pred_anom_test['SmaAt_chlm_avw_variables_all'])

    signal_nonseas_true=br*chl_anom_test
    signal_nonseas_pred=br*chl_pred_anom_test['SmaAt_chlm_avw_variables_all']
    
    res,a,b=compute_residu(br*chl_test,br*chl_pred_test['SmaAt_chlm_avw_variables_all'],
                       signal_seas_true,signal_seas_pred,
                       signal_nonseas_true,signal_nonseas_pred)
    print(f"\n{br_name} absolute difference : {res}, rmse signal : {a}, rmse somme signaux : {b}")


### mystère

In [ ]:
# Number of biomes
n = len(biome_names)
x = np.arange(n)
width = 0.25
categories = [(x[1]-width, x[2]+width, 'lightyellow'), 
              (x[3]-width, x[3]+width, 'lightpink'),
             (x[4]-width, x[5]+width, 'lightblue'),
             (x[6]-width, x[-1]+width, 'lightgreen')]
# Create the figure
fig, axes = plt.subplots(1, 1, sharex=True, figsize=(14, 7))
fig.subplots_adjust(hspace=0.25)   # spacing between plots

# -----------------------------
# DEFINE A FUNCTION TO DRAW ONE PANEL
# -----------------------------
def plot_panel(ax, rmse_vals_season, rmse_vals_anom, title, y_lim=[0,0.35], y_label='mg/m3'):
    # Add shaded regions for different periods
    for start, end, color in categories:
        ax.axvspan(start, end, facecolor=color, alpha=0.5)

    # Bars
    ax.bar(x - width/2, rmse_vals_season, width,
           label='Seasonal error', color='white', edgecolor='black')
    ax.bar(x + width/2, rmse_vals_best, width,
           label='Non-seasonal error', color='grey', edgecolor='black')

    ax.set_title(title)
    ax.set_ylabel(y_label)
    ax.set_ylim(y_lim)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

# -----------------------------
# PLOT YOUR FOUR PANELS
# -----------------------------
plot_panel(axes, np.array([rmse_chl_seas[b]['SmaAt_chlm_avw_variables_all'] for b in biome_names]),
           np.array([rmse_chl_anom[b]['SmaAt_chlm_avw_variables_all'] for b in biome_names]),
           "Chl error")

# Set xticklabels on bottom subplot
axes.set_xticks(x)
axes.set_xticklabels(biome_names, rotation=45, ha='right')
add_biome_color_patches(axes, biome_names, bio_region_color)


# Global legend (optional)
handles, labels = axes.get_legend_handles_labels()
#fig.legend(handles, labels, loc='upper right')
axes.legend(handles, labels, loc='upper left')
#axes[1].legend(handles, labels, loc='upper left')
#axes[2].legend(handles, labels, loc='upper left')
#axes[3].legend(handles, labels, loc='upper left')

plt.tight_layout()
#plt.savefig("/home/luther/Documents/plot/article_1/result_rmse_br_anom_season_chl_only.png",bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Number of biomes
n = len(biome_names)
x = np.arange(n)
width = 0.25
categories = [(x[1]-width, x[2]+width, 'lightyellow'), 
              (x[3]-width, x[3]+width, 'lightpink'),
             (x[4]-width, x[5]+width, 'lightblue'),
             (x[6]-width, x[-1]+width, 'lightgreen')]
# Create the figure
fig, axes = plt.subplots(4, 1, sharex=True, figsize=(14, 16))
fig.subplots_adjust(hspace=0.25)   # spacing between plots

# -----------------------------
# DEFINE A FUNCTION TO DRAW ONE PANEL
# -----------------------------
def plot_panel(ax, height_1, height_2, title, y_lim=None, y_label='mg/m3'):
    # Add shaded regions for different periods
    for start, end, color in categories:
        ax.axvspan(start, end, facecolor=color, alpha=0.5)

    # Bars
    ax.bar(x - width/2, height_1, width,
           label='Seasonal error', color='white', edgecolor='black')
    ax.bar(x + width/2, height_2, width,
           label='Non-seasonal error', color='grey', edgecolor='black')

    ax.set_title(title)
    ax.set_ylabel(y_label)
    ax.set_ylim(y_lim)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

# -----------------------------
# PLOT YOUR FOUR PANELS
# -----------------------------
plot_panel(axes[0], np.array([relative_log_chl_seas[b]['SmaAt_chlm_avw_variables_all'] for b in biome_names]),
           np.array([relative_log_chl_anom[b]['SmaAt_chlm_avw_variables_all'] for b in biome_names]),
           "Chl error")

plot_panel(axes[1], np.array([rmse_psc_seas[b]['SmaAt_chlm_avw_variables_all'][0] for b in biome_names]),
           np.array([rmse_psc_anom[b]['SmaAt_chlm_avw_variables_all'][0] for b in biome_names]), 
           "Micro error")

plot_panel(axes[2], np.array([rmse_psc_seas[b]['SmaAt_chlm_avw_variables_all'][1] for b in biome_names]),
           np.array([rmse_psc_anom[b]['SmaAt_chlm_avw_variables_all'][1] for b in biome_names]), 
           "Nano error")

plot_panel(axes[3],  np.array([rmse_psc_seas[b]['SmaAt_chlm_avw_variables_all'][2] for b in biome_names]),
           np.array([rmse_psc_anom[b]['SmaAt_chlm_avw_variables_all'][2] for b in biome_names]), 
           "Pico error")



# Set xticklabels on bottom subplot
axes[-1].set_xticks(x)
axes[-1].set_xticklabels(biome_names, rotation=45, ha='right')
add_biome_color_patches(axes[-1], biome_names, bio_region_color)

# ---- ADD COLOR PATCHES ON BOTTOM AXIS ----
add_biome_color_patches(axes[0], biome_names, bio_region_color)
add_biome_color_patches(axes[1], biome_names, bio_region_color)
add_biome_color_patches(axes[2], biome_names, bio_region_color)


# Global legend (optional)
handles, labels = axes[0].get_legend_handles_labels()
#fig.legend(handles, labels, loc='upper right')
axes[0].legend(handles, labels, loc='upper left')
#axes[1].legend(handles, labels, loc='upper left')
#axes[2].legend(handles, labels, loc='upper left')
#axes[3].legend(handles, labels, loc='upper left')

plt.tight_layout()
#plt.savefig("/home/luther/Documents/plot/article_1/result_rmse_br_anom_season_2.png",bbox_inches="tight")
plt.show()


In [ ]:
rmse_psc_2={}
for br in rmse_psc:
    rmse_psc_2[br]={i:rmse_psc[br][i] for i in NRMSE_experiments['Global'].keys()}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

bio_regions_for_plot_2=bio_regions_for_plot
if 'Seasonal strat Subtropics North' in bio_regions_for_plot_2.keys():
    bio_regions_for_plot_2.pop('Seasonal strat Subtropics North')
# Marker styles (one per experiment)
marker_styles = ['o', 's', 'D', '^', 'v', '<', '>', 'p', '*', 'X']

# Colors for bioregions
#colors = plt.cm.tab20c(np.linspace(0, 1, n_bioregions))
colors =  tc.tol_cmap('rainbow_discrete')( np.linspace(0,1,20 ))

# roup mapping for bioregions
# Define how each bioregion is grouped (you can replace this logic with your own)
group_mapping = {}
colors_mapping = {}
colors_counter=[0,6,12]
for name_br,br in bio_regions_for_plot_2.items():
    if np.nanmean(chl_snr*br)>1.2:
        group_mapping[name_br] = 'Strong Chl-a seasonal signal (Ratio>1.2)'

    elif np.nanmean(chl_snr*br)<0.8:
        group_mapping[name_br] = 'Weak  Chl-a seasonal signal (Ratio<0.8)'

    else:
        group_mapping[name_br] = 'Medium  Chl-a seasonal signal'


# Sort bioregions by group to ensure consistent legend order
sorted_bioregions = sorted(bio_regions_for_plot_2.items(), key=lambda item: group_mapping[item[0]])

colors_mapping['Global']='black'
# ----------------------------
# Plotting
# ----------------------------

plt.figure(figsize=(12, 7))

# Plot each data point with the corresponding experiment marker and bioregion color
for j,(name_br,br) in enumerate(bio_regions_for_plot_2.items()):
    
    vmax=np.nanmax(np.array(list(rmse_psc_2[name_br].values()))[:,0])
    vmin=np.nanmin(np.array(list(rmse_psc_2[name_br].values()))[:,0])
    for i,(e_name,phy_index) in enumerate(index_experiments.items()):
        points=(rmse_psc_2[name_br][e_name][0]-vmin)/(vmax-vmin)
        plt.scatter(
            points,
            np.nanmean(physics_ratio[phy_index]*br),
            marker=marker_styles[i % len(marker_styles)],
            color=bio_region_color[name_br],#colors[j],
            edgecolor='black',
            s=150,
            alpha=0.85
        )
        """
    if name_br in ['Global','Seasonal strat Subtropics Atlantic North','Tropical Pacific']:
        x_points=[]
        y_points=[]
        for i,(e_name,phy_index) in enumerate(index_experiments.items()):
            x_points.append((NRMSE_experiments[name_br][e_name]-vmin)/(vmax-vmin))
            y_points.append(np.nanmean(physics_ratio[phy_index]*br))
        x_points,y_points=zip(*sorted(zip(x_points,y_points)))
        plt.plot(
            list(x_points),
            list(y_points),
            color=bio_region_color[name_br],#colors[j],
            alpha=0.5,
            linestyle='dashed'
        )
        """
plt.yscale('log')
plt.ylim([1e-1,10e1])
#plt.xlim([0,1])
# ----------------------------
# Legend 1: Experiments (Markers)
# ----------------------------
keys=['SmaAt_chlm_avw_variables_winds',
 'SmaAt_chlm_avw_variables_sss',
 'SmaAt_chlm_avw_variables_all',
 'SmaAt_chlm_avw_variables_ssh',
 'SmaAt_chlm_avw_variables_sst',
 'SmaAt_chlm_avw_variables_solar',
 'SmaAt_chlm_avw_variables_mld',
 'SmaAt_chlm_avw_variables_currents',]

rename=['E_Winds','E_SSS','E_All','E_SSH','E_SST','E_Cs','E_MLD','E_Currents',] #'cosin','lat'

experiment_name_map = dict(zip(keys, rename))

experiment_handles = [
    Line2D(
        [0], [0],
        marker=marker_styles[i],
        color='gray',
        linestyle='None',
        markersize=8,
        markerfacecolor='gray',
        label=experiment_name_map[e_name]#+'\n'
    ) for i,(e_name,phy_index) in enumerate(index_experiments.items())
]

legend1 = plt.legend(
    handles=experiment_handles,
    title="Experiments",
    loc='upper center',
    #bbox_to_anchor=(1.01, 1),
    fontsize=9
)
plt.gca().add_artist(legend1)  # Add first legend manually

# ----------------------------
# Legend 2: Bioregions (Colors)
# ----------------------------

# Bioregion legend handles grouped under >>1, <<1, ~1
bioregion_handles = []

last_group = None
for j, (name_br, br) in enumerate(sorted_bioregions):
    group = group_mapping[name_br]
    
    bioregion_handles.append(
        Line2D(
            [0], [0],
            marker='o',
            color=bio_region_color[name_br],
            linestyle='None',
            markersize=8,
            label=bio_region_name[name_br],
            markeredgecolor='black'
        )
    )


legend2 = plt.legend(
    handles=bioregion_handles,
    title="Bioregions",
    loc='upper right',
    #bbox_to_anchor=(1.01, 0.35),
    fontsize=8
)

# ----------------------------
# Plot Formatting
# ----------------------------

plt.xlabel("NRMSE", fontsize=12)
plt.yticks(fontsize=12)
plt.ylabel("Input ratio between seasonal climatology std \nand \nanomalies std" , fontsize=12)
plt.title("                     NRMSE vs Input ratio (seasonal climatology std on anomalies std) \nby Experiment and Bioregion", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# Show plot
#plt.savefig("/home/luther/Documents/plot/article_1/discussion_plot_4.png")
#plt.savefig("/home/luther/Documents/plot/article_1/discussion_plot_4.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

bio_regions_for_plot_2=bio_regions_for_plot
if 'Seasonal strat Subtropics North' in bio_regions_for_plot_2.keys():
    bio_regions_for_plot_2.pop('Seasonal strat Subtropics North')
# Marker styles (one per experiment)
marker_styles = ['o', 's', 'D', '^', 'v', '<', '>', 'p', '*', 'X']

# Colors for bioregions
#colors = plt.cm.tab20c(np.linspace(0, 1, n_bioregions))
colors =  tc.tol_cmap('rainbow_discrete')( np.linspace(0,1,20 ))

# roup mapping for bioregions
# Define how each bioregion is grouped (you can replace this logic with your own)
group_mapping = {}
colors_mapping = {}
colors_counter=[0,6,12]
for name_br,br in bio_regions_for_plot_2.items():
    if np.nanmean(chl_snr*br)>1.2:
        group_mapping[name_br] = 'Strong Chl-a seasonal signal (Ratio>1.2)'

    elif np.nanmean(chl_snr*br)<0.8:
        group_mapping[name_br] = 'Weak  Chl-a seasonal signal (Ratio<0.8)'

    else:
        group_mapping[name_br] = 'Medium  Chl-a seasonal signal'


# Sort bioregions by group to ensure consistent legend order
sorted_bioregions = sorted(bio_regions_for_plot_2.items(), key=lambda item: group_mapping[item[0]])

colors_mapping['Global']='black'
# ----------------------------
# Plotting
# ----------------------------

plt.figure(figsize=(12, 7))

# Plot each data point with the corresponding experiment marker and bioregion color
for j,(name_br,br) in enumerate(bio_regions_for_plot_2.items()):
    
    vmax=np.nanmax(np.array(list(rmse_psc_2[name_br].values()))[:,1])
    vmin=np.nanmin(np.array(list(rmse_psc_2[name_br].values()))[:,1])
    for i,(e_name,phy_index) in enumerate(index_experiments.items()):
        points=(rmse_psc_2[name_br][e_name][1]-vmin)/(vmax-vmin)
        plt.scatter(
            points,
            np.nanmean(physics_ratio[phy_index]*br),
            marker=marker_styles[i % len(marker_styles)],
            color=bio_region_color[name_br],#colors[j],
            edgecolor='black',
            s=150,
            alpha=0.85
        )
        """
    if name_br in ['Global','Seasonal strat Subtropics Atlantic North','Tropical Pacific']:
        x_points=[]
        y_points=[]
        for i,(e_name,phy_index) in enumerate(index_experiments.items()):
            x_points.append((NRMSE_experiments[name_br][e_name]-vmin)/(vmax-vmin))
            y_points.append(np.nanmean(physics_ratio[phy_index]*br))
        x_points,y_points=zip(*sorted(zip(x_points,y_points)))
        plt.plot(
            list(x_points),
            list(y_points),
            color=bio_region_color[name_br],#colors[j],
            alpha=0.5,
            linestyle='dashed'
        )
        """
plt.yscale('log')
plt.ylim([1e-1,10e1])
#plt.xlim([0,1])
# ----------------------------
# Legend 1: Experiments (Markers)
# ----------------------------
keys=['SmaAt_chlm_avw_variables_winds',
 'SmaAt_chlm_avw_variables_sss',
 'SmaAt_chlm_avw_variables_all',
 'SmaAt_chlm_avw_variables_ssh',
 'SmaAt_chlm_avw_variables_sst',
 'SmaAt_chlm_avw_variables_solar',
 'SmaAt_chlm_avw_variables_mld',
 'SmaAt_chlm_avw_variables_currents',
'seasonal',]

rename=['E_Winds','E_SSS','E_All','E_SSH','E_SST','E_Cs','E_MLD','E_Currents','Monthly Climatology'] #'cosin','lat'

experiment_name_map = dict(zip(keys, rename))

experiment_handles = [
    Line2D(
        [0], [0],
        marker=marker_styles[i],
        color='gray',
        linestyle='None',
        markersize=8,
        markerfacecolor='gray',
        label=experiment_name_map[e_name]#+'\n'
    ) for i,(e_name,phy_index) in enumerate(index_experiments.items())
]

legend1 = plt.legend(
    handles=experiment_handles,
    title="Experiments",
    loc='upper center',
    #bbox_to_anchor=(1.01, 1),
    fontsize=9
)
plt.gca().add_artist(legend1)  # Add first legend manually

# ----------------------------
# Legend 2: Bioregions (Colors)
# ----------------------------

# Bioregion legend handles grouped under >>1, <<1, ~1
bioregion_handles = []

last_group = None
for j, (name_br, br) in enumerate(sorted_bioregions):
    group = group_mapping[name_br]
    
    bioregion_handles.append(
        Line2D(
            [0], [0],
            marker='o',
            color=bio_region_color[name_br],
            linestyle='None',
            markersize=8,
            label=bio_region_name[name_br],
            markeredgecolor='black'
        )
    )


legend2 = plt.legend(
    handles=bioregion_handles,
    title="Bioregions",
    loc='upper right',
    #bbox_to_anchor=(1.01, 0.35),
    fontsize=8
)

# ----------------------------
# Plot Formatting
# ----------------------------

plt.xlabel("NRMSE", fontsize=12)
plt.yticks(fontsize=12)
plt.ylabel("Input ratio between seasonal climatology std \nand \nanomalies std" , fontsize=12)
plt.title("                     NRMSE vs Input ratio (seasonal climatology std on anomalies std) \nby Experiment and Bioregion", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# Show plot
#plt.savefig("/home/luther/Documents/plot/article_1/discussion_plot_4.png")
#plt.savefig("/home/luther/Documents/plot/article_1/discussion_plot_4.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

bio_regions_for_plot_2=bio_regions_for_plot
if 'Seasonal strat Subtropics North' in bio_regions_for_plot_2.keys():
    bio_regions_for_plot_2.pop('Seasonal strat Subtropics North')
# Marker styles (one per experiment)
marker_styles = ['o', 's', 'D', '^', 'v', '<', '>', 'p', '*', 'X']

# Colors for bioregions
#colors = plt.cm.tab20c(np.linspace(0, 1, n_bioregions))
colors =  tc.tol_cmap('rainbow_discrete')( np.linspace(0,1,20 ))

# roup mapping for bioregions
# Define how each bioregion is grouped (you can replace this logic with your own)
group_mapping = {}
colors_mapping = {}
colors_counter=[0,6,12]
for name_br,br in bio_regions_for_plot_2.items():
    if np.nanmean(chl_snr*br)>1.2:
        group_mapping[name_br] = 'Strong Chl-a seasonal signal (Ratio>1.2)'

    elif np.nanmean(chl_snr*br)<0.8:
        group_mapping[name_br] = 'Weak  Chl-a seasonal signal (Ratio<0.8)'

    else:
        group_mapping[name_br] = 'Medium  Chl-a seasonal signal'


# Sort bioregions by group to ensure consistent legend order
sorted_bioregions = sorted(bio_regions_for_plot_2.items(), key=lambda item: group_mapping[item[0]])

colors_mapping['Global']='black'
# ----------------------------
# Plotting
# ----------------------------

plt.figure(figsize=(12, 7))

# Plot each data point with the corresponding experiment marker and bioregion color
for j,(name_br,br) in enumerate(bio_regions_for_plot_2.items()):
    
    vmax=np.nanmax(np.array(list(rmse_psc_2[name_br].values()))[:,2])
    vmin=np.nanmin(np.array(list(rmse_psc_2[name_br].values()))[:,2])
    for i,(e_name,phy_index) in enumerate(index_experiments.items()):
        points=(rmse_psc_2[name_br][e_name][2]-vmin)/(vmax-vmin)
        plt.scatter(
            points,
            np.nanmean(physics_ratio[phy_index]*br),
            marker=marker_styles[i % len(marker_styles)],
            color=bio_region_color[name_br],#colors[j],
            edgecolor='black',
            s=150,
            alpha=0.85
        )
        """
    if name_br in ['Global','Seasonal strat Subtropics Atlantic North','Tropical Pacific']:
        x_points=[]
        y_points=[]
        for i,(e_name,phy_index) in enumerate(index_experiments.items()):
            x_points.append((NRMSE_experiments[name_br][e_name]-vmin)/(vmax-vmin))
            y_points.append(np.nanmean(physics_ratio[phy_index]*br))
        x_points,y_points=zip(*sorted(zip(x_points,y_points)))
        plt.plot(
            list(x_points),
            list(y_points),
            color=bio_region_color[name_br],#colors[j],
            alpha=0.5,
            linestyle='dashed'
        )
        """
plt.yscale('log')
plt.ylim([1e-1,10e1])
#plt.xlim([0,1])
# ----------------------------
# Legend 1: Experiments (Markers)
# ----------------------------
keys=['SmaAt_chlm_avw_variables_winds',
 'SmaAt_chlm_avw_variables_sss',
 'SmaAt_chlm_avw_variables_all',
 'SmaAt_chlm_avw_variables_ssh',
 'SmaAt_chlm_avw_variables_sst',
 'SmaAt_chlm_avw_variables_solar',
 'SmaAt_chlm_avw_variables_mld',
 'SmaAt_chlm_avw_variables_currents',
'seasonal',]

rename=['E_Winds','E_SSS','E_All','E_SSH','E_SST','E_Cs','E_MLD','E_Currents','Monthly Climatology'] #'cosin','lat'

experiment_name_map = dict(zip(keys, rename))

experiment_handles = [
    Line2D(
        [0], [0],
        marker=marker_styles[i],
        color='gray',
        linestyle='None',
        markersize=8,
        markerfacecolor='gray',
        label=experiment_name_map[e_name]#+'\n'
    ) for i,(e_name,phy_index) in enumerate(index_experiments.items())
]

legend1 = plt.legend(
    handles=experiment_handles,
    title="Experiments",
    loc='upper center',
    #bbox_to_anchor=(1.01, 1),
    fontsize=9
)
plt.gca().add_artist(legend1)  # Add first legend manually

# ----------------------------
# Legend 2: Bioregions (Colors)
# ----------------------------

# Bioregion legend handles grouped under >>1, <<1, ~1
bioregion_handles = []

last_group = None
for j, (name_br, br) in enumerate(sorted_bioregions):
    group = group_mapping[name_br]
    
    bioregion_handles.append(
        Line2D(
            [0], [0],
            marker='o',
            color=bio_region_color[name_br],
            linestyle='None',
            markersize=8,
            label=bio_region_name[name_br],
            markeredgecolor='black'
        )
    )


legend2 = plt.legend(
    handles=bioregion_handles,
    title="Bioregions",
    loc='upper right',
    #bbox_to_anchor=(1.01, 0.35),
    fontsize=8
)

# ----------------------------
# Plot Formatting
# ----------------------------

plt.xlabel("NRMSE", fontsize=12)
plt.yticks(fontsize=12)
plt.ylabel("Input ratio between seasonal climatology std \nand \nanomalies std" , fontsize=12)
plt.title("                     NRMSE vs Input ratio (seasonal climatology std on anomalies std) \nby Experiment and Bioregion", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# Show plot
#plt.savefig("/home/luther/Documents/plot/article_1/discussion_plot_4.png")
#plt.savefig("/home/luther/Documents/plot/article_1/discussion_plot_4.pdf", format="pdf", bbox_inches="tight")
plt.show()


###  Giga plot br

In [ ]:
marker_styles = ['o', 's', 'D', '^', 'v', '<', '>', 'p', '*', 'X']
colors =  tc.tol_cmap('rainbow_discrete')( np.linspace(0,1,20 ))
# roup mapping for bioregions
# Define how each bioregion is grouped (you can replace this logic with your own)
group_mapping = {}
colors_mapping = {}
colors_counter=[0,6,12]
for name_br,br in bio_regions_for_plot_2.items():
    if np.nanmean(chl_snr*br)>1.2:
        group_mapping[name_br] = 'Strong Chl-a seasonal signal (Ratio>1.2)'

    elif np.nanmean(chl_snr*br)<0.8:
        group_mapping[name_br] = 'Weak  Chl-a seasonal signal (Ratio<0.8)'

    else:
        group_mapping[name_br] = 'Medium  Chl-a seasonal signal'

# Sort bioregions by group to ensure consistent legend order
sorted_bioregions = sorted(bio_regions_for_plot_2.items(), key=lambda item: group_mapping[item[0]])

colors_mapping['Global']='black'


def scatter_plot_br(ax,psc=False,title='0'):

    # Plot each data point with the corresponding experiment marker and bioregion color
    for j,(name_br,br) in enumerate(bio_regions_for_plot_2.items()):

        if psc :
            vmax=np.nanmax(np.array(list(rmse_psc_2[name_br].values()))[:,psc-1])
            vmin=np.nanmin(np.array(list(rmse_psc_2[name_br].values()))[:,psc-1])
        else : 
            vmax=np.nanmax(list(NRMSE_experiments[name_br].values()))
            vmin=np.nanmin(list(NRMSE_experiments[name_br].values()))
            
        for i,(e_name,phy_index) in enumerate(index_experiments.items()):
            if psc:
                points=(rmse_psc_2[name_br][e_name][psc-1]-vmin)/(vmax-vmin)
            else : 
                points=(NRMSE_experiments[name_br][e_name]-vmin)/(vmax-vmin)
            ax.scatter(
                points,
                np.nanmean(physics_ratio[phy_index]*br),#np.abs(np.nanmean(corr_map[phy_index]*br))
                marker=marker_styles[i % len(marker_styles)],
                color=bio_region_color[name_br],#colors[j],
                edgecolor='black',
                s=150,
                alpha=0.85
            )
    ax.set_title(title)
    ax.set_yscale('log')
    ax.set_ylim(1e-1,1e2)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.set_xlabel("Relative Error", fontsize=12)
    ax.set_ylabel("Seasonal Index of physical variable", fontsize=12)
    #ax.sharex(ax_sharex)
    
    # ----------------------------
    # Legend 1: Experiments (Markers)
    # ----------------------------
    keys=['SmaAt_chlm_avw_variables_winds',
     'SmaAt_chlm_avw_variables_sss',
     'SmaAt_chlm_avw_variables_all',
     'SmaAt_chlm_avw_variables_ssh',
     'SmaAt_chlm_avw_variables_sst',
     'SmaAt_chlm_avw_variables_solar',
     'SmaAt_chlm_avw_variables_mld',
     'SmaAt_chlm_avw_variables_currents',
    'seasonal',]
    
    rename=['E_Winds','E_SSS','E_All','E_SSH','E_SST','E_Cs','E_MLD','E_Currents','Monthly Climatology'] #'cosin','lat'
    
    experiment_name_map = dict(zip(keys, rename))
    
    experiment_handles = [
        Line2D(
            [0], [0],
            marker=marker_styles[i],
            color='gray',
            linestyle='None',
            markersize=8,
            markerfacecolor='gray',
            label=experiment_name_map[e_name]#+'\n'
        ) for i,(e_name,phy_index) in enumerate(index_experiments.items())
    ]
    
    legend1 = ax.legend(
        handles=experiment_handles,
        title="Experiments",
        loc='upper center',
        #bbox_to_anchor=(1.01, 1),
        fontsize=9
    )
    ax.add_artist(legend1)  # Add first legend manually

# ----------------------------
# Legend 2: Bioregions (Colors)
# ----------------------------

# Bioregion legend handles grouped under >>1, <<1, ~1
    bioregion_handles = []
    
    last_group = None
    for j, (name_br, br) in enumerate(sorted_bioregions):
        group = group_mapping[name_br]
        
        bioregion_handles.append(
            Line2D(
                [0], [0],
                marker='o',
                color=bio_region_color[name_br],
                linestyle='None',
                markersize=8,
                label=bio_region_name[name_br],
                markeredgecolor='black'
            )
        )
    
    
    legend2 = ax.legend(
        handles=bioregion_handles,
        title="Bioregions",
        loc='upper right',
        #bbox_to_anchor=(1.01, 0.35),
        fontsize=8
    )
    

In [ ]:
fig, axes = plt.subplots(2, 2, sharex='col', sharey='row', figsize=(16, 12))
fig.subplots_adjust(hspace=0.)   # spacing between plots


scatter_plot_br(axes[0,0],title='Chl-a')
scatter_plot_br(axes[0,1],psc=1,title='Micro%')
scatter_plot_br(axes[1,0],psc=2,title='Nano%')
scatter_plot_br(axes[1,1],psc=3,title='Pico%')

'E_var Chl-a relatives error in function of the Seasonal Index of physical variable'
# ----------------------------
# Plot Formatting
# ----------------------------

#plt.yticks(fontsize=12)
#plt.ylabel("Input ratio between seasonal climatology std \nand \nanomalies std" , fontsize=12)
#plt.title("                     NRMSE vs Input ratio (seasonal climatology std on anomalies std) \nby Experiment and Bioregion", fontsize=14)
#plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# Show plot
#plt.savefig("/home/luther/Documents/plot/article_1/giga_plot_br_0.png")
#plt.savefig("/home/luther/Documents/plot/article_1/discussion_plot_4.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
marker_styles = ['o', 's', 'D', '^', 'v', '<', '>', 'p', '*', 'X']
colors =  tc.tol_cmap('rainbow_discrete')( np.linspace(0,1,20 ))
# roup mapping for bioregions
# Define how each bioregion is grouped (you can replace this logic with your own)
group_mapping = {}
colors_mapping = {}
colors_counter=[0,6,12]


# Sort bioregions by group to ensure consistent legend order
selected_bioregions = {'Global':bio_regions_for_plot_2['Global'],
                       'Subtropics North':bio_regions_for_plot_2['Subtropics North'],
                       'Tropical Pacific': bio_regions_for_plot_2['Tropical Pacific'],
                       'Indian ocean':bio_regions_for_plot_2['Indian ocean'],
                       'Seasonal strat Subtropics Atlantic North':bio_regions_for_plot_2['Seasonal strat Subtropics Atlantic North'],
                      }

colors_mapping['Global']='black'


def scatter_plot_br(ax,psc=False,title='0'):

    # Plot each data point with the corresponding experiment marker and bioregion color
    for j,(name_br,br) in enumerate(selected_bioregions.items()):

        if psc :
            vmax=np.nanmax(np.array(list(rmse_psc_2[name_br].values()))[:,psc-1])
            vmin=np.nanmin(np.array(list(rmse_psc_2[name_br].values()))[:,psc-1])
        else : 
            vmax=np.nanmax(list(NRMSE_experiments[name_br].values()))
            vmin=np.nanmin(list(NRMSE_experiments[name_br].values()))
            
        x_points=[]
        y_points=[]
        for i,(e_name,phy_index) in enumerate(index_experiments.items()):
            if psc:
                points=(rmse_psc_2[name_br][e_name][psc-1]-vmin)/(vmax-vmin)
            else : 
                points=(NRMSE_experiments[name_br][e_name]-vmin)/(vmax-vmin)
            ax.scatter(
                points,
                np.nanmean(physics_ratio[phy_index]*br),#np.abs(np.nanmean(corr_map[phy_index]*br))
                marker=marker_styles[i % len(marker_styles)],
                color=bio_region_color[name_br],#colors[j],
                edgecolor='black',
                s=150,
                alpha=0.85
            )

            x_points.append(points)
            y_points.append(np.nanmean(physics_ratio[phy_index]*br))
        x_points,y_points=zip(*sorted(zip(x_points,y_points)))
        ax.plot(
            list(x_points),
            list(y_points),
            color=bio_region_color[name_br],#colors[j],
            alpha=0.5,
            linestyle='dashed'
        )

    ax.set_title(title)
    ax.set_yscale('log')
    ax.set_ylim(1e-1,1e2)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.set_xlabel("Relative Error", fontsize=12)
    ax.set_ylabel("Seasonal Index of physical variable", fontsize=12)
    #ax.sharex(ax_sharex)
    
    # ----------------------------
    # Legend 1: Experiments (Markers)
    # ----------------------------
    keys=['SmaAt_chlm_avw_variables_winds',
     'SmaAt_chlm_avw_variables_sss',
     'SmaAt_chlm_avw_variables_all',
     'SmaAt_chlm_avw_variables_ssh',
     'SmaAt_chlm_avw_variables_sst',
     'SmaAt_chlm_avw_variables_solar',
     'SmaAt_chlm_avw_variables_mld',
     'SmaAt_chlm_avw_variables_currents',]
    
    rename=['E_Winds','E_SSS','E_All','E_SSH','E_SST','E_Cs','E_MLD','E_Currents'] #'cosin','lat'
    
    experiment_name_map = dict(zip(keys, rename))
    
    experiment_handles = [
        Line2D(
            [0], [0],
            marker=marker_styles[i],
            color='gray',
            linestyle='None',
            markersize=8,
            markerfacecolor='gray',
            label=experiment_name_map[e_name]#+'\n'
        ) for i,(e_name,phy_index) in enumerate(index_experiments.items())
    ]
    
    legend1 = ax.legend(
        handles=experiment_handles,
        title="Experiments",
        loc='upper center',
        #bbox_to_anchor=(1.01, 1),
        fontsize=9
    )
    ax.add_artist(legend1)  # Add first legend manually

# ----------------------------
# Legend 2: Bioregions (Colors)
# ----------------------------

# Bioregion legend handles grouped under >>1, <<1, ~1
    bioregion_handles = []
    
    last_group = None
    for j, (name_br, br) in enumerate(selected_bioregions.items()):
        
        bioregion_handles.append(
            Line2D(
                [0], [0],
                marker='o',
                color=bio_region_color[name_br],
                linestyle='None',
                markersize=8,
                label=bio_region_name[name_br],
                markeredgecolor='black'
            )
        )
    
    
    legend2 = ax.legend(
        handles=bioregion_handles,
        title="Bioregions",
        loc='upper right',
        #bbox_to_anchor=(1.01, 0.35),
        fontsize=8
    )
    

In [ ]:
fig, axes = plt.subplots(2, 2, sharex='col', sharey='row', figsize=(16, 12))
fig.subplots_adjust(hspace=0.)   # spacing between plots


scatter_plot_br(axes[0,0],title='Chl-a')
scatter_plot_br(axes[0,1],psc=1,title='Micro%')
scatter_plot_br(axes[1,0],psc=2,title='Nano%')
scatter_plot_br(axes[1,1],psc=3,title='Pico%')

'E_var Chl-a relatives error in function of the Seasonal Index of physical variable'
# ----------------------------
# Plot Formatting
# ----------------------------

#plt.yticks(fontsize=12)
#plt.ylabel("Input ratio between seasonal climatology std \nand \nanomalies std" , fontsize=12)
#plt.title("                     NRMSE vs Input ratio (seasonal climatology std on anomalies std) \nby Experiment and Bioregion", fontsize=14)
#plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# Show plot
#plt.savefig("/home/luther/Documents/plot/article_1/giga_plot_br_2.png")
#plt.savefig("/home/luther/Documents/plot/article_1/discussion_plot_4.pdf", format="pdf", bbox_inches="tight")
plt.show()

## Update

In [ ]:
marker_styles = ['o', 's', 'D', '^', 'v', '<', '>', 'p', '*', 'X']
colors =  tc.tol_cmap('rainbow_discrete')( np.linspace(0,1,20 ))
# roup mapping for bioregions
# Define how each bioregion is grouped (you can replace this logic with your own)
group_mapping = {}
colors_mapping = {}
colors_counter=[0,6,12]


# Sort bioregions by group to ensure consistent legend order
selected_bioregions = {'Global':bio_regions_for_plot_2['Global'],
                       'Subtropics North':bio_regions_for_plot_2['Subtropics North'],
                       'Tropical Pacific': bio_regions_for_plot_2['Tropical Pacific'],
                       'Indian ocean':bio_regions_for_plot_2['Indian ocean'],
                       'Seasonal strat Subtropics Atlantic North':bio_regions_for_plot_2['Seasonal strat Subtropics Atlantic North'],
                      }

colors_mapping['Global']='black'


def scatter_plot_br(ax,psc=False,title='0'):

    # Plot each data point with the corresponding experiment marker and bioregion color
    for j,(name_br,br) in enumerate(selected_bioregions.items()):

        if psc :
            vmax_x=np.nanmax(np.array(list(rmse_psc_conc_seas[name_br].values()))[:,psc-1])
            vmin_x=np.nanmin(np.array(list(rmse_psc_conc_seas[name_br].values()))[:,psc-1])

            vmax_y=np.nanmax(np.array(list(rmse_psc_conc_anom[name_br].values()))[:,psc-1])
            vmin_y=np.nanmin(np.array(list(rmse_psc_conc_anom[name_br].values()))[:,psc-1])
        else : 
            vmax_x=np.nanmax(list(rmse_chl_seas[name_br].values()))
            vmin_x=np.nanmin(list(rmse_chl_seas[name_br].values()))

            vmax_y=np.nanmax(list(rmse_chl_anom[name_br].values()))
            vmin_y=np.nanmin(list(rmse_chl_anom[name_br].values()))
            
        x_points=[]
        y_points=[]
        for i,(e_name,phy_index) in enumerate(index_experiments.items()):
            if psc:
                x_point=(rmse_psc_conc_seas[name_br][e_name][psc-1]-vmin_x)/(vmax_x-vmin_x)
                y_point=(rmse_psc_conc_anom[name_br][e_name][psc-1]-vmin_y)/(vmax_y-vmin_y)
            else : 
                x_point=(rmse_chl_seas[name_br][e_name]-vmin_x)/(vmax_x-vmin_x)
                y_point=(rmse_chl_anom[name_br][e_name]-vmin_y)/(vmax_y-vmin_y)
                
            ax.scatter(
                x_point,
                y_point,
                marker=marker_styles[i % len(marker_styles)],
                color=bio_region_color[name_br],#colors[j],
                edgecolor='black',
                s=150,
                alpha=0.85
            )

            x_points.append(x_point)
            y_points.append(y_point)
        x_points,y_points=zip(*sorted(zip(x_points,y_points)))
        ax.plot(
            list(x_points),
            list(y_points),
            color=bio_region_color[name_br],#colors[j],
            alpha=0.5,
            linestyle='dashed'
        )

    ax.set_title(title)
    #ax.set_yscale('log')
    #ax.set_ylim(1e-1,1e2)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.set_xlabel("Relative Error", fontsize=12)
    ax.set_ylabel("Seasonal Index of physical variable", fontsize=12)
    #ax.sharex(ax_sharex)
    
    # ----------------------------
    # Legend 1: Experiments (Markers)
    # ----------------------------
    keys=['SmaAt_chlm_avw_variables_winds',
     'SmaAt_chlm_avw_variables_sss',
     'SmaAt_chlm_avw_variables_all',
     'SmaAt_chlm_avw_variables_ssh',
     'SmaAt_chlm_avw_variables_sst',
     'SmaAt_chlm_avw_variables_solar',
     'SmaAt_chlm_avw_variables_mld',
     'SmaAt_chlm_avw_variables_currents',]
    
    rename=['E_Winds','E_SSS','E_All','E_SSH','E_SST','E_Cs','E_MLD','E_Currents'] #'cosin','lat'
    
    experiment_name_map = dict(zip(keys, rename))
    
    experiment_handles = [
        Line2D(
            [0], [0],
            marker=marker_styles[i],
            color='gray',
            linestyle='None',
            markersize=8,
            markerfacecolor='gray',
            label=experiment_name_map[e_name]#+'\n'
        ) for i,(e_name,phy_index) in enumerate(index_experiments.items())
    ]
    
    legend1 = ax.legend(
        handles=experiment_handles,
        title="Experiments",
        loc='upper center',
        #bbox_to_anchor=(1.01, 1),
        fontsize=9
    )
    ax.add_artist(legend1)  # Add first legend manually

# ----------------------------
# Legend 2: Bioregions (Colors)
# ----------------------------

# Bioregion legend handles grouped under >>1, <<1, ~1
    bioregion_handles = []
    
    last_group = None
    for j, (name_br, br) in enumerate(selected_bioregions.items()):
        
        bioregion_handles.append(
            Line2D(
                [0], [0],
                marker='o',
                color=bio_region_color[name_br],
                linestyle='None',
                markersize=8,
                label=bio_region_name[name_br],
                markeredgecolor='black'
            )
        )
    
    
    legend2 = ax.legend(
        handles=bioregion_handles,
        title="Bioregions",
        loc='upper right',
        #bbox_to_anchor=(1.01, 0.35),
        fontsize=8
    )
    

In [ ]:
fig, axes = plt.subplots(2, 2, sharex='col', sharey='row', figsize=(16, 12))
fig.subplots_adjust(hspace=0.)   # spacing between plots


scatter_plot_br(axes[0,0],title='Chl-a')
scatter_plot_br(axes[0,1],psc=1,title='Micro%')
scatter_plot_br(axes[1,0],psc=2,title='Nano%')
scatter_plot_br(axes[1,1],psc=3,title='Pico%')

'E_var Chl-a relatives error in function of the Seasonal Index of physical variable'
# ----------------------------
# Plot Formatting
# ----------------------------

#plt.yticks(fontsize=12)
#plt.ylabel("Input ratio between seasonal climatology std \nand \nanomalies std" , fontsize=12)
#plt.title("                     NRMSE vs Input ratio (seasonal climatology std on anomalies std) \nby Experiment and Bioregion", fontsize=14)
#plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# Show plot
#plt.savefig("/home/luther/Documents/plot/article_1/giga_plot_br_2.png")
#plt.savefig("/home/luther/Documents/plot/article_1/discussion_plot_4.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
print(np.nanmedian(chl_test*bio_regions_for_plot_2['Global']))
rmse_chl['Tropical Atlantique']

In [ ]:

def rmsre(pred, true, axis=None):
    ratio = np.abs(pred / true)
    ratio = np.where(np.isinf(ratio), np.nan, ratio)   # convert +/-inf to nan
    return np.nanmean(np.abs(ratio - 1), axis=axis)
    
keys=['SmaAt_chlm_avw_variables_winds',
     'SmaAt_chlm_avw_variables_sss',
     'SmaAt_chlm_avw_variables_all',
     'SmaAt_chlm_avw_variables_ssh',
     'SmaAt_chlm_avw_variables_sst',
     'SmaAt_chlm_avw_variables_solar',
     'SmaAt_chlm_avw_variables_mld',
     'SmaAt_chlm_avw_variables_currents',]
    
rmsre_chl={}
rmsre_psc={}

for key in keys :
    rmsre_chl[key]={}
    rmsre_psc[key]={}
    
    for br_name,br in new_bio_regions.items():
        rmsre_chl[key][br_name]=rmsre(np.clip(chl_pred_test[key],1e-5,10)*br,chl_test*br)
        rmsre_psc[key][br_name]=rmsre(psc_pred_test[key]*br,np.clip(psc_test,0,None)*br,axis=(0,2,3))


In [ ]:
a=np.where(psc_test>0,np.nan,psc_test)

In [ ]:
a[~np.isnan(a)].shape

In [ ]:
a=(psc_pred_test[key][:,0]/psc_test[:,0])*bio_regions_for_plot_2['Subtropics North']
plt.plot(a[~np.isnan(a)].flatten())
plt.ylim(-2,2)
plt.show()

In [ ]:
rmse_chl['Global'].keys()

In [ ]:
marker_styles = ['o', 's', 'D', '^', 'v', '<', '>', 'p', '*', 'X']

colors =  tc.tol_cmap('rainbow_discrete')( np.linspace(0,1,20 ))
# roup mapping for bioregions
# Define how each bioregion is grouped (you can replace this logic with your own)
group_mapping = {}
colors_mapping = {}
colors_counter=[0,6,12]


# Sort bioregions by group to ensure consistent legend order
selected_bioregions = {'Global':bio_regions_for_plot_2['Global'],
                       'Subtropics North':bio_regions_for_plot_2['Subtropics North'],
                       'Tropical Pacific': bio_regions_for_plot_2['Tropical Pacific'],
                       'Tropical Atlantique':bio_regions_for_plot_2['Tropical Atlantique'],
                       'Seasonal strat Subtropics Atlantic North':bio_regions_for_plot_2['Seasonal strat Subtropics Atlantic North'],
                      }

colors_mapping['Global']='black'


def scatter_plot_final(ax,name_br,br,title='0',y_lim_chl=(0,1)):
    
    keys=['SmaAt_chlm_avw_variables_winds',
         'SmaAt_chlm_avw_variables_sss',
         'SmaAt_chlm_avw_variables_all',
         'SmaAt_chlm_avw_variables_ssh',
         'SmaAt_chlm_avw_variables_sst',
         'SmaAt_chlm_avw_variables_solar',
         'SmaAt_chlm_avw_variables_mld',
         'SmaAt_chlm_avw_variables_currents',]
    
    rename=['E_Winds','E_SSS','E_All','E_SSH','E_SST','E_Cs','E_MLD','E_Currents'] #'cosin','lat'
    experiment_name_map = dict(zip(keys, rename))
    #ax_psc=ax.twinx()

    for i,(e_name,phy_index) in enumerate(index_experiments.items()):
            
        x_chl=1
        x_psc=[2,3,4]

        
        vmax_y_psc=np.nanmax(np.array([rmse_psc[name_br][i] for i in keys]),axis=0)
        vmin_y_psc=np.nanmin(np.array([rmse_psc[name_br][i]for i in keys]),axis=0)

        vmax_y_chl=np.nanmax(np.array([rmse_chl[name_br][i] for i in keys]))
        vmin_y_chl=np.nanmin(np.array([rmse_chl[name_br][i] for i in keys]))

        y_chl=(rmse_chl[name_br][e_name]-vmin_y_chl)/(vmax_y_chl-vmin_y_chl)
        y_psc=(rmse_psc[name_br][e_name]-vmin_y_psc)/(vmax_y_psc-vmin_y_psc)

        ax.scatter(
            x_chl,
            y_chl,
            marker=marker_styles[i % len(marker_styles)],
            color=bio_region_color[name_br],#colors[j],
            edgecolor='black',
            s=250,
            alpha=0.85
        )

        ax.scatter(
            x_psc,
            y_psc,
            marker=marker_styles[i % len(marker_styles)],
            color=bio_region_color[name_br],#colors[j],
            edgecolor='black',
            s=250,
            alpha=0.85
        )

        if y_chl==0:
            ax.annotate(f'error : {rmse_chl[name_br][e_name]:.2f} mg/m3',(1.1,0.))
        for i,psc_rmse_iter in enumerate(y_psc):
            if psc_rmse_iter==0:
                ax.annotate(f'error : {100*rmse_psc[name_br][e_name][i]:.1f} %',(i+2.1,0.))
                
    ax.set_title(title)
    ax.set_ylim(-0.1,1.1)
    ax.set_xlim(0.8,4.5)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.set_xticks([1,2,3,4],['Chl','Micro','Nano','Pico'], fontsize=12)
    ax.set_ylabel('relative error within experiments', fontsize=12)

    #ax_psc.set_ylim(0,1)
    #ax_psc.set_ylabel('%',fontsize=12)
    
    # ----------------------------
    # Legend 1: Experiments (Markers)
    # ----------------------------

    
    
    experiment_handles = [
        Line2D(
            [0], [0],
            marker=marker_styles[i],
            color='gray',
            linestyle='None',
            markersize=8,
            markerfacecolor='gray',
            label=experiment_name_map[e_name]#+'\n'
        ) for i,(e_name,phy_index) in enumerate(index_experiments.items())
    ]
    
    legend1 = ax.legend(
        handles=experiment_handles,
        title="Experiments",
        loc='upper center',
        #bbox_to_anchor=(1.01, 1),
        fontsize=9
    )
    #ax.add_artist(legend1)  # Add first legend manually['SmaAt_chlm_avw_variables_all']


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.subplots_adjust(hspace=0.)   # spacing between plots


scatter_plot_final(axes[0,0],'Global',selected_bioregions['Global'],title='Global')#,y_lim_chl=(0.1,0.15))
scatter_plot_final(axes[0,1],'Tropical Pacific',selected_bioregions['Tropical Pacific'],title='Tropical Pacific')#,y_lim_chl=(0.04,0.06))
scatter_plot_final(axes[1,0],'Tropical Atlantique',selected_bioregions['Tropical Atlantique'],title='Tropical Atlantique')#,y_lim_chl=(0.3,0.35))
scatter_plot_final(axes[1,1],'Seasonal strat Subtropics Atlantic North',
                   selected_bioregions['Seasonal strat Subtropics Atlantic North'],title='Seasonal strat Subtropics Atlantic North')#,y_lim_chl=(0.07,0.11))

# ----------------------------
# Plot Formatting
# ----------------------------

#plt.yticks(fontsize=12)
#plt.ylabel("Input ratio between seasonal climatology std \nand \nanomalies std" , fontsize=12)
#plt.title("                     NRMSE vs Input ratio (seasonal climatology std on anomalies std) \nby Experiment and Bioregion", fontsize=14)
#plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# Show plot
#plt.savefig("/home/luther/Documents/plot/article_1/giga_plot_br_3.png")
#plt.savefig("/home/luther/Documents/plot/article_1/discussion_plot_4.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
rmse_psc